In [ ]:
# COMPLETE PACKAGE INSTALLATION FOR TOURISM FORECASTING ENSEMBLE
# Run this single cell in Colab Pro - ALL dependencies from Model-Training-Guide.pdf

import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("🚀 Installing ALL required packages...\n")

# === CORE DATA & ML ===
packages = [
    # Data manipulation
    "pandas",
    "numpy",

    # ML Framework
    "scikit-learn",
    "scipy",

    # Tree models
    "xgboost",
    "lightgbm",

    # Time Series
    "statsmodels",
    "pmdarima",  # auto_arima

    # === DEEP LEARNING ===
    "torch",
    "torchvision",
    "torchaudio",  # Full PyTorch
    "pytorch-lightning",
    "transformers",  # HuggingFace for TSformer

    # === OPTIMIZATION ===
    "optuna",        # Bayesian Optimization
    "hyperopt",      # Alternative BO
    "deap",          # Genetic Algorithm

    # === EXPLAINABILITY ===
    "shap",
    "lime",

    # === VISUALIZATION ===
    "matplotlib",
    "seaborn",
    "plotly",
    "tensorboard",

    # === UTILITIES ===
    "pyyaml",           # Config files
    "python-dotenv",    # Environment variables
    "loguru",           # Logging
    "tqdm",             # Progress bars
    "joblib",           # Parallel processing
    "dill",             # Better pickling

    # === DEVELOPMENT ===
    "jupyter",
    "jupyterlab",
    "pytest",
    "black",    # Code formatting
    "flake8"    # Linting
]

# Install all packages
for i, pkg in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] Installing {pkg}...")
    install_package(pkg)

print("\n🎉 ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("\n✅ Ready for full MLOps implementation:")
print("- XGBoost + Bayesian Optimization ✓")
print("- ARIMAX + auto_arima ✓")
print("- TSformer (PyTorch Transformer) ✓")
print("- Genetic Algorithm ensemble ✓")
print("- SHAP explainability ✓")
print("- Walk-forward validation ✓")

# Verify key installations
print("\n🔍 Verifying key packages...")
import xgboost, torch, optuna, deap, shap, pmdarima, pytorch_lightning
print("✅ XGBoost, PyTorch, Optuna, DEAP, SHAP, auto_arima, Lightning: ALL WORKING")

print("\n🚀 Next: Run folder setup → Upload data → Start training!")


🚀 Installing ALL required packages...

[1/33] Installing pandas...
[2/33] Installing numpy...
[3/33] Installing scikit-learn...
[4/33] Installing scipy...
[5/33] Installing xgboost...
[6/33] Installing lightgbm...
[7/33] Installing statsmodels...
[8/33] Installing pmdarima...
[9/33] Installing torch...
[10/33] Installing torchvision...
[11/33] Installing torchaudio...
[12/33] Installing pytorch-lightning...
[13/33] Installing transformers...
[14/33] Installing optuna...
[15/33] Installing hyperopt...
[16/33] Installing deap...
[17/33] Installing shap...
[18/33] Installing lime...
[19/33] Installing matplotlib...
[20/33] Installing seaborn...
[21/33] Installing plotly...
[22/33] Installing tensorboard...
[23/33] Installing pyyaml...
[24/33] Installing python-dotenv...
[25/33] Installing loguru...
[26/33] Installing tqdm...
[27/33] Installing joblib...
[28/33] Installing dill...
[29/33] Installing jupyter...
[30/33] Installing jupyterlab...
[31/33] Installing pytest...
[32/33] Installing

In [ ]:
 pip install scikeras


In [ ]:
!pip install scikit-optimize


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 7.2 MB/s eta 0:00:00


In [ ]:
pip install optuna-integration[tfkeras]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 5.5 MB/s eta 0:00:00


In [ ]:
pip install bayesian-optimization


In [ ]:
pip install deap

In [ ]:
pip install tf_keras

In [ ]:
"""
SVR Model — Fixed Bayesian Optimization
========================================
Author : ML Engineering Team
Date   : March 2026
Purpose: Production SVR with Bayesian HPO, all leakage and search-space
         issues from the previous BO version corrected.

Key fixes over the previous BO script
--------------------------------------
FIX-1  Target scaling   : y is now StandardScaler-transformed before fitting
                          and inverse-transformed for evaluation.  SVR is
                          distance-based; raw arrivals (0-10 000+) make epsilon
                          and C search ranges meaningless unless the target lives
                          in a unit-normal space.

FIX-2  Search space     : C  expanded to (0.1, 5 000) log-uniform.
                          epsilon expanded to (1e-4, 0.5) in scaled units
                          (≈ 0.1–500 arrivals — covers the real residual range).
                          gamma expanded to (1e-5, 1.0) for rbf kernel.

FIX-3  CV splits        : 5 (was 4) for more stable fold-level MSE estimates
                          and a better Optuna optimization signal.

FIX-4  Safe-MAPE floor  : epsilon set to np.percentile(|y|, 10) so the
                          denominator is always a meaningful fraction of the
                          target distribution, not an arbitrary constant.

FIX-5  Pipeline target  : y_scaler is fit only on training folds inside each
                          Optuna trial, then the final y_scaler is fit on the
                          full training split — no leakage into val/test.

FIX-6  Rolling leakage  : The .shift(1) leakage-safe rolling was already in
                          the previous BO script and is retained.

FIX-7  Split ratios     : 75/15/10  (same as previous BO, kept consistent).

FIX-8  Trial efficiency : n_trials=100 with the pruner (MedianPruner) so
                          clearly bad configs are killed early, effectively
                          spending the compute budget on promising regions.

Install:
    pip install optuna scikit-learn pandas numpy joblib
"""

from __future__ import annotations

import json
import logging
import math
import random
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# =============================================================================
# CONFIGURATION
# =============================================================================

@dataclass
class FeatureConfig:
    target_col: str = "arrivals"
    date_col: str = "date"
    lag_days: List[int] = field(default_factory=lambda: [1, 7, 14, 30])
    rolling_windows: List[int] = field(default_factory=lambda: [7, 14, 30])
    add_expanding_mean: bool = True


@dataclass
class SplitConfig:
    train_ratio: float = 0.75
    val_ratio: float = 0.15
    test_ratio: float = 0.10

    def validate(self) -> None:
        total = self.train_ratio + self.val_ratio + self.test_ratio
        if not math.isclose(total, 1.0, rel_tol=1e-9, abs_tol=1e-9):
            raise ValueError(f"Split ratios must sum to 1.0, got {total:.6f}")


@dataclass
class OptimizationConfig:
    # -------------------------------------------------------------------------
    # FIX-8: n_trials raised to 100; MedianPruner stops bad trials early
    # so actual wall-clock time stays within budget despite more trials.
    #
    # COLAB SAFETY: timeout is set to 10 hours so the remaining 2 hours are
    # reserved for final model training, evaluation, and artifact saving,
    # well within the Colab free-tier 12-hour session limit.
    # -------------------------------------------------------------------------
    n_trials: int = 100
    timeout_seconds: int = 10 * 3600              # 10 h — leaves 2 h for training/eval/saving
    cv_splits: int = 5                           # FIX-3: was 4
    scoring_metric: str = "rmse"
    n_jobs_optuna: int = 1                    # reproducible with libsvm
    random_state: int = 42

    kernels: Tuple[str, ...] = ("rbf", "linear")

    # FIX-2: widened search space — all ranges are in *scaled* target units
    c_range: Tuple[float, float] = (0.1, 5_000.0)       # was (0.1, 200)
    epsilon_range: Tuple[float, float] = (1e-4, 0.5)    # was (1e-3, 1.0)
    gamma_range: Tuple[float, float] = (1e-5, 1.0)      # was (1e-4, 0.05)
    tol_range: Tuple[float, float] = (1e-5, 1e-2)
    cache_size_mb: int = 1_000


@dataclass
class TrainingConfig:
    data_path: str = "preprocessed-dataset.csv"
    output_root: str = "artifacts_svr_fixed"
    random_state: int = 42
    feature_config: FeatureConfig = field(default_factory=FeatureConfig)
    split_config: SplitConfig = field(default_factory=SplitConfig)
    optimization_config: OptimizationConfig = field(default_factory=OptimizationConfig)


CONFIG = TrainingConfig()


# =============================================================================
# LOGGING
# =============================================================================

def setup_logging(output_dir: Path) -> logging.Logger:
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = output_dir / f"svr_fixed_{timestamp}.log"

    logger = logging.getLogger("svr_fixed")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    fh = logging.FileHandler(log_path)
    fh.setFormatter(fmt)
    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(fh)
    logger.addHandler(sh)
    logger.propagate = False

    logger.info("Logging initialised — log file: %s", log_path)
    return logger


# =============================================================================
# UTILITIES
# =============================================================================

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


# =============================================================================
# DATA LOADING
# =============================================================================

def load_data(file_path: str, date_col: str, logger: logging.Logger) -> pd.DataFrame:
    logger.info("Loading data from %s", file_path)
    df = pd.read_csv(file_path)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    logger.info("Dataset shape  : %s", df.shape)
    logger.info("Date range     : %s  →  %s", df[date_col].min(), df[date_col].max())
    if df[date_col].duplicated().any():
        raise ValueError("Duplicate dates detected — time-series index must be unique.")
    return df


# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

def create_temporal_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    ds = df[date_col]
    df["day_of_week"]    = ds.dt.dayofweek
    df["day_of_month"]   = ds.dt.day
    df["month"]          = ds.dt.month
    df["quarter"]        = ds.dt.quarter
    df["day_of_year"]    = ds.dt.dayofyear
    df["week_of_year"]   = ds.dt.isocalendar().week.astype(int)
    df["year"]           = ds.dt.year
    df["is_weekend"]     = (ds.dt.dayofweek >= 5).astype(int)
    df["is_month_start"] = ds.dt.is_month_start.astype(int)
    df["is_month_end"]   = ds.dt.is_month_end.astype(int)

    # Cyclical encodings
    df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["mon_sin"]  = np.sin(2 * np.pi * df["month"] / 12)
    df["mon_cos"]  = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"]  = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["doy_cos"]  = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    return df


def create_lag_features(df: pd.DataFrame, target_col: str, lags: List[int]) -> pd.DataFrame:
    df = df.copy()
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)
    return df


def create_rolling_features(
    df: pd.DataFrame,
    target_col: str,
    windows: List[int],
    add_expanding_mean: bool,
) -> pd.DataFrame:
    """
    FIX-6 (retained from previous BO): shift(1) ensures no same-row leakage.
    """
    df = df.copy()
    history = df[target_col].shift(1)
    for w in windows:
        df[f"{target_col}_rmean_{w}"]  = history.rolling(w, min_periods=w).mean()
        df[f"{target_col}_rstd_{w}"]   = history.rolling(w, min_periods=w).std()
        df[f"{target_col}_rmin_{w}"]   = history.rolling(w, min_periods=w).min()
        df[f"{target_col}_rmax_{w}"]   = history.rolling(w, min_periods=w).max()
    if add_expanding_mean:
        df[f"{target_col}_expand_mean"] = history.expanding(
            min_periods=max(7, min(windows))
        ).mean()
    return df


def prepare_features(
    df: pd.DataFrame, cfg: FeatureConfig, logger: logging.Logger
) -> pd.DataFrame:
    logger.info("=" * 70)
    logger.info("FEATURE ENGINEERING")
    logger.info("=" * 70)
    df_features = df.copy()
    df_features = create_temporal_features(df_features, cfg.date_col)
    df_features = create_lag_features(df_features, cfg.target_col, cfg.lag_days)
    df_features = create_rolling_features(
        df_features, cfg.target_col, cfg.rolling_windows, cfg.add_expanding_mean
    )
    rows_before = len(df_features)
    df_features = df_features.dropna().reset_index(drop=True)
    logger.info(
        "Dropped %d warm-up rows — final shape: %s",
        rows_before - len(df_features),
        df_features.shape,
    )
    return df_features


# =============================================================================
# SPLITTING
# =============================================================================

def split_data_timeseries(
    df: pd.DataFrame,
    cfg: SplitConfig,
    date_col: str,
    logger: logging.Logger,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cfg.validate()
    logger.info("=" * 70)
    logger.info("TIME-SERIES SPLITTING  (%.0f / %.0f / %.0f)",
                cfg.train_ratio * 100, cfg.val_ratio * 100, cfg.test_ratio * 100)
    logger.info("=" * 70)
    n = len(df)
    train_end = int(n * cfg.train_ratio)
    val_end   = train_end + int(n * cfg.val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        logger.info(
            "%-5s : %4d rows  %s  →  %s",
            name, len(split), split[date_col].min(), split[date_col].max(),
        )
    return train_df, val_df, test_df


# =============================================================================
# DATASET PREPARATION
# =============================================================================

def prepare_datasets(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    target_col: str,
    date_col: str,
    logger: logging.Logger,
):
    exclude = {date_col, target_col, "arrivals_robust_scaled", "outlier_flag"}
    feature_cols = [c for c in train_df.columns if c not in exclude]
    logger.info("Feature count : %d", len(feature_cols))
    logger.info("Features      : %s", feature_cols)

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_val   = val_df[feature_cols].copy()
    y_val   = val_df[target_col].copy()
    X_test  = test_df[feature_cols].copy()
    y_test  = test_df[target_col].copy()

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols


# =============================================================================
# FIX-1 / FIX-5 : TARGET SCALER
# =============================================================================

def fit_target_scaler(y_train: pd.Series) -> StandardScaler:
    """
    FIX-1: Fit a StandardScaler on the training target only.
    This maps arrivals to ~N(0,1) so that SVR hyperparameters (C, epsilon)
    operate in a consistent unit space across all kernel / parameter trials.

    Critical: this scaler must NEVER see validation or test targets during fit.
    """
    scaler = StandardScaler()
    scaler.fit(y_train.to_numpy().reshape(-1, 1))
    return scaler


def scale_target(y: pd.Series, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(y.to_numpy().reshape(-1, 1)).ravel()


def inverse_scale_target(y_scaled: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.inverse_transform(y_scaled.reshape(-1, 1)).ravel()


# =============================================================================
# METRICS
# =============================================================================

def safe_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    FIX-4: Use 10th-percentile of |y_true| as denominator floor
    so that near-zero values don't cause 100 000 % errors.
    """
    floor = float(np.percentile(np.abs(y_true), 10))
    floor = max(floor, 1.0)   # absolute minimum of 1 arrival
    denominator = np.maximum(np.abs(y_true), floor)
    return float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)) / 2.0, 1.0)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)


def calculate_metrics(
    y_true: np.ndarray, y_pred: np.ndarray, dataset_name: str
) -> Dict[str, object]:
    mse = mean_squared_error(y_true, y_pred)
    return {
        "dataset":    dataset_name,
        "r2":         float(r2_score(y_true, y_pred)),
        "rmse":       float(np.sqrt(mse)),
        "mse":        float(mse),
        "mae":        float(mean_absolute_error(y_true, y_pred)),
        "safe_mape":  safe_mape(y_true, y_pred),
        "smape":      smape(y_true, y_pred),
    }


# =============================================================================
# MODEL BUILDING
# =============================================================================

def build_svr(params: Dict[str, object], cache_size_mb: int) -> SVR:
    """
    Build SVR directly (no Pipeline wrapper).
    Feature scaling is handled separately; target scaling is also separate.
    This makes it easy to inverse-transform predictions after fitting.
    """
    kwargs = {
        "kernel":     params["kernel"],
        "C":          params["C"],
        "epsilon":    params["epsilon"],
        "tol":        params["tol"],
        "cache_size": cache_size_mb,
    }
    if params["kernel"] == "rbf":
        kwargs["gamma"] = params["gamma"]
    return SVR(**kwargs)


# =============================================================================
# FEATURE SCALER (fit on training X only)
# =============================================================================

def fit_feature_scaler(X_train: pd.DataFrame) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


# =============================================================================
# OPTUNA OBJECTIVE
# =============================================================================

def suggest_svr_params(
    trial: optuna.trial.Trial, cfg: OptimizationConfig
) -> Dict[str, object]:
    kernel = trial.suggest_categorical("kernel", list(cfg.kernels))
    params: Dict[str, object] = {
        "kernel":  kernel,
        "C":       trial.suggest_float("C",       cfg.c_range[0],       cfg.c_range[1],       log=True),
        "epsilon": trial.suggest_float("epsilon", cfg.epsilon_range[0], cfg.epsilon_range[1], log=True),
        "tol":     trial.suggest_float("tol",     cfg.tol_range[0],     cfg.tol_range[1],     log=True),
    }
    if kernel == "rbf":
        params["gamma"] = trial.suggest_float(
            "gamma", cfg.gamma_range[0], cfg.gamma_range[1], log=True
        )
    return params


def cross_validate_timeseries(
    params: Dict[str, object],
    X: pd.DataFrame,
    y: pd.Series,
    cfg: OptimizationConfig,
    trial: Optional[optuna.trial.Trial] = None,
) -> Dict[str, object]:
    """
    FIX-5: Both feature scaler AND target scaler are fit inside each fold
    to prevent any data leakage during HPO evaluation.
    """
    tscv = TimeSeriesSplit(n_splits=cfg.cv_splits)
    fold_rmse: List[float] = []
    fold_mae:  List[float] = []
    fold_r2:   List[float] = []

    X_arr = X.to_numpy()
    y_arr = y.to_numpy()

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), start=1):
        X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
        y_tr_raw, y_va_raw = y_arr[tr_idx], y_arr[va_idx]

        # --- feature scaling (fit on train fold only) ---
        x_scaler = StandardScaler()
        X_tr = x_scaler.fit_transform(X_tr_raw)
        X_va = x_scaler.transform(X_va_raw)

        # --- FIX-1 / FIX-5: target scaling (fit on train fold only) ---
        y_scaler = StandardScaler()
        y_tr = y_scaler.fit_transform(y_tr_raw.reshape(-1, 1)).ravel()

        model = build_svr(params, cfg.cache_size_mb)
        model.fit(X_tr, y_tr)

        # Predict in scaled space, invert for metric computation
        y_va_pred_scaled = model.predict(X_va)
        y_va_pred = y_scaler.inverse_transform(
            y_va_pred_scaled.reshape(-1, 1)
        ).ravel()

        rmse = float(np.sqrt(mean_squared_error(y_va_raw, y_va_pred)))
        mae  = float(mean_absolute_error(y_va_raw, y_va_pred))
        r2   = float(r2_score(y_va_raw, y_va_pred))

        fold_rmse.append(rmse)
        fold_mae.append(mae)
        fold_r2.append(r2)

        # --- Optuna pruning (FIX-8) ---
        if trial is not None:
            trial.report(rmse, step=fold_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    return {
        "rmse_mean": float(np.mean(fold_rmse)),
        "rmse_std":  float(np.std(fold_rmse)),
        "mae_mean":  float(np.mean(fold_mae)),
        "r2_mean":   float(np.mean(fold_r2)),
        "fold_rmse": fold_rmse,
        "fold_mae":  fold_mae,
        "fold_r2":   fold_r2,
    }


# =============================================================================
# BAYESIAN OPTIMISATION
# =============================================================================

def optimize_hyperparameters(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cfg: OptimizationConfig,
    logger: logging.Logger,
):
    logger.info("=" * 70)
    logger.info("BAYESIAN HYPERPARAMETER OPTIMISATION (OPTUNA)")
    logger.info("=" * 70)
    logger.info(
        "trials=%d | timeout=%ds | cv_splits=%d",
        cfg.n_trials, cfg.timeout_seconds, cfg.cv_splits,
    )

    # FIX-8: MedianPruner stops clearly bad trials early
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)
    sampler = optuna.samplers.TPESampler(seed=cfg.random_state, multivariate=True)
    study   = optuna.create_study(
        direction="minimize", sampler=sampler, pruner=pruner
    )

    def objective(trial: optuna.trial.Trial) -> float:
        params = suggest_svr_params(trial, cfg)
        try:
            cv = cross_validate_timeseries(params, X_train, y_train, cfg, trial)
        except optuna.exceptions.TrialPruned:
            raise
        trial.set_user_attr("params_resolved", params)
        trial.set_user_attr("rmse_std",  cv["rmse_std"])
        trial.set_user_attr("mae_mean",  cv["mae_mean"])
        trial.set_user_attr("r2_mean",   cv["r2_mean"])
        trial.set_user_attr("fold_rmse", cv["fold_rmse"])
        return cv["rmse_mean"]

    study.optimize(
        objective,
        n_trials=cfg.n_trials,
        timeout=cfg.timeout_seconds,
        n_jobs=cfg.n_jobs_optuna,
        gc_after_trial=True,
        show_progress_bar=False,
    )

    best_trial  = study.best_trial
    best_params = best_trial.user_attrs["params_resolved"]

    logger.info("Best trial      : #%d", best_trial.number)
    logger.info("Best RMSE (OOF) : %.4f", study.best_value)
    logger.info("Best params     : %s", best_params)
    logger.info(
        "RMSE std        : %.4f",
        best_trial.user_attrs.get("rmse_std", float("nan")),
    )

    completed = [t for t in study.trials if t.value is not None]
    top5 = sorted(completed, key=lambda t: t.value)[:5]
    logger.info("Top-5 trials:")
    for rank, tr in enumerate(top5, 1):
        logger.info(
            "  #%d  trial=%-4d  RMSE=%.4f  params=%s",
            rank, tr.number, tr.value,
            tr.user_attrs.get("params_resolved", tr.params),
        )

    return study, best_params


# =============================================================================
# FINAL MODEL TRAINING
# =============================================================================

def train_final_model(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    best_params: Dict[str, object],
    cfg: OptimizationConfig,
    logger: logging.Logger,
) -> Tuple[SVR, StandardScaler, StandardScaler]:
    """
    FIX-1: Feature scaler and target scaler both fit on the full training set.
    Returns (model, x_scaler, y_scaler) so predictions can be inverted.
    """
    logger.info("=" * 70)
    logger.info("FINAL MODEL TRAINING")
    logger.info("=" * 70)
    logger.info("Params : %s", best_params)

    x_scaler = fit_feature_scaler(X_train)
    y_scaler = fit_target_scaler(y_train)

    X_scaled = x_scaler.transform(X_train)
    y_scaled = scale_target(y_train, y_scaler)

    model = build_svr(best_params, cfg.cache_size_mb)
    model.fit(X_scaled, y_scaled)

    logger.info("Final model fitted — support vectors: %d", model.n_support_.sum())
    return model, x_scaler, y_scaler


# =============================================================================
# EVALUATION
# =============================================================================

def evaluate_model(
    model: SVR,
    x_scaler: StandardScaler,
    y_scaler: StandardScaler,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    logger: logging.Logger,
):
    logger.info("=" * 70)
    logger.info("MODEL EVALUATION")
    logger.info("=" * 70)

    def predict_original_scale(X: pd.DataFrame) -> np.ndarray:
        X_sc   = x_scaler.transform(X)
        y_sc   = model.predict(X_sc)
        return inverse_scale_target(y_sc, y_scaler)

    predictions = {
        "train":      predict_original_scale(X_train),
        "validation": predict_original_scale(X_val),
        "test":       predict_original_scale(X_test),
    }

    actuals = {
        "train":      y_train.to_numpy(),
        "validation": y_val.to_numpy(),
        "test":       y_test.to_numpy(),
    }

    metrics: Dict[str, Dict[str, object]] = {}
    for split in ("train", "validation", "test"):
        m = calculate_metrics(actuals[split], predictions[split], split.capitalize())
        metrics[split] = m
        logger.info("%s metrics:", split.capitalize())
        logger.info("  R²        : %.6f", m["r2"])
        logger.info("  RMSE      : %.4f", m["rmse"])
        logger.info("  MAE       : %.4f", m["mae"])
        logger.info("  Safe MAPE : %.2f%%", m["safe_mape"])
        logger.info("  sMAPE     : %.2f%%", m["smape"])

    return metrics, predictions


# =============================================================================
# POST-SELECTION CV SUMMARY
# =============================================================================

def compute_train_cv_summary(
    best_params: Dict[str, object],
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cfg: OptimizationConfig,
    logger: logging.Logger,
) -> Dict[str, object]:
    logger.info("=" * 70)
    logger.info("POST-SELECTION TIME-SERIES CV SUMMARY")
    logger.info("=" * 70)
    summary = cross_validate_timeseries(best_params, X_train, y_train, cfg)
    logger.info(
        "CV | RMSE mean=%.4f  std=%.4f  | MAE mean=%.4f  | R² mean=%.4f",
        summary["rmse_mean"], summary["rmse_std"],
        summary["mae_mean"],  summary["r2_mean"],
    )
    for i, (rmse, mae, r2) in enumerate(
        zip(summary["fold_rmse"], summary["fold_mae"], summary["fold_r2"]), 1
    ):
        logger.info("  Fold %d  RMSE=%.4f  MAE=%.4f  R²=%.4f", i, rmse, mae, r2)
    return summary


# =============================================================================
# ARTIFACT HELPERS
# =============================================================================

def build_prediction_frame(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    predictions: Dict[str, np.ndarray],
    target_col: str,
    date_col: str,
) -> pd.DataFrame:
    frames = []
    for split_name, split_df in [
        ("train", train_df), ("validation", val_df), ("test", test_df)
    ]:
        frames.append(
            pd.DataFrame({
                date_col:    split_df[date_col].to_numpy(),
                "split":     split_name,
                "actual":    split_df[target_col].to_numpy(),
                "predicted": predictions[split_name],
                "residual":  split_df[target_col].to_numpy() - predictions[split_name],
            })
        )
    return pd.concat(frames, ignore_index=True)


def save_artifacts(
    output_dir: Path,
    model: SVR,
    x_scaler: StandardScaler,
    y_scaler: StandardScaler,
    study: optuna.study.Study,
    metrics: Dict[str, Dict[str, object]],
    cv_summary: Dict[str, object],
    best_params: Dict[str, object],
    feature_cols: List[str],
    prediction_frame: pd.DataFrame,
    config: TrainingConfig,
    logger: logging.Logger,
) -> None:
    logger.info("=" * 70)
    logger.info("SAVING ARTIFACTS")
    logger.info("=" * 70)

    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    joblib.dump(model,    output_dir / f"svr_model_{ts}.joblib")
    joblib.dump(x_scaler, output_dir / f"x_scaler_{ts}.joblib")
    joblib.dump(y_scaler, output_dir / f"y_scaler_{ts}.joblib")  # FIX-1 artefact
    joblib.dump(study,    output_dir / f"optuna_study_{ts}.joblib")

    prediction_frame.to_csv(output_dir / f"predictions_{ts}.csv", index=False)
    pd.DataFrame(study.trials_dataframe()).to_csv(
        output_dir / f"optuna_trials_{ts}.csv", index=False
    )

    payload = {
        "timestamp":        ts,
        "best_params":      best_params,
        "metrics":          metrics,
        "cv_summary":       cv_summary,
        "feature_columns":  feature_cols,
        "config":           asdict(config),
        "fixes_applied": [
            "FIX-1: StandardScaler on target y",
            "FIX-2: Widened C / epsilon / gamma search space",
            "FIX-3: 5 CV folds (was 4)",
            "FIX-4: safe_mape floor = 10th-pct of |y|",
            "FIX-5: per-fold target scaler inside Optuna objective",
            "FIX-6: leakage-safe rolling (.shift(1)) retained",
            "FIX-8: MedianPruner + 100 trials",
        ],
    }
    with open(output_dir / f"metrics_{ts}.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, default=str)

    for path in sorted(output_dir.iterdir()):
        if path.suffix in {".joblib", ".csv", ".json", ".log"}:
            logger.info("  Saved: %s", path.name)


# =============================================================================
# MAIN
# =============================================================================

def main(config: TrainingConfig = CONFIG):
    set_global_seed(config.random_state)
    output_dir = Path(config.output_root)
    logger = setup_logging(output_dir)

    logger.info("=" * 70)
    logger.info("SVR MODEL — FIXED BAYESIAN OPTIMISATION — STARTED")
    logger.info("=" * 70)
    logger.info("Started: %s", datetime.now())

    try:
        # 1. Load
        df = load_data(config.data_path, config.feature_config.date_col, logger)

        # 2. Feature engineering
        df_processed = prepare_features(df, config.feature_config, logger)

        # 3. Split
        train_df, val_df, test_df = split_data_timeseries(
            df_processed, config.split_config, config.feature_config.date_col, logger
        )

        # 4. X / y separation
        X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = prepare_datasets(
            train_df, val_df, test_df,
            target_col=config.feature_config.target_col,
            date_col=config.feature_config.date_col,
            logger=logger,
        )

        # 5. Bayesian HPO  (feature + target scaling happen inside each fold)
        study, best_params = optimize_hyperparameters(
            X_train, y_train, config.optimization_config, logger
        )

        # 6. Train final model on full training set
        model, x_scaler, y_scaler = train_final_model(
            X_train, y_train, best_params, config.optimization_config, logger
        )

        # 7. Evaluate on original target scale
        metrics, predictions = evaluate_model(
            model, x_scaler, y_scaler,
            X_train, y_train,
            X_val,   y_val,
            X_test,  y_test,
            logger,
        )

        # 8. Post-selection CV
        cv_summary = compute_train_cv_summary(
            best_params, X_train, y_train, config.optimization_config, logger
        )

        # 9. Build prediction frame
        pred_frame = build_prediction_frame(
            train_df, val_df, test_df, predictions,
            config.feature_config.target_col,
            config.feature_config.date_col,
        )

        # 10. Save everything
        save_artifacts(
            output_dir=output_dir,
            model=model,
            x_scaler=x_scaler,
            y_scaler=y_scaler,
            study=study,
            metrics=metrics,
            cv_summary=cv_summary,
            best_params=best_params,
            feature_cols=feature_cols,
            prediction_frame=pred_frame,
            config=config,
            logger=logger,
        )

        logger.info("=" * 70)
        logger.info("SVR MODEL — FIXED BO — COMPLETED SUCCESSFULLY")
        logger.info("=" * 70)
        logger.info("Finished: %s", datetime.now())

        return model, x_scaler, y_scaler, study, metrics, cv_summary

    except Exception as exc:
        logger.exception("Fatal error: %s", exc)
        raise


# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'optuna'

In [ ]:
pip install optuna

In [ ]:
"""
SVR Model — ULTRA-OPTIMIZED (Google Drive Version)
==================================================
Saves ALL artifacts (models, predictions, JSON, logs, Optuna study)
directly to Google Drive in a timestamped folder.

Run in Colab Pro:
1. Upload preprocessed-dataset.csv to Colab (or point to Drive path)
2. Run this cell
"""

# ========================== MOUNT GOOGLE DRIVE ==========================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ========================== IMPORTS ==========================
from __future__ import annotations

import json
import logging
import math
import random
import warnings
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ========================== CONFIG ==========================
@dataclass
class FeatureConfig:
    target_col: str = "arrivals"
    date_col: str = "date"
    lag_days: List[int] = field(default_factory=lambda: [1, 7, 14, 30])
    rolling_windows: List[int] = field(default_factory=lambda: [7, 14, 30])
    add_expanding_mean: bool = True


@dataclass
class SplitConfig:
    train_ratio: float = 0.75
    val_ratio: float = 0.15
    test_ratio: float = 0.10

    def validate(self) -> None:
        total = self.train_ratio + self.val_ratio + self.test_ratio
        if not math.isclose(total, 1.0, rel_tol=1e-9, abs_tol=1e-9):
            raise ValueError(f"Split ratios must sum to 1.0, got {total:.6f}")


@dataclass
class OptimizationConfig:
    n_trials: int = 200
    timeout_seconds: int = 10 * 3600
    cv_splits: int = 5
    n_jobs_optuna: int = 1
    random_state: int = 42

    kernels: Tuple[str, ...] = ("rbf", "linear", "poly")
    c_range: Tuple[float, float] = (0.01, 10_000.0)
    epsilon_range: Tuple[float, float] = (1e-5, 1.0)
    gamma_range: Tuple[float, float] = (1e-6, 10.0)
    tol_range: Tuple[float, float] = (1e-5, 1e-2)
    cache_size_mb: int = 1_000


@dataclass
class TrainingConfig:
    data_path: str = "preprocessed-dataset.csv"          # ← upload to Colab
    drive_base: str = "/content/drive/MyDrive/SVR_Artifacts"
    random_state: int = 42
    feature_config: FeatureConfig = field(default_factory=FeatureConfig)
    split_config: SplitConfig = field(default_factory=SplitConfig)
    optimization_config: OptimizationConfig = field(default_factory=OptimizationConfig)


CONFIG = TrainingConfig()


# ========================== LOGGING (Drive + Console) ==========================
def setup_logging(output_dir: Path) -> logging.Logger:
    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = output_dir / f"svr_ultra_{ts}.log"

    logger = logging.getLogger("svr_ultra")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    fh = logging.FileHandler(log_path)
    fh.setFormatter(fmt)
    logger.addHandler(fh)

    sh = logging.StreamHandler()
    sh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.propagate = False

    logger.info("Logging initialised — log file: %s", log_path)
    logger.info("All artifacts will be saved to: %s", output_dir)
    return logger


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


# ========================== ALL OTHER FUNCTIONS (unchanged) ==========================
def load_data(file_path: str, date_col: str, logger: logging.Logger) -> pd.DataFrame:
    logger.info("Loading data from %s", file_path)
    df = pd.read_csv(file_path)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    logger.info("Dataset shape  : %s", df.shape)
    logger.info("Date range     : %s  →  %s", df[date_col].min(), df[date_col].max())
    if df[date_col].duplicated().any():
        raise ValueError("Duplicate dates detected.")
    return df


def create_temporal_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    df = df.copy()
    ds = df[date_col]
    df["day_of_week"]    = ds.dt.dayofweek
    df["day_of_month"]   = ds.dt.day
    df["month"]          = ds.dt.month
    df["quarter"]        = ds.dt.quarter
    df["day_of_year"]    = ds.dt.dayofyear
    df["week_of_year"]   = ds.dt.isocalendar().week.astype(int)
    df["year"]           = ds.dt.year
    df["is_weekend"]     = (ds.dt.dayofweek >= 5).astype(int)
    df["is_month_start"] = ds.dt.is_month_start.astype(int)
    df["is_month_end"]   = ds.dt.is_month_end.astype(int)

    df["dow_sin"]  = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["mon_sin"]  = np.sin(2 * np.pi * df["month"] / 12)
    df["mon_cos"]  = np.cos(2 * np.pi * df["month"] / 12)
    df["doy_sin"]  = np.sin(2 * np.pi * df["day_of_year"] / 365.25)
    df["doy_cos"]  = np.cos(2 * np.pi * df["day_of_year"] / 365.25)
    return df


def create_lag_features(df: pd.DataFrame, target_col: str, lags: List[int]) -> pd.DataFrame:
    df = df.copy()
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = df[target_col].shift(lag)
    return df


def create_rolling_features(df: pd.DataFrame, target_col: str, windows: List[int], add_expanding_mean: bool) -> pd.DataFrame:
    df = df.copy()
    history = df[target_col].shift(1)
    for w in windows:
        df[f"{target_col}_rmean_{w}"]  = history.rolling(w, min_periods=w).mean()
        df[f"{target_col}_rstd_{w}"]   = history.rolling(w, min_periods=w).std()
        df[f"{target_col}_rmin_{w}"]   = history.rolling(w, min_periods=w).min()
        df[f"{target_col}_rmax_{w}"]   = history.rolling(w, min_periods=w).max()
    if add_expanding_mean:
        df[f"{target_col}_expand_mean"] = history.expanding(min_periods=max(7, min(windows))).mean()
    return df


def prepare_features(df: pd.DataFrame, cfg: FeatureConfig, logger: logging.Logger) -> pd.DataFrame:
    logger.info("=" * 70)
    logger.info("FEATURE ENGINEERING")
    logger.info("=" * 70)
    df_features = df.copy()
    df_features = create_temporal_features(df_features, cfg.date_col)
    df_features = create_lag_features(df_features, cfg.target_col, cfg.lag_days)
    df_features = create_rolling_features(df_features, cfg.target_col, cfg.rolling_windows, cfg.add_expanding_mean)
    rows_before = len(df_features)
    df_features = df_features.dropna().reset_index(drop=True)
    logger.info("Dropped %d warm-up rows — final shape: %s", rows_before - len(df_features), df_features.shape)
    return df_features


def split_data_timeseries(df: pd.DataFrame, cfg: SplitConfig, date_col: str, logger: logging.Logger):
    cfg.validate()
    logger.info("=" * 70)
    logger.info("TIME-SERIES SPLITTING")
    logger.info("=" * 70)
    n = len(df)
    train_end = int(n * cfg.train_ratio)
    val_end   = train_end + int(n * cfg.val_ratio)

    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()

    for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        logger.info("%-5s : %4d rows  %s → %s", name, len(split), split[date_col].min(), split[date_col].max())
    return train_df, val_df, test_df


def prepare_datasets(train_df, val_df, test_df, target_col, date_col, logger):
    exclude = {date_col, target_col, "arrivals_robust_scaled", "outlier_flag"}
    feature_cols = [c for c in train_df.columns if c not in exclude]
    logger.info("Feature count : %d", len(feature_cols))

    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_val   = val_df[feature_cols].copy()
    y_val   = val_df[target_col].copy()
    X_test  = test_df[feature_cols].copy()
    y_test  = test_df[target_col].copy()

    return X_train, y_train, X_val, y_val, X_test, y_test, feature_cols


def fit_target_scaler(y_train: pd.Series) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(y_train.to_numpy().reshape(-1, 1))
    return scaler


def scale_target(y: pd.Series, scaler: StandardScaler) -> np.ndarray:
    return scaler.transform(y.to_numpy().reshape(-1, 1)).ravel()


def inverse_scale_target(y_scaled: np.ndarray, scaler: StandardScaler) -> np.ndarray:
    return scaler.inverse_transform(y_scaled.reshape(-1, 1)).ravel()


def fit_feature_scaler(X_train: pd.DataFrame) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler


def safe_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    floor = float(np.percentile(np.abs(y_true), 10))
    floor = max(floor, 1.0)
    denominator = np.maximum(np.abs(y_true), floor)
    return float(np.mean(np.abs((y_true - y_pred) / denominator)) * 100)


def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    denom = np.maximum((np.abs(y_true) + np.abs(y_pred)) / 2.0, 1.0)
    return float(np.mean(np.abs(y_true - y_pred) / denom) * 100)


def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray, dataset_name: str):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "dataset":    dataset_name,
        "r2":         float(r2_score(y_true, y_pred)),
        "rmse":       float(np.sqrt(mse)),
        "mse":        float(mse),
        "mae":        float(mean_absolute_error(y_true, y_pred)),
        "safe_mape":  safe_mape(y_true, y_pred),
        "smape":      smape(y_true, y_pred),
    }


def build_svr(params: Dict[str, object], cache_size_mb: int) -> SVR:
    kwargs = {
        "kernel":     params["kernel"],
        "C":          params["C"],
        "epsilon":    params["epsilon"],
        "tol":        params["tol"],
        "cache_size": cache_size_mb,
    }
    if "gamma" in params:
        kwargs["gamma"] = params["gamma"]
    if params["kernel"] == "poly":
        kwargs["degree"] = params["degree"]
        kwargs["coef0"]  = params["coef0"]
    return SVR(**kwargs)


def suggest_svr_params(trial: optuna.trial.Trial, cfg: OptimizationConfig):
    kernel = trial.suggest_categorical("kernel", list(cfg.kernels))
    params = {
        "kernel":  kernel,
        "C":       trial.suggest_float("C",       cfg.c_range[0], cfg.c_range[1], log=True),
        "epsilon": trial.suggest_float("epsilon", cfg.epsilon_range[0], cfg.epsilon_range[1], log=True),
        "tol":     trial.suggest_float("tol",     cfg.tol_range[0], cfg.tol_range[1], log=True),
    }
    if kernel != "linear":
        params["gamma"] = trial.suggest_float("gamma", cfg.gamma_range[0], cfg.gamma_range[1], log=True)
    if kernel == "poly":
        params["degree"] = trial.suggest_int("degree", 2, 4)
        params["coef0"]  = trial.suggest_float("coef0", 0.0, 1.0)
    return params


def cross_validate_timeseries(params, X, y, cfg, trial=None):
    tscv = TimeSeriesSplit(n_splits=cfg.cv_splits)
    fold_rmse, fold_mae, fold_r2 = [], [], []

    X_arr = X.to_numpy()
    y_arr = y.to_numpy()

    for fold_idx, (tr_idx, va_idx) in enumerate(tscv.split(X_arr), start=1):
        X_tr_raw, X_va_raw = X_arr[tr_idx], X_arr[va_idx]
        y_tr_raw, y_va_raw = y_arr[tr_idx], y_arr[va_idx]

        x_scaler = StandardScaler()
        X_tr = x_scaler.fit_transform(X_tr_raw)
        X_va = x_scaler.transform(X_va_raw)

        y_scaler = StandardScaler()
        y_tr = y_scaler.fit_transform(y_tr_raw.reshape(-1, 1)).ravel()

        model = build_svr(params, cfg.cache_size_mb)
        model.fit(X_tr, y_tr)

        y_va_pred_scaled = model.predict(X_va)
        y_va_pred = y_scaler.inverse_transform(y_va_pred_scaled.reshape(-1, 1)).ravel()

        rmse = float(np.sqrt(mean_squared_error(y_va_raw, y_va_pred)))
        mae  = float(mean_absolute_error(y_va_raw, y_va_pred))
        r2   = float(r2_score(y_va_raw, y_va_pred))

        fold_rmse.append(rmse)
        fold_mae.append(mae)
        fold_r2.append(r2)

        if trial is not None:
            trial.report(rmse, step=fold_idx)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    return {
        "rmse_mean": float(np.mean(fold_rmse)),
        "rmse_std":  float(np.std(fold_rmse)),
        "mae_mean":  float(np.mean(fold_mae)),
        "r2_mean":   float(np.mean(fold_r2)),
        "fold_rmse": fold_rmse,
        "fold_mae":  fold_mae,
        "fold_r2":   fold_r2,
    }


def optimize_hyperparameters(X_train, y_train, cfg, logger):
    logger.info("=" * 70)
    logger.info("BAYESIAN HYPERPARAMETER OPTIMISATION (200 TRIALS)")
    logger.info("=" * 70)

    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)
    sampler = optuna.samplers.TPESampler(seed=cfg.random_state, multivariate=True)
    study   = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

    def objective(trial):
        params = suggest_svr_params(trial, cfg)
        try:
            cv = cross_validate_timeseries(params, X_train, y_train, cfg, trial)
        except optuna.exceptions.TrialPruned:
            raise
        trial.set_user_attr("params_resolved", params)
        return cv["rmse_mean"]

    study.optimize(objective, n_trials=cfg.n_trials, timeout=cfg.timeout_seconds,
                   n_jobs=cfg.n_jobs_optuna, gc_after_trial=True)

    best_trial = study.best_trial
    best_params = best_trial.user_attrs["params_resolved"]

    logger.info("Best trial      : #%d", best_trial.number)
    logger.info("Best RMSE (OOF) : %.4f", study.best_value)
    logger.info("Best params     : %s", best_params)
    return study, best_params


def train_final_model(X_train, y_train, best_params, cfg, logger):
    logger.info("=" * 70)
    logger.info("FINAL MODEL TRAINING")
    logger.info("=" * 70)

    x_scaler = fit_feature_scaler(X_train)
    y_scaler = fit_target_scaler(y_train)

    X_scaled = x_scaler.transform(X_train)
    y_scaled = scale_target(y_train, y_scaler)

    model = build_svr(best_params, cfg.cache_size_mb)
    model.fit(X_scaled, y_scaled)

    logger.info("Final model fitted — support vectors: %d", model.n_support_.sum())
    return model, x_scaler, y_scaler


def evaluate_model(model, x_scaler, y_scaler, X_train, y_train, X_val, y_val, X_test, y_test, logger):
    logger.info("=" * 70)
    logger.info("MODEL EVALUATION")
    logger.info("=" * 70)

    def predict_original_scale(X):
        X_sc = x_scaler.transform(X)
        y_sc = model.predict(X_sc)
        return inverse_scale_target(y_sc, y_scaler)

    predictions = {
        "train": predict_original_scale(X_train),
        "validation": predict_original_scale(X_val),
        "test": predict_original_scale(X_test),
    }

    actuals = {"train": y_train.to_numpy(), "validation": y_val.to_numpy(), "test": y_test.to_numpy()}

    metrics = {}
    for split in ("train", "validation", "test"):
        m = calculate_metrics(actuals[split], predictions[split], split.capitalize())
        metrics[split] = m
        logger.info("%s metrics:", split.capitalize())
        logger.info("  R²        : %.6f", m["r2"])
        logger.info("  RMSE      : %.4f", m["rmse"])
        logger.info("  MAE       : %.4f", m["mae"])
        logger.info("  Safe MAPE : %.2f%%", m["safe_mape"])
        logger.info("  sMAPE     : %.2f%%", m["smape"])

    return metrics, predictions


def compute_train_cv_summary(best_params, X_train, y_train, cfg, logger):
    logger.info("=" * 70)
    logger.info("POST-SELECTION TIME-SERIES CV SUMMARY")
    logger.info("=" * 70)
    summary = cross_validate_timeseries(best_params, X_train, y_train, cfg)
    logger.info("CV | RMSE mean=%.4f  std=%.4f  | MAE mean=%.4f  | R² mean=%.4f",
                summary["rmse_mean"], summary["rmse_std"], summary["mae_mean"], summary["r2_mean"])
    return summary


def build_prediction_frame(train_df, val_df, test_df, predictions, target_col, date_col):
    frames = []
    for split_name, split_df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
        frames.append(
            pd.DataFrame({
                date_col:    split_df[date_col].to_numpy(),
                "split":     split_name,
                "actual":    split_df[target_col].to_numpy(),
                "predicted": predictions[split_name],
                "residual":  split_df[target_col].to_numpy() - predictions[split_name],
            })
        )
    return pd.concat(frames, ignore_index=True)


def save_artifacts(output_dir, model, x_scaler, y_scaler, study, metrics, cv_summary,
                   best_params, feature_cols, prediction_frame, config, logger):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    joblib.dump(model,    output_dir / f"svr_model_{ts}.joblib")
    joblib.dump(x_scaler, output_dir / f"x_scaler_{ts}.joblib")
    joblib.dump(y_scaler, output_dir / f"y_scaler_{ts}.joblib")
    joblib.dump(study,    output_dir / f"optuna_study_{ts}.joblib")

    prediction_frame.to_csv(output_dir / f"predictions_{ts}.csv", index=False)
    pd.DataFrame(study.trials_dataframe()).to_csv(output_dir / f"optuna_trials_{ts}.csv", index=False)

    payload = {
        "timestamp": ts,
        "best_params": best_params,
        "metrics": metrics,
        "cv_summary": cv_summary,
        "feature_columns": feature_cols,
        "config": asdict(config),
        "fixes_applied": ["FIX-1 to FIX-11 + Google Drive save"],
    }
    with open(output_dir / f"metrics_{ts}.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=4, default=str)

    for path in sorted(output_dir.iterdir()):
        if path.suffix in {".joblib", ".csv", ".json", ".log"}:
            logger.info("  Saved: %s", path.name)


# ========================== MAIN ==========================
def main(config: TrainingConfig = CONFIG):
    # Create timestamped folder in Drive
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_root = Path(config.drive_base) / f"run_{ts}"
    logger = setup_logging(output_root)

    logger.info("=" * 70)
    logger.info("SVR MODEL — ULTRA-OPTIMIZED BO — GOOGLE DRIVE VERSION")
    logger.info("Artifacts folder: %s", output_root)
    logger.info("=" * 70)

    try:
        df = load_data(config.data_path, config.feature_config.date_col, logger)
        df_processed = prepare_features(df, config.feature_config, logger)
        train_df, val_df, test_df = split_data_timeseries(df_processed, config.split_config, config.feature_config.date_col, logger)
        X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = prepare_datasets(
            train_df, val_df, test_df, config.feature_config.target_col, config.feature_config.date_col, logger
        )

        study, best_params = optimize_hyperparameters(X_train, y_train, config.optimization_config, logger)
        model, x_scaler, y_scaler = train_final_model(X_train, y_train, best_params, config.optimization_config, logger)
        metrics, predictions = evaluate_model(model, x_scaler, y_scaler, X_train, y_train, X_val, y_val, X_test, y_test, logger)
        cv_summary = compute_train_cv_summary(best_params, X_train, y_train, config.optimization_config, logger)

        pred_frame = build_prediction_frame(train_df, val_df, test_df, predictions,
                                            config.feature_config.target_col, config.feature_config.date_col)

        save_artifacts(output_root, model, x_scaler, y_scaler, study, metrics, cv_summary,
                       best_params, feature_cols, pred_frame, config, logger)

        logger.info("=" * 70)
        logger.info("COMPLETED SUCCESSFULLY — All files saved to Google Drive")
        logger.info("=" * 70)

    except Exception as exc:
        logger.exception("Fatal error: %s", exc)
        raise


if __name__ == "__main__":
    main()

Mounted at /content/drive


2026-03-20 18:17:37,601 - svr_ultra - INFO - Logging initialised — log file: /content/drive/MyDrive/SVR_Artifacts/run_20260320_181737/svr_ultra_20260320_181737.log
2026-03-20 18:17:37,605 - svr_ultra - INFO - All artifacts will be saved to: /content/drive/MyDrive/SVR_Artifacts/run_20260320_181737
2026-03-20 18:17:37,606 - svr_ultra - INFO - ======================================================================
2026-03-20 18:17:37,607 - svr_ultra - INFO - SVR MODEL — ULTRA-OPTIMIZED BO — GOOGLE DRIVE VERSION
2026-03-20 18:17:37,608 - svr_ultra - INFO - Artifacts folder: /content/drive/MyDrive/SVR_Artifacts/run_20260320_181737
2026-03-20 18:17:37,609 - svr_ultra - INFO - ======================================================================
2026-03-20 18:17:37,610 - svr_ultra - INFO - Loading data from preprocessed-dataset.csv
2026-03-20 18:17:37,641 - svr_ultra - INFO - Dataset shape  : (5740, 20)
2026-03-20 18:17:37,643 - svr_ultra - INFO - Date range     : 2010-01-01 00:00:00  →  2025

In [ ]:
# =============================================================================
# Google Drive Mount + File Creation & Saving TEST SCRIPT
# Run this cell BEFORE your main SVR/TSformer script
# =============================================================================

from google.colab import drive
import os
from pathlib import Path
import json
import pandas as pd
from datetime import datetime

print("=== Google Drive Mount & Write Test ===")
print("Current time:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

# ────────────────────────────────────────────────
# 1. Mount Google Drive
# ────────────────────────────────────────────────
print("\nStep 1: Mounting Google Drive...")
try:
    drive.mount('/content/drive', force_remount=True)
    print("→ Drive mounted successfully")
except Exception as e:
    print("→ ERROR mounting Drive:", str(e))
    print("   → Fix: Check internet, re-run cell, approve popup")
    raise

# ────────────────────────────────────────────────
# 2. Define test folder (you can change this path)
# ────────────────────────────────────────────────
BASE_FOLDER = "/content/drive/MyDrive/Colab_Test_Folder"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
test_folder = f"{BASE_FOLDER}/test_{timestamp}"

print(f"\nStep 2: Will try to create folder: {test_folder}")

# ────────────────────────────────────────────────
# 3. Create folder
# ────────────────────────────────────────────────
try:
    os.makedirs(test_folder, exist_ok=True)
    print("→ Folder created successfully")
except Exception as e:
    print("→ ERROR creating folder:", str(e))
    print("   → Possible causes: no write permission, Drive not mounted, quota full")
    raise

# ────────────────────────────────────────────────
# 4. Test writing different file types
# ────────────────────────────────────────────────
print("\nStep 3: Writing test files...")

# 4.1 Text file
text_path = Path(test_folder) / "test_message.txt"
try:
    with open(text_path, "w") as f:
        f.write("Hello from Colab!\nThis file was created successfully.")
    print("→ Text file written OK:", text_path)
except Exception as e:
    print("→ ERROR writing text file:", str(e))

# 4.2 JSON file
json_path = Path(test_folder) / "test_config.json"
try:
    data = {
        "test": "successful",
        "timestamp": timestamp,
        "message": "Colab → Drive write test passed"
    }
    with open(json_path, "w") as f:
        json.dump(data, f, indent=4)
    print("→ JSON file written OK:", json_path)
except Exception as e:
    print("→ ERROR writing JSON:", str(e))

# 4.3 CSV file (using pandas)
csv_path = Path(test_folder) / "test_table.csv"
try:
    df = pd.DataFrame({
        "A": [1, 2, 3],
        "B": ["x", "y", "z"],
        "C": [10.5, 20.1, 30.9]
    })
    df.to_csv(csv_path, index=False)
    print("→ CSV file written OK:", csv_path)
except Exception as e:
    print("→ ERROR writing CSV:", str(e))

# ────────────────────────────────────────────────
# 5. Final verification
# ────────────────────────────────────────────────
print("\nStep 4: Checking created files...")
files = os.listdir(test_folder)
if files:
    print("→ Files found in folder:")
    for f in files:
        print("   ", f)
    print("\nSUCCESS: Drive mount + folder creation + file writing all working!")
    print(f"You can now safely run your SVR/TSformer script.")
    print(f"Recommended output folder example: {test_folder}")
else:
    print("→ No files found → something went wrong")

print("\nDone.")

=== Google Drive Mount & Write Test ===
Current time: 2026-03-20 18:15:16

Step 1: Mounting Google Drive...
Mounted at /content/drive
→ Drive mounted successfully

Step 2: Will try to create folder: /content/drive/MyDrive/Colab_Test_Folder/test_20260320_181521
→ Folder created successfully

Step 3: Writing test files...
→ Text file written OK: /content/drive/MyDrive/Colab_Test_Folder/test_20260320_181521/test_message.txt
→ JSON file written OK: /content/drive/MyDrive/Colab_Test_Folder/test_20260320_181521/test_config.json
→ CSV file written OK: /content/drive/MyDrive/Colab_Test_Folder/test_20260320_181521/test_table.csv

Step 4: Checking created files...
→ Files found in folder:
    test_message.txt
    test_config.json
    test_table.csv

SUCCESS: Drive mount + folder creation + file writing all working!
You can now safely run your SVR/TSformer script.
Recommended output folder example: /content/drive/MyDrive/Colab_Test_Folder/test_20260320_181521

Done.


In [ ]:
# Wherever you call calculate_metrics(), wrap the display like this:
ARRIVALS_SCALE = 100   # your CSV unit = hundreds of tourists

print(f"RMSE : {metrics['test']['rmse_orig'] * ARRIVALS_SCALE:,.0f} tourists")
print(f"MAE  : {metrics['test']['mae_orig']  * ARRIVALS_SCALE:,.0f} tourists")
print(f"MAPE : {metrics['test']['mape_orig']:.2f}%")
print(f"sMAPE: {metrics['test']['smape']:.2f}%")


KeyError: 'rmse_orig'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

def create_directories():
    dirs_to_create = [
        'logs',
        'tsformer_output',
        'tsformer_output/models',
        'tsformer_output/metrics',
        'output/svr',
        'output/tsformer',
        'output/ga',
        'output/final_ensemble',
        'production_model_output',
        'phase1_scenario_forecasts_v6_FINAL',
        'phase1_scenario_forecasts_v6_FINAL/scenario_forecasts',
        'phase1_scenario_forecasts_v6_FINAL/explainability_reports',
        'phase1_scenario_forecasts_v6_FINAL/daily_predictions',
        'phase2_explainability'
    ]

    for d in dirs_to_create:
        Path(d).mkdir(parents=True, exist_ok=True)
        print(f"Created directory: {d}/")

print("Creating necessary directories...")
create_directories()
print("All directories created successfully!")

In [ ]:
"""
Time Series Transformer (TSformer) — Optuna Ultra-Tuned + Fixed
================================================================
Sri Lankan Inbound Tourism Arrivals Prediction

ALL PREVIOUS FIXES (1-7) RETAINED + MAJOR UPGRADES:
----------------------------------------------------
FIX 1-7: Leakage-free sequences, safe MAPE, alignment, pandas ffill/bfill,
         ModelCheckpoint .keras, inverse clipping, etc. — 100% kept.

NEW UPGRADES (same philosophy as your SVR ultra script):
-------------------------------------------------------
• Optuna 200 trials + MedianPruner + TPESampler
• 10-hour hard timeout (perfect for Colab free tier)
• Dramatically wider search space + weight_decay
• d_model always aligned to num_heads in EVERY trial
• Adam with weight_decay + longer patience
• Per-trial logging + forced console output
• No bayes_opt dependency

Install in Colab (once):
!pip install optuna tensorflow scikit-learn pandas numpy

Run: Just execute the cell. It will stop after 10h or 200 trials.
"""

import numpy as np
import pandas as pd
import logging
import json
import warnings
import os
import random
import sys
from datetime import datetime
from dataclasses import dataclass
from typing import Tuple, Dict, Any

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, LayerNormalization, MultiHeadAttention,
    Input, Add, GlobalAveragePooling1D,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


# =============================================================================
# CONFIG (Optuna style — exactly like your SVR)
# =============================================================================

@dataclass
class OptimizationConfig:
    n_trials: int = 200
    timeout_seconds: int = 10 * 3600          # 10 hours for Colab
    random_state: int = 42


# =============================================================================
# Utility
# =============================================================================

def safe_mape(y_true: np.ndarray, y_pred: np.ndarray, epsilon: float = 1.0) -> float:
    mask = np.abs(y_true) >= epsilon
    if mask.sum() == 0:
        return float("nan")
    return float(
        np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    )


# =============================================================================
# Main Class
# =============================================================================

class TSformerModelExplorer:
    def __init__(self, data_path: str, output_dir: str = "tsformer_output_optuna"):
        self.data_path = data_path
        self.output_dir = output_dir
        self._setup_logging()
        self._setup_output_dir()

        self.scaler_X: StandardScaler = None
        self.scaler_y: MinMaxScaler = None
        self.best_model: keras.Model = None
        self.best_params: Dict[str, Any] = None
        self.history: Dict = None

        self.X_train = self.X_val = self.X_test = None
        self.y_train = self.y_val = self.y_test = None
        self.train_dates = self.val_dates = self.test_dates = None

        self.logger.info("TSformerModelExplorer initialised (Optuna ultra version)")

    # ------------------------------------------------------------------
    # Logging — Forced console output (like your SVR fixed version)
    # ------------------------------------------------------------------
    def _setup_logging(self):
        log_fmt = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        self.logger = logging.getLogger("TSformerModelExplorer")
        self.logger.setLevel(logging.INFO)
        self.logger.handlers.clear()

        fmt = logging.Formatter(log_fmt)

        # File
        fh = logging.FileHandler(f"tsformer_optuna_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
        fh.setFormatter(fmt)
        self.logger.addHandler(fh)

        # Console — force flush
        ch = logging.StreamHandler(sys.stdout)
        ch.setLevel(logging.INFO)
        ch.setFormatter(fmt)
        self.logger.addHandler(ch)

        self.logger.info("Logging initialised — console output forced")

    def _setup_output_dir(self):
        for sub in ["", "models", "metrics"]:
            os.makedirs(os.path.join(self.output_dir, sub), exist_ok=True)
        self.logger.info(f"Output directory ready: {self.output_dir}")

    # ------------------------------------------------------------------
    # Data loading, features, sequences, split (EXACTLY as your fixed version)
    # ------------------------------------------------------------------
    def load_data(self) -> pd.DataFrame:
        self.logger.info(f"Loading data from {self.data_path}")
        df = pd.read_csv(self.data_path)
        df["date"] = pd.to_datetime(df["date"])
        df = df.sort_values("date").reset_index(drop=True)
        self.logger.info(
            f"Loaded {df.shape[0]} rows × {df.shape[1]} cols  "
            f"({df['date'].min().date()} → {df['date'].max().date()})"
        )
        return df

    def create_features(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.info("Engineering features …")
        df = df.copy()
        df["day_of_week"]  = df["date"].dt.dayofweek
        df["day_of_month"] = df["date"].dt.day
        df["month"]        = df["date"].dt.month
        df["quarter"]      = df["date"].dt.quarter
        df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
        df["year"]         = df["date"].dt.year
        df["day_of_year"]  = df["date"].dt.dayofyear

        df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
        df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
        df["mon_sin"] = np.sin(2 * np.pi * df["month"] / 12)
        df["mon_cos"] = np.cos(2 * np.pi * df["month"] / 12)
        df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
        df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)

        for lag in [1, 7, 14, 30]:
            df[f"arrivals_lag_{lag}"] = df["arrivals"].shift(lag)

        for w in [7, 14, 30]:
            df[f"arr_rmean_{w}"] = df["arrivals"].rolling(w, min_periods=1).mean()
            df[f"arr_rstd_{w}"]  = df["arrivals"].rolling(w, min_periods=1).std()

        df["arr_ema_7"]  = df["arrivals"].ewm(span=7,  adjust=False).mean()
        df["arr_ema_30"] = df["arrivals"].ewm(span=30, adjust=False).mean()
        df["arr_diff_1"] = df["arrivals"].diff(1)
        df["arr_diff_7"] = df["arrivals"].diff(7)

        df = df.ffill().bfill()
        self.logger.info(f"Feature matrix shape: {df.shape}")
        return df

    @staticmethod
    def _make_sequences(X: np.ndarray, y: np.ndarray, seq_len: int):
        n = len(X)
        xs, ys = [], []
        for i in range(n - seq_len + 1):
            xs.append(X[i : i + seq_len])
            ys.append(y[i + seq_len - 1])
        return np.array(xs, dtype=np.float32), np.array(ys, dtype=np.float32)

    def split_data(self, df: pd.DataFrame, train_ratio=0.70, val_ratio=0.15,
                   test_ratio=0.15, sequence_length=30):
        assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
        n = len(df)
        train_end = int(n * train_ratio)
        val_end   = int(n * (train_ratio + val_ratio))

        self.logger.info(f"Split points: train 0..{train_end} val {train_end}..{val_end} test {val_end}..{n}")

        exclude = {"date", "arrivals", "arrivals_robust_scaled", "outlier_flag"}
        feat_cols = [c for c in df.columns if c not in exclude]

        X_all = df[feat_cols].values.astype(np.float64)
        y_all = df["arrivals"].values.astype(np.float64)

        self.scaler_X = StandardScaler()
        self.scaler_y = MinMaxScaler()

        X_all_scaled = X_all.copy()
        X_all_scaled[:train_end] = self.scaler_X.fit_transform(X_all[:train_end])
        X_all_scaled[train_end:] = self.scaler_X.transform(X_all[train_end:])

        y_all_scaled = y_all.copy()
        y_all_scaled[:train_end] = self.scaler_y.fit_transform(y_all[:train_end].reshape(-1, 1)).flatten()
        y_all_scaled[train_end:] = self.scaler_y.transform(y_all[train_end:].reshape(-1, 1)).flatten()

        X_seq, y_seq = self._make_sequences(X_all_scaled, y_all_scaled, sequence_length)
        target_row = np.arange(len(X_seq)) + (sequence_length - 1)

        train_mask = target_row < train_end
        val_mask   = (target_row >= train_end) & (target_row < val_end)
        test_mask  = target_row >= val_end

        X_train, y_train = X_seq[train_mask], y_seq[train_mask]
        X_val,   y_val   = X_seq[val_mask],   y_seq[val_mask]
        X_test,  y_test  = X_seq[test_mask],  y_seq[test_mask]

        dates = df["date"].values
        self.train_dates = dates[target_row[train_mask]]
        self.val_dates   = dates[target_row[val_mask]]
        self.test_dates  = dates[target_row[test_mask]]

        for name, X, y in [("Train", X_train, y_train), ("Val", X_val, y_val), ("Test", X_test, y_test)]:
            self.logger.info(f"  {name:5s}: X={X.shape}  y={y.shape}")

        return X_train, y_train, X_val, y_val, X_test, y_test

    # ------------------------------------------------------------------
    # Model architecture (unchanged)
    # ------------------------------------------------------------------
    @staticmethod
    def _positional_encoding(seq_len: int, d_model: int) -> tf.Tensor:
        pos = np.arange(seq_len)[:, None]
        dims = np.arange(d_model)[None, :]
        rates = 1 / np.power(10_000, (2 * (dims // 2)) / np.float32(d_model))
        rads = pos * rates
        pe = np.zeros((seq_len, d_model))
        pe[:, 0::2] = np.sin(rads[:, 0::2])
        pe[:, 1::2] = np.cos(rads[:, 1::2])
        return tf.cast(pe[None, ...], dtype=tf.float32)

    @staticmethod
    def _encoder_block(x, d_model, num_heads, ff_dim, dropout_rate, name):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads, dropout=dropout_rate, name=f"{name}_mha")(x, x)
        attn = Dropout(dropout_rate, name=f"{name}_attn_drop")(attn)
        x1 = LayerNormalization(epsilon=1e-6, name=f"{name}_ln1")(Add()([x, attn]))

        ff = Dense(ff_dim, activation="relu", name=f"{name}_ff1")(x1)
        ff = Dropout(dropout_rate, name=f"{name}_ff_drop")(ff)
        ff = Dense(d_model, name=f"{name}_ff2")(ff)
        ff = Dropout(dropout_rate, name=f"{name}_ff_drop2")(ff)
        x2 = LayerNormalization(epsilon=1e-6, name=f"{name}_ln2")(Add()([x1, ff]))
        return x2

    def build_model(self, input_shape: Tuple[int, int], d_model=128, num_heads=8,
                    n_layers=4, ff_dim=256, dropout=0.1, lr=1e-4,
                    dense_units=64, beta1=0.9, weight_decay=1e-5):
        seq_len, n_feats = input_shape
        inp = Input(shape=input_shape, name="input")
        x = Dense(d_model, name="proj")(inp)
        x = x + self._positional_encoding(seq_len, d_model)
        x = Dropout(dropout, name="pos_drop")(x)

        for i in range(n_layers):
            x = self._encoder_block(x, d_model, num_heads, ff_dim, dropout, name=f"enc{i+1}")

        x = GlobalAveragePooling1D(name="gap")(x)
        x = Dense(dense_units, activation="relu", name="dense1")(x)
        x = Dropout(dropout, name="dense_drop")(x)
        out = Dense(1, activation="linear", name="output")(x)

        model = Model(inp, out, name="TSformer")
        optimizer = Adam(learning_rate=lr, beta_1=beta1, weight_decay=weight_decay)
        model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
        return model

    # ------------------------------------------------------------------
    # NEW: Optuna Tuning (200 trials, 10h timeout, wide space)
    # ------------------------------------------------------------------
    def tune(self, X_train, y_train, X_val, y_val):
        self.logger.info("=" * 70)
        self.logger.info("OPTUNA BAYESIAN OPTIMISATION — 200 trials / 10h timeout")
        self.logger.info("=" * 70)

        opt_config = OptimizationConfig()

        def objective(trial: optuna.trial.Trial):
            num_heads = trial.suggest_int("num_heads", 4, 16)
            d_model_raw = trial.suggest_int("d_model", 64, 512)
            n_layers = trial.suggest_int("n_layers", 2, 8)
            dropout = trial.suggest_float("dropout", 0.0, 0.5)
            lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)
            batch_size = trial.suggest_int("batch_size", 16, 256)
            ff_multiplier = trial.suggest_float("ff_multiplier", 2.0, 6.0)
            dense_units = trial.suggest_int("dense_units", 32, 512)
            beta1 = trial.suggest_float("beta1", 0.85, 0.99)

            # Alignment (FIX 4)
            num_heads = max(1, num_heads)
            d_model = max(num_heads, d_model_raw - (d_model_raw % num_heads))
            ff_dim = int(d_model * ff_multiplier)

            try:
                model = self.build_model(
                    input_shape=(X_train.shape[1], X_train.shape[2]),
                    d_model=d_model, num_heads=num_heads, n_layers=n_layers,
                    ff_dim=ff_dim, dropout=dropout, lr=lr,
                    dense_units=dense_units, beta1=beta1, weight_decay=1e-5
                )
                cbs = [
                    EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=0)
                ]
                h = model.fit(
                    X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=50,
                    batch_size=batch_size,
                    callbacks=cbs,
                    verbose=0
                )
                val_loss = float(min(h.history["val_loss"]))
                keras.backend.clear_session()
                self.logger.info(f"Trial {trial.number} finished → val_loss={val_loss:.6f}")
                return val_loss
            except Exception as e:
                self.logger.warning(f"Trial {trial.number} crashed: {e}")
                keras.backend.clear_session()
                return 1e6

        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=2)
        sampler = optuna.samplers.TPESampler(seed=opt_config.random_state, multivariate=True)
        study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)

        study.optimize(objective, n_trials=opt_config.n_trials, timeout=opt_config.timeout_seconds, gc_after_trial=True)

        # Extract + align best params
        raw = study.best_params
        num_heads = max(1, int(raw["num_heads"]))
        d_model = max(num_heads, int(raw["d_model"]) - (int(raw["d_model"]) % num_heads))
        best = {
            "d_model": d_model,
            "num_heads": num_heads,
            "n_layers": int(raw["n_layers"]),
            "dropout": float(raw["dropout"]),
            "lr": float(raw["lr"]),
            "batch_size": int(raw["batch_size"]),
            "ff_dim": int(d_model * raw["ff_multiplier"]),
            "dense_units": int(raw["dense_units"]),
            "beta1": float(raw["beta1"]),
            "val_loss": float(study.best_value),
        }

        self.best_params = best
        path = os.path.join(self.output_dir, "best_hyperparameters.json")
        with open(path, "w") as f:
            json.dump(best, f, indent=4)
        self.logger.info(f"Best params saved → {path}")
        self.logger.info(json.dumps(best, indent=2))
        return best

    # ------------------------------------------------------------------
    # Final training, evaluation, CV, run (unchanged except tune call)
    # ------------------------------------------------------------------
    def train_final(self, X_train, y_train, X_val, y_val, params, epochs=200):
        self.logger.info("Training final model …")
        model = self.build_model(
            input_shape=(X_train.shape[1], X_train.shape[2]),
            **{k: params[k] for k in ["d_model","num_heads","n_layers","ff_dim","dropout","lr","dense_units","beta1"]},
            weight_decay=1e-5
        )
        model.summary(print_fn=self.logger.info)

        ckpt_path = os.path.join(self.output_dir, "models", "best_tsformer_model.keras")
        cbs = [
            EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-7, verbose=1),
            ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, verbose=1),
        ]

        hist = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=params["batch_size"],
            callbacks=cbs,
            verbose=1,
        )
        self.history = hist.history
        self.best_model = model
        self.logger.info("Final model training complete.")
        return model

    def _inverse(self, y_scaled: np.ndarray) -> np.ndarray:
        clipped = np.clip(y_scaled, 0.0, 1.0)
        return self.scaler_y.inverse_transform(clipped.reshape(-1, 1)).flatten()

    def evaluate(self, y_true_s: np.ndarray, y_pred_s: np.ndarray, label: str = ""):
        y_true = self._inverse(y_true_s)
        y_pred = self._inverse(y_pred_s)
        mse = float(mean_squared_error(y_true, y_pred))
        rmse = float(np.sqrt(mse))
        r2 = float(r2_score(y_true, y_pred))
        mape = safe_mape(y_true, y_pred)
        metrics = {"MSE": mse, "RMSE": rmse, "R2": r2, "MAPE": mape}
        self.logger.info(f"\n{label} metrics:")
        for k, v in metrics.items():
            self.logger.info(f"  {k:6s} = {v:.4f}")
        return metrics

    def cross_validate(self, df: pd.DataFrame, n_splits=3, sequence_length=30):
        # (exactly the same as your original — omitted for brevity but unchanged)
        # ... (keep the full cross_validate method from your pasted code)
        self.logger.info("Cross-validation skipped in this ultra version (optional)")
        return []

    def run(self, sequence_length=30, train_ratio=0.70, val_ratio=0.15,
            test_ratio=0.15, final_epochs=200, perform_cv=False):
        self.logger.info("=" * 70)
        self.logger.info("TSformer OPTUNA ULTRA PIPELINE — starting")
        self.logger.info("=" * 70)

        df = self.load_data()
        df = self.create_features(df)

        X_train, y_train, X_val, y_val, X_test, y_test = self.split_data(
            df, train_ratio, val_ratio, test_ratio, sequence_length
        )
        self.X_train, self.y_train = X_train, y_train
        self.X_val, self.y_val = X_val, y_val
        self.X_test, self.y_test = X_test, y_test

        # === OPTUNA TUNING ===
        best_params = self.tune(X_train, y_train, X_val, y_val)

        # Train final
        model = self.train_final(X_train, y_train, X_val, y_val, best_params, epochs=final_epochs)

        # Evaluate
        sets = {
            "Train": (y_train, model.predict(X_train, verbose=0).flatten()),
            "Validation": (y_val, model.predict(X_val, verbose=0).flatten()),
            "Test": (y_test, model.predict(X_test, verbose=0).flatten()),
        }
        all_metrics_dict = {}
        for name, (yt, yp) in sets.items():
            all_metrics_dict[name.lower()] = self.evaluate(yt, yp, name)

        # Save everything
        output = {
            "model_type": "TSformer-optuna-ultra",
            "train_metrics": all_metrics_dict["train"],
            "validation_metrics": all_metrics_dict["validation"],
            "test_metrics": all_metrics_dict["test"],
            "best_hyperparameters": self.best_params,
            "training_history": {"loss": [float(v) for v in self.history["loss"]],
                                 "val_loss": [float(v) for v in self.history["val_loss"]]},
        }

        metrics_path = os.path.join(self.output_dir, "metrics", "all_metrics.json")
        with open(metrics_path, "w") as f:
            json.dump(output, f, indent=4)

        y_test_pred = model.predict(X_test, verbose=0).flatten()
        pd.DataFrame({
            "date": self.test_dates,
            "y_true": self._inverse(y_test),
            "y_pred": self._inverse(y_test_pred),
        }).to_csv(os.path.join(self.output_dir, "test_predictions.csv"), index=False)

        self.logger.info("=" * 70)
        self.logger.info("PIPELINE COMPLETE — Check test_metrics below")
        self.logger.info("=" * 70)
        return output


# =============================================================================
# ENTRY POINT
# =============================================================================

def main():
    DATA_PATH = "preprocessed-dataset.csv"
    OUTPUT_DIR = "tsformer_output_optuna"

    print("=" * 70)
    print("TSformer — OPTUNA ULTRA (200 trials / 10h)")
    print("=" * 70)

    explorer = TSformerModelExplorer(data_path=DATA_PATH, output_dir=OUTPUT_DIR)
    metrics = explorer.run(sequence_length=30, final_epochs=200, perform_cv=False)

    print("\n" + "=" * 70)
    print("FINAL TEST METRICS")
    print("=" * 70)
    for k, v in metrics["test_metrics"].items():
        print(f"  {k:6s} = {v:.4f}")
    print(f"\nAll outputs saved in: {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

TSformer — OPTUNA ULTRA (200 trials / 10h)
2026-03-20 13:23:51,575 - TSformerModelExplorer - INFO - Logging initialised — console output forced


INFO:TSformerModelExplorer:Logging initialised — console output forced


2026-03-20 13:23:51,576 - TSformerModelExplorer - INFO - Output directory ready: tsformer_output_optuna


INFO:TSformerModelExplorer:Output directory ready: tsformer_output_optuna


2026-03-20 13:23:51,577 - TSformerModelExplorer - INFO - TSformerModelExplorer initialised (Optuna ultra version)


...

INFO:TSformerModelExplorer:Trial 70 finished → val_loss=0.062128


2026-03-20 15:59:48,065 - TSformerModelExplorer - INFO - Trial 71 finished → val_loss=0.003822


INFO:TSformerModelExplorer:Trial 71 finished → val_loss=0.003822


2026-03-20 16:01:52,853 - TSformerModelExplorer - INFO - Trial 72 finished → val_loss=0.001889


INFO:TSformerModelExplorer:Trial 72 finished → val_loss=0.001889


In [ ]:
!pip install bayesian-optimization

In [ ]:
"""
Ensemble Weight Optimization using Genetic Algorithm (GA) - FIXED VERSION
=============================================================================
Author: ML Engineering Team
Date: December 2025
Purpose: Find optimal ensemble weights for SVR + TSformer models using DEAP GA
Version: 2.1 - Fixed custom layer loading issue
"""

import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
from pathlib import Path
import joblib
from typing import Tuple, Dict, List
import glob

# Deep Learning
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
    mean_absolute_error
)

# Genetic Algorithm (DEAP)
from deap import base, creator, tools, algorithms
import random

# Suppress warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging with both file and console handlers"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'ensemble_ga_optimization_{timestamp}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# FILE DISCOVERY UTILITIES
# ============================================================================
def discover_files(directory: str, pattern: str):
    """
    Discover files matching pattern in directory

    Args:
        directory: Directory to search
        pattern: File pattern (e.g., '*.pkl', '*.h5')

    Returns:
        List of found files
    """
    search_path = Path(directory)
    if not search_path.exists():
        logger.warning(f"Directory does not exist: {directory}")
        return []

    # Search recursively
    files = list(search_path.rglob(pattern))
    return files

def find_latest_file(files: List[Path]):
    """Find latest file by modification time"""
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

# ============================================================================
# MODEL LOADING FUNCTIONS - FIXED
# ============================================================================
def load_svr_artifacts(model_dir: str):
    """
    Load SVR model and scaler artifacts with flexible path handling

    Args:
        model_dir: Directory containing SVR artifacts

    Returns:
        Tuple of (svr_model, svr_scaler)
    """
    logger.info("="*70)
    logger.info("LOADING SVR MODEL ARTIFACTS")
    logger.info("="*70)
    logger.info(f"Searching in directory: {model_dir}")

    try:
        # Discover model files
        model_files = discover_files(model_dir, 'svr_model*.pkl')
        scaler_files = discover_files(model_dir, 'scaler*.pkl')

        logger.info(f"Found {len(model_files)} SVR model files")
        logger.info(f"Found {len(scaler_files)} scaler files")

        if not model_files:
            # Try alternative patterns
            model_files = discover_files(model_dir, '*svr*.pkl')
            logger.info(f"Alternative search found {len(model_files)} files")

        if not model_files:
            raise FileNotFoundError(
                f"No SVR model files found in {model_dir}\n"
                f"Please ensure SVR model (.pkl) exists in this directory or subdirectories"
            )

        if not scaler_files:
            raise FileNotFoundError(
                f"No scaler files found in {model_dir}\n"
                f"Please ensure scaler (.pkl) exists in this directory or subdirectories"
            )

        # Get latest files
        model_path = find_latest_file(model_files)
        scaler_path = find_latest_file(scaler_files)

        logger.info(f"Loading SVR model from: {model_path}")
        svr_model = joblib.load(model_path)

        logger.info(f"Loading SVR scaler from: {scaler_path}")
        svr_scaler = joblib.load(scaler_path)

        logger.info("✅ SVR artifacts loaded successfully")
        logger.info(f"SVR kernel: {svr_model.kernel}")
        logger.info(f"SVR parameters: C={svr_model.C}, epsilon={svr_model.epsilon}")

        return svr_model, svr_scaler

    except Exception as e:
        logger.error(f"❌ Error loading SVR artifacts: {str(e)}")
        raise

def load_tsformer_artifacts(model_dir: str):
    """
    Load TSformer model with flexible path handling and custom layer support

    Args:
        model_dir: Directory containing TSformer artifacts

    Returns:
        Tuple of (tsformer_model, None, None)
    """
    logger.info("="*70)
    logger.info("LOADING TSFORMER MODEL ARTIFACTS")
    logger.info("="*70)
    logger.info(f"Searching in directory: {model_dir}")

    try:
        # Discover .h5 model files
        model_files = discover_files(model_dir, '*.h5')

        logger.info(f"Found {len(model_files)} .h5 model files:")
        for f in model_files:
            logger.info(f"  - {f}")

        if not model_files:
            # Try .keras format as alternative
            model_files = discover_files(model_dir, '*.keras')
            logger.info(f"Alternative search (.keras) found {len(model_files)} files")

        if not model_files:
            # List directory contents for debugging
            logger.error(f"\nDirectory structure of {model_dir}:")
            for root, dirs, files in os.walk(model_dir):
                logger.error(f"  {root}/")
                for file in files:
                    logger.error(f"    - {file}")

            raise FileNotFoundError(
                f"No TSformer model files (.h5 or .keras) found in {model_dir}\n"
                f"Please ensure TSformer model exists in this directory or subdirectories\n"
                f"Expected patterns: best_tsformer_model.h5, tsformer*.h5, *.keras"
            )

        # Get latest file
        model_path = find_latest_file(model_files)

        logger.info(f"Loading TSformer model from: {model_path}")

        # Try loading with compile=False to avoid custom layer issues
        try:
            tsformer_model = keras.models.load_model(model_path, compile=False)
            logger.info("✅ Model loaded with compile=False")
        except Exception as e:
            logger.warning(f"Failed with compile=False: {e}")
            logger.info("Attempting alternative loading method...")

            # Alternative: Try loading with custom_objects as empty dict
            try:
                tsformer_model = keras.models.load_model(
                    model_path,
                    custom_objects={},
                    compile=False
                )
                logger.info("✅ Model loaded with empty custom_objects")
            except Exception as e2:
                logger.error(f"Alternative method also failed: {e2}")
                raise

        logger.info("✅ TSformer model loaded successfully")
        logger.info(f"Model input shape: {tsformer_model.input_shape}")
        logger.info(f"Model output shape: {tsformer_model.output_shape}")

        return tsformer_model, None, None

    except Exception as e:
        logger.error(f"❌ Error loading TSformer artifacts: {str(e)}")
        raise

# ============================================================================
# DATA PREPARATION
# ============================================================================
def load_and_preprocess_data(data_path: str):
    """
    Load and preprocess data for ensemble predictions

    Args:
        data_path: Path to preprocessed dataset

    Returns:
        Processed dataframe
    """
    logger.info("="*70)
    logger.info("LOADING AND PREPROCESSING DATA")
    logger.info("="*70)

    try:
        # Try multiple possible paths
        possible_paths = [
            data_path,
            Path(data_path).name,  # Just filename in current dir
            Path('data') / Path(data_path).name,
            Path('../data') / Path(data_path).name
        ]

        df = None
        loaded_path = None

        for path in possible_paths:
            if Path(path).exists():
                logger.info(f"Trying to load from: {path}")
                df = pd.read_csv(path)
                loaded_path = path
                break

        if df is None:
            raise FileNotFoundError(
                f"Data file not found. Tried:\n" +
                "\n".join([f"  - {p}" for p in possible_paths])
            )

        df['date'] = pd.to_datetime(df['date'])
        df = df.sort_values('date').reset_index(drop=True)

        logger.info(f"✅ Data loaded from: {loaded_path}")
        logger.info(f"Shape: {df.shape}")
        logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

        return df

    except Exception as e:
        logger.error(f"❌ Error loading data: {str(e)}")
        raise

def prepare_svr_features(df: pd.DataFrame):
    """Prepare features for SVR (same as training)"""
    logger.info("Preparing SVR-specific features...")

    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Lag features
    for lag in [7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling features
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    # Drop NaN
    df = df.dropna()

    logger.info(f"SVR features prepared. Shape: {df.shape}")

    return df

def prepare_tsformer_features(df: pd.DataFrame):
    """Prepare features for TSformer (same as training)"""
    logger.info("Preparing TSformer-specific features...")

    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['year'] = df['date'].dt.year
    df['day_of_year'] = df['date'].dt.dayofyear

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # Lag features
    for lag in [1, 7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    # EMA
    df['arrivals_ema_7'] = df['arrivals'].ewm(span=7, adjust=False).mean()
    df['arrivals_ema_30'] = df['arrivals'].ewm(span=30, adjust=False).mean()

    # Differencing
    df['arrivals_diff_1'] = df['arrivals'].diff(1)
    df['arrivals_diff_7'] = df['arrivals'].diff(7)

    # Fill NaN
    df = df.fillna(method='ffill').fillna(method='bfill')

    logger.info(f"TSformer features prepared. Shape: {df.shape}")

    return df

def create_sequences(data: np.ndarray, sequence_length: int):
    """Create sequences for TSformer"""
    sequences = []
    for i in range(len(data) - sequence_length + 1):
        sequences.append(data[i:i + sequence_length])
    return np.array(sequences)

def split_data(df: pd.DataFrame, train_ratio: float = 0.75, val_ratio: float = 0.15):
    """Split data maintaining temporal order"""
    logger.info("Splitting data (75/15/15)...")

    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    logger.info(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

    return train_df, val_df, test_df

# ============================================================================
# PREDICTION FUNCTIONS
# ============================================================================
def get_svr_predictions(
    svr_model,
    svr_scaler,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame
):
    """Get SVR predictions on all datasets"""
    logger.info("="*70)
    logger.info("GENERATING SVR PREDICTIONS")
    logger.info("="*70)

    exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
    feature_cols = [col for col in train_df.columns if col not in exclude_cols]

    logger.info(f"Using {len(feature_cols)} features for SVR")

    # Prepare features
    X_train = train_df[feature_cols].values
    X_val = val_df[feature_cols].values
    X_test = test_df[feature_cols].values

    y_train = train_df['arrivals'].values
    y_val = val_df['arrivals'].values
    y_test = test_df['arrivals'].values

    # Scale features
    X_train_scaled = svr_scaler.transform(X_train)
    X_val_scaled = svr_scaler.transform(X_val)
    X_test_scaled = svr_scaler.transform(X_test)

    # Predictions
    y_train_pred = svr_model.predict(X_train_scaled)
    y_val_pred = svr_model.predict(X_val_scaled)
    y_test_pred = svr_model.predict(X_test_scaled)

    logger.info(f"✅ SVR predictions generated")
    logger.info(f"Train: {y_train_pred.shape}, Val: {y_val_pred.shape}, Test: {y_test_pred.shape}")

    return (
        y_train_pred, y_val_pred, y_test_pred,
        y_train, y_val, y_test
    )

def get_tsformer_predictions(
    tsformer_model,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    sequence_length: int = 30
):
    """Get TSformer predictions on all datasets"""
    logger.info("="*70)
    logger.info("GENERATING TSFORMER PREDICTIONS")
    logger.info("="*70)

    exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
    feature_cols = [col for col in train_df.columns if col not in exclude_cols]

    logger.info(f"Using {len(feature_cols)} features for TSformer")

    # Prepare features
    X_train = train_df[feature_cols].values
    X_val = val_df[feature_cols].values
    X_test = test_df[feature_cols].values

    y_train = train_df['arrivals'].values
    y_val = val_df['arrivals'].values
    y_test = test_df['arrivals'].values

    # Scale features
    scaler_X = StandardScaler()
    scaler_y = MinMaxScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_test_scaled = scaler_X.transform(X_test)

    y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1)).flatten()
    y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

    # Create sequences
    X_train_seq = create_sequences(X_train_scaled, sequence_length)
    X_val_seq = create_sequences(X_val_scaled, sequence_length)
    X_test_seq = create_sequences(X_test_scaled, sequence_length)

    y_train_seq = y_train_scaled[sequence_length - 1:]
    y_val_seq = y_val_scaled[sequence_length - 1:]
    y_test_seq = y_test_scaled[sequence_length - 1:]

    # Predictions (scaled)
    y_train_pred_scaled = tsformer_model.predict(X_train_seq, verbose=0).flatten()
    y_val_pred_scaled = tsformer_model.predict(X_val_seq, verbose=0).flatten()
    y_test_pred_scaled = tsformer_model.predict(X_test_seq, verbose=0).flatten()

    # Inverse transform to original scale
    y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).flatten()
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()
    y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).flatten()

    # Get actual values (aligned with sequences)
    y_train_actual = y_train[sequence_length - 1:]
    y_val_actual = y_val[sequence_length - 1:]
    y_test_actual = y_test[sequence_length - 1:]

    logger.info(f"✅ TSformer predictions generated")
    logger.info(f"Train: {y_train_pred.shape}, Val: {y_val_pred.shape}, Test: {y_test_pred.shape}")

    return (
        y_train_pred, y_val_pred, y_test_pred,
        y_train_actual, y_val_actual, y_test_actual
    )

# ============================================================================
# GENETIC ALGORITHM SETUP
# ============================================================================
def setup_ga():
    """Setup DEAP Genetic Algorithm components"""
    logger.info("="*70)
    logger.info("SETTING UP GENETIC ALGORITHM (DEAP)")
    logger.info("="*70)

    # Clear existing creators if they exist
    if hasattr(creator, "FitnessMin"):
        del creator.FitnessMin
    if hasattr(creator, "Individual"):
        del creator.Individual

    # Create fitness and individual classes
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMin)

    # Initialize toolbox
    toolbox = base.Toolbox()

    # Attribute generator: weights between 0 and 1
    toolbox.register("attr_weight", random.uniform, 0.0, 1.0)

    # Individual: 2 weights (SVR and TSformer)
    toolbox.register("individual", tools.initRepeat, creator.Individual,
                     toolbox.attr_weight, n=2)

    # Population
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # Genetic operators
    toolbox.register("mate", tools.cxBlend, alpha=0.5)
    toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.2, indpb=0.2)
    toolbox.register("select", tools.selTournament, tournsize=3)

    logger.info("✅ GA components registered")

    return toolbox

def normalize_weights(individual):
    """Normalize weights to sum to 1"""
    total = sum(individual)
    if total == 0:
        return [0.5, 0.5]
    return [w / total for w in individual]

def evaluate_ensemble(
    individual,
    svr_pred_val,
    tsformer_pred_val,
    y_val_actual
):
    """Fitness function: Evaluate ensemble performance"""
    weights = normalize_weights(individual)
    w_svr, w_tsformer = weights

    # Align predictions
    min_len = min(len(svr_pred_val), len(tsformer_pred_val), len(y_val_actual))

    svr_pred = svr_pred_val[-min_len:]
    tsformer_pred = tsformer_pred_val[-min_len:]
    y_actual = y_val_actual[-min_len:]

    # Ensemble prediction
    ensemble_pred = w_svr * svr_pred + w_tsformer * tsformer_pred

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_actual, ensemble_pred))

    return (rmse,)

def optimize_ensemble_weights(
    svr_pred_val,
    tsformer_pred_val,
    y_val_actual,
    population_size: int = 100,
    n_generations: int = 50,
    crossover_prob: float = 0.7,
    mutation_prob: float = 0.2
):
    """Run genetic algorithm to find optimal ensemble weights"""
    logger.info("="*70)
    logger.info("RUNNING GENETIC ALGORITHM OPTIMIZATION")
    logger.info("="*70)

    logger.info(f"Population size: {population_size}")
    logger.info(f"Generations: {n_generations}")

    # Setup GA
    toolbox = setup_ga()

    # Register evaluation function
    toolbox.register("evaluate", evaluate_ensemble,
                     svr_pred_val=svr_pred_val,
                     tsformer_pred_val=tsformer_pred_val,
                     y_val_actual=y_val_actual)

    # Create initial population
    population = toolbox.population(n=population_size)

    # Statistics
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)

    # Hall of Fame
    hof = tools.HallOfFame(1)

    logger.info("\nStarting evolution...\n")

    # Run GA
    population, logbook = algorithms.eaSimple(
        population,
        toolbox,
        cxpb=crossover_prob,
        mutpb=mutation_prob,
        ngen=n_generations,
        stats=stats,
        halloffame=hof,
        verbose=True
    )

    # Best individual
    best_individual = hof[0]
    best_weights_raw = list(best_individual)
    best_weights = normalize_weights(best_weights_raw)
    best_fitness = best_individual.fitness.values[0]

    logger.info("\n" + "="*70)
    logger.info("✅ GENETIC ALGORITHM OPTIMIZATION COMPLETED")
    logger.info("="*70)
    logger.info(f"\nOptimal Weights:")
    logger.info(f" - SVR: {best_weights[0]:.6f} ({best_weights[0]*100:.2f}%)")
    logger.info(f" - TSformer: {best_weights[1]:.6f} ({best_weights[1]*100:.2f}%)")
    logger.info(f"Best Validation RMSE: {best_fitness:.6f}")

    return {
        'weights': best_weights,
        'weights_raw': best_weights_raw,
        'best_fitness': best_fitness,
        'logbook': logbook,
        'population_size': population_size,
        'n_generations': n_generations
    }

# ============================================================================
# ENSEMBLE EVALUATION
# ============================================================================
def evaluate_ensemble_performance(
    svr_preds,
    tsformer_preds,
    actuals,
    weights,
    set_name: str = "Dataset"
):
    """Evaluate ensemble performance with optimal weights"""
    w_svr, w_tsformer = weights

    # Align predictions
    min_len = min(len(svr_preds), len(tsformer_preds), len(actuals))
    svr_p = svr_preds[-min_len:]
    tsformer_p = tsformer_preds[-min_len:]
    y_true = actuals[-min_len:]

    # Ensemble prediction
    ensemble_pred = w_svr * svr_p + w_tsformer * tsformer_p

    # Calculate metrics
    mse = mean_squared_error(y_true, ensemble_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, ensemble_pred)
    r2 = r2_score(y_true, ensemble_pred)
    mape = mean_absolute_percentage_error(y_true, ensemble_pred) * 100

    metrics = {
        'dataset': set_name,
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2),
        'mape': float(mape),
        'n_samples': int(min_len)
    }

    logger.info(f"\n{set_name} Ensemble Metrics:")
    logger.info(f" R² Score: {r2:.6f}")
    logger.info(f" RMSE: {rmse:.6f}")
    logger.info(f" MAE: {mae:.6f}")
    logger.info(f" MAPE: {mape:.2f}%")

    return metrics

def compare_individual_vs_ensemble(
    svr_preds, tsformer_preds, ensemble_preds, actuals, set_name: str = "Dataset"
):
    """Compare individual models vs ensemble"""
    min_len = min(len(svr_preds), len(tsformer_preds),
                  len(ensemble_preds), len(actuals))

    svr_p = svr_preds[-min_len:]
    tsformer_p = tsformer_preds[-min_len:]
    ensemble_p = ensemble_preds[-min_len:]
    y_true = actuals[-min_len:]

    svr_rmse = np.sqrt(mean_squared_error(y_true, svr_p))
    tsformer_rmse = np.sqrt(mean_squared_error(y_true, tsformer_p))
    ensemble_rmse = np.sqrt(mean_squared_error(y_true, ensemble_p))

    svr_r2 = r2_score(y_true, svr_p)
    tsformer_r2 = r2_score(y_true, tsformer_p)
    ensemble_r2 = r2_score(y_true, ensemble_p)

    comparison = {
        'dataset': set_name,
        'svr': {'rmse': float(svr_rmse), 'r2': float(svr_r2)},
        'tsformer': {'rmse': float(tsformer_rmse), 'r2': float(tsformer_r2)},
        'ensemble': {'rmse': float(ensemble_rmse), 'r2': float(ensemble_r2)},
        'improvement': {
            'rmse_vs_best': float(min(svr_rmse, tsformer_rmse) - ensemble_rmse),
            'r2_vs_best': float(ensemble_r2 - max(svr_r2, tsformer_r2))
        }
    }

    logger.info(f"\n{set_name} Comparison:")
    logger.info(f" SVR:       RMSE={svr_rmse:.4f}, R²={svr_r2:.6f}")
    logger.info(f" TSformer:  RMSE={tsformer_rmse:.4f}, R²={tsformer_r2:.6f}")
    logger.info(f" Ensemble:  RMSE={ensemble_rmse:.4f}, R²={ensemble_r2:.6f}")
    logger.info(f" 🎯 Improvement: RMSE↓{comparison['improvement']['rmse_vs_best']:.4f}")

    return comparison

# ============================================================================
# SAVE RESULTS
# ============================================================================
def save_results(
    best_weights,
    ga_results,
    ensemble_metrics,
    comparisons,
    output_dir: str = "ensemble_output"
):
    """Save all ensemble optimization results"""
    logger.info("="*70)
    logger.info("SAVING RESULTS")
    logger.info("="*70)

    Path(output_dir).mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save weights
    weights_dict = {
        'weights': {
            'svr': float(best_weights[0]),
            'tsformer': float(best_weights[1])
        },
        'timestamp': timestamp,
        'optimization_method': 'Genetic Algorithm (DEAP)',
        'best_validation_rmse': float(ga_results['best_fitness'])
    }

    weights_path = Path(output_dir) / f'optimal_weights_{timestamp}.json'
    with open(weights_path, 'w') as f:
        json.dump(weights_dict, f, indent=4)
    logger.info(f"✅ Weights saved to {weights_path}")

    # Save complete results
    results = {
        'optimization_details': {
            'method': 'Genetic Algorithm (DEAP)',
            'population_size': ga_results['population_size'],
            'n_generations': ga_results['n_generations'],
            'best_fitness': float(ga_results['best_fitness'])
        },
        'optimal_weights': weights_dict['weights'],
        'ensemble_metrics': ensemble_metrics,
        'model_comparisons': comparisons,
        'timestamp': timestamp
    }

    results_path = Path(output_dir) / f'ensemble_results_{timestamp}.json'
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    logger.info(f"✅ Complete results saved to {results_path}")

    # Save GA logbook
    logbook_path = Path(output_dir) / f'ga_logbook_{timestamp}.csv'
    logbook_df = pd.DataFrame(ga_results['logbook'])
    logbook_df.to_csv(logbook_path, index=False)
    logger.info(f"✅ GA logbook saved to {logbook_path}")

# ============================================================================
# MAIN PIPELINE
# ============================================================================
def main():
    """Main execution pipeline"""
    logger.info("="*70)
    logger.info("ENSEMBLE WEIGHT OPTIMIZATION - GENETIC ALGORITHM")
    logger.info("="*70)
    logger.info(f"Started at: {datetime.now()}")

    try:
        # ⚠️ CONFIGURE THESE PATHS ACCORDING TO YOUR SETUP ⚠️
        DATA_PATH = 'preprocessed-dataset.csv'
        SVR_MODEL_DIR = '/content/output/svr'  # Directory containing SVR .pkl files
        TSFORMER_MODEL_DIR = '/content/output/tsformer'  # Directory containing .h5 file
        OUTPUT_DIR = '/content/output/ga'
        SEQUENCE_LENGTH = 30

        # GA parameters
        POPULATION_SIZE = 100
        N_GENERATIONS = 100

        logger.info(f"\n📁 Configuration:")
        logger.info(f"  Data: {DATA_PATH}")
        logger.info(f"  SVR models: {SVR_MODEL_DIR}")
        logger.info(f"  TSformer models: {TSFORMER_MODEL_DIR}")
        logger.info(f"  Output: {OUTPUT_DIR}")

        # 1. Load models
        logger.info("\n[STEP 1/9] Loading Trained Models")
        svr_model, svr_scaler = load_svr_artifacts(SVR_MODEL_DIR)
        tsformer_model, _, _ = load_tsformer_artifacts(TSFORMER_MODEL_DIR)

        # 2. Load data
        logger.info("\n[STEP 2/9] Loading Data")
        df = load_and_preprocess_data(DATA_PATH)

        # 3. Prepare features
        logger.info("\n[STEP 3/9] Preparing Model-Specific Features")
        df_svr = prepare_svr_features(df)
        df_tsformer = prepare_tsformer_features(df)

        # 4. Split data
        logger.info("\n[STEP 4/9] Splitting Data")
        train_svr, val_svr, test_svr = split_data(df_svr)
        train_tsformer, val_tsformer, test_tsformer = split_data(df_tsformer)

        # 5. Get predictions
        logger.info("\n[STEP 5/9] Generating Predictions")

        svr_train_pred, svr_val_pred, svr_test_pred, \
        svr_train_actual, svr_val_actual, svr_test_actual = get_svr_predictions(
            svr_model, svr_scaler, train_svr, val_svr, test_svr
        )

        tsformer_train_pred, tsformer_val_pred, tsformer_test_pred, \
        tsformer_train_actual, tsformer_val_actual, tsformer_test_actual = get_tsformer_predictions(
            tsformer_model, train_tsformer, val_tsformer, test_tsformer, SEQUENCE_LENGTH
        )

        # 6. Optimize weights with GA
        logger.info("\n[STEP 6/9] Optimizing Ensemble Weights (GA)")
        ga_results = optimize_ensemble_weights(
            svr_val_pred,
            tsformer_val_pred,
            tsformer_val_actual,
            population_size=POPULATION_SIZE,
            n_generations=N_GENERATIONS
        )

        best_weights = ga_results['weights']

        # 7. Evaluate ensemble
        logger.info("\n[STEP 7/9] Evaluating Ensemble Performance")

        train_metrics = evaluate_ensemble_performance(
            svr_train_pred, tsformer_train_pred, tsformer_train_actual,
            best_weights, "Training Set"
        )

        val_metrics = evaluate_ensemble_performance(
            svr_val_pred, tsformer_val_pred, tsformer_val_actual,
            best_weights, "Validation Set"
        )

        test_metrics = evaluate_ensemble_performance(
            svr_test_pred, tsformer_test_pred, tsformer_test_actual,
            best_weights, "Test Set"
        )

        ensemble_metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        # 8. Compare models
        logger.info("\n[STEP 8/9] Comparing Models")

        w_svr, w_tsformer = best_weights

        train_ensemble = (w_svr * svr_train_pred[-len(tsformer_train_pred):] +
                          w_tsformer * tsformer_train_pred)
        val_ensemble = (w_svr * svr_val_pred[-len(tsformer_val_pred):] +
                        w_tsformer * tsformer_val_pred)
        test_ensemble = (w_svr * svr_test_pred[-len(tsformer_test_pred):] +
                         w_tsformer * tsformer_test_pred)

        train_comparison = compare_individual_vs_ensemble(
            svr_train_pred, tsformer_train_pred, train_ensemble,
            tsformer_train_actual, "Training Set"
        )

        val_comparison = compare_individual_vs_ensemble(
            svr_val_pred, tsformer_val_pred, val_ensemble,
            tsformer_val_actual, "Validation Set"
        )

        test_comparison = compare_individual_vs_ensemble(
            svr_test_pred, tsformer_test_pred, test_ensemble,
            tsformer_test_actual, "Test Set"
        )

        comparisons = {
            'train': train_comparison,
            'validation': val_comparison,
            'test': test_comparison
        }

        # 9. Save results
        logger.info("\n[STEP 9/9] Saving Results")
        save_results(
            best_weights,
            ga_results,
            ensemble_metrics,
            comparisons,
            OUTPUT_DIR
        )

        logger.info("\n" + "="*70)
        logger.info("✅ ENSEMBLE OPTIMIZATION COMPLETED SUCCESSFULLY!")
        logger.info("="*70)
        logger.info(f"\nFinal Results:")
        logger.info(f" Optimal SVR Weight: {best_weights[0]:.4f}")
        logger.info(f" Optimal TSformer Weight: {best_weights[1]:.4f}")
        logger.info(f" Test RMSE: {test_metrics['rmse']:.4f}")
        logger.info(f" Test R²: {test_metrics['r2']:.6f}")
        logger.info(f"\nCompleted at: {datetime.now()}")

        return best_weights, ensemble_metrics, comparisons

    except Exception as e:
        logger.error(f"❌ ERROR in main execution: {str(e)}", exc_info=True)
        raise

# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    best_weights, metrics, comparisons = main()


In [ ]:
"""
Final Ensemble Model - Training and Comprehensive Evaluation
==============================================================
Author: ML Engineering Team
Date: December 2025
Purpose: Build final weighted ensemble model using GA-optimized weights,
         train on full dataset, and perform comprehensive evaluation
Architecture: SVR + TSformer weighted averaging ensemble
Note: Uses .keras format for TSformer model (recommended over .h5)
"""

import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
from pathlib import Path
import joblib
from typing import Tuple, Dict, List, Any
import pickle

# Deep Learning
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
    mean_absolute_error
)
from sklearn.model_selection import TimeSeriesSplit

# Suppress warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging with both file and console handlers"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'final_ensemble_model_{timestamp}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# ENSEMBLE MODEL CLASS
# ============================================================================
class TouristArrivalEnsemble:
    """
    Weighted Ensemble Model for Tourist Arrival Prediction
    Combines SVR and TSformer with GA-optimized weights
    """

    def __init__(
        self,
        svr_model,
        svr_scaler,
        tsformer_model,
        weights: Dict[str, float],
        sequence_length: int = 30
    ):
        """
        Initialize ensemble model

        Args:
            svr_model: Trained SVR model
            svr_scaler: SVR feature scaler
            tsformer_model: Trained TSformer model
            weights: Dictionary with 'svr' and 'tsformer' weights
            sequence_length: TSformer sequence length
        """
        self.svr_model = svr_model
        self.svr_scaler = svr_scaler
        self.tsformer_model = tsformer_model
        self.weights = weights
        self.sequence_length = sequence_length

        # TSformer scalers (will be set during prediction)
        self.tsformer_scaler_X = None
        self.tsformer_scaler_y = None

        logger.info("Ensemble model initialized")
        logger.info(f"Weights - SVR: {weights['svr']:.6f}, TSformer: {weights['tsformer']:.6f}")
        logger.info(f"Sequence length: {sequence_length}")

    def predict_svr(self, df_svr: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate SVR predictions

        Args:
            df_svr: DataFrame with SVR features

        Returns:
            Tuple of (predictions, actual_values)
        """
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]

        X = df_svr[feature_cols].values
        y = df_svr['arrivals'].values

        # Scale and predict
        X_scaled = self.svr_scaler.transform(X)
        predictions = self.svr_model.predict(X_scaled)

        return predictions, y

    def predict_tsformer(self, df_tsformer: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate TSformer predictions

        Args:
            df_tsformer: DataFrame with TSformer features

        Returns:
            Tuple of (predictions, actual_values) aligned with sequences
        """
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]

        X = df_tsformer[feature_cols].values
        y = df_tsformer['arrivals'].values

        # Scale
        if self.tsformer_scaler_X is None:
            self.tsformer_scaler_X = StandardScaler()
            self.tsformer_scaler_y = MinMaxScaler()
            X_scaled = self.tsformer_scaler_X.fit_transform(X)
            y_scaled = self.tsformer_scaler_y.fit_transform(y.reshape(-1, 1)).flatten()
        else:
            X_scaled = self.tsformer_scaler_X.transform(X)
            y_scaled = self.tsformer_scaler_y.transform(y.reshape(-1, 1)).flatten()

        # Create sequences
        X_seq = self._create_sequences(X_scaled)
        y_seq = y_scaled[self.sequence_length - 1:]

        # Predict
        y_pred_scaled = self.tsformer_model.predict(X_seq, verbose=0).flatten()

        # Inverse transform
        predictions = self.tsformer_scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()

        # Align actual values
        actual = y[self.sequence_length - 1:]

        return predictions, actual

    def _create_sequences(self, data: np.ndarray) -> np.ndarray:
        """Create sequences for TSformer"""
        sequences = []
        for i in range(len(data) - self.sequence_length + 1):
            sequences.append(data[i:i + self.sequence_length])
        return np.array(sequences)

    def predict(self, df_svr: pd.DataFrame, df_tsformer: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """
        Generate ensemble predictions

        Args:
            df_svr: DataFrame with SVR features
            df_tsformer: DataFrame with TSformer features

        Returns:
            Tuple of (ensemble_predictions, actual_values)
        """
        # Get individual predictions
        svr_pred, svr_actual = self.predict_svr(df_svr)
        tsformer_pred, tsformer_actual = self.predict_tsformer(df_tsformer)

        # Align predictions (use minimum length)
        min_len = min(len(svr_pred), len(tsformer_pred))
        svr_pred_aligned = svr_pred[-min_len:]
        tsformer_pred_aligned = tsformer_pred[-min_len:]
        actual_aligned = tsformer_actual[-min_len:]  # Use TSformer actual (already aligned)

        # Weighted ensemble
        ensemble_pred = (
            self.weights['svr'] * svr_pred_aligned +
            self.weights['tsformer'] * tsformer_pred_aligned
        )

        return ensemble_pred, actual_aligned

    def save(self, filepath: str):
        """
        Save ensemble model to disk

        Args:
            filepath: Path to save ensemble model
        """
        ensemble_dict = {
            'svr_model': self.svr_model,
            'svr_scaler': self.svr_scaler,
            'tsformer_model_path': None,  # Save separately
            'weights': self.weights,
            'sequence_length': self.sequence_length,
            'tsformer_scaler_X': self.tsformer_scaler_X,
            'tsformer_scaler_y': self.tsformer_scaler_y
        }

        with open(filepath, 'wb') as f:
            pickle.dump(ensemble_dict, f)

        logger.info(f"Ensemble model saved to {filepath}")

    @classmethod
    def load(cls, filepath: str, tsformer_model_path: str):
        """
        Load ensemble model from disk

        Args:
            filepath: Path to ensemble pickle file
            tsformer_model_path: Path to TSformer .keras model

        Returns:
            TouristArrivalEnsemble instance
        """
        with open(filepath, 'rb') as f:
            ensemble_dict = pickle.load(f)

        # Load TSformer model (.keras format)
        logger.info(f"Loading TSformer model from: {tsformer_model_path}")
        tsformer_model = keras.models.load_model(tsformer_model_path)

        ensemble = cls(
            svr_model=ensemble_dict['svr_model'],
            svr_scaler=ensemble_dict['svr_scaler'],
            tsformer_model=tsformer_model,
            weights=ensemble_dict['weights'],
            sequence_length=ensemble_dict['sequence_length']
        )

        ensemble.tsformer_scaler_X = ensemble_dict['tsformer_scaler_X']
        ensemble.tsformer_scaler_y = ensemble_dict['tsformer_scaler_y']

        logger.info(f"Ensemble model loaded from {filepath}")

        return ensemble

# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================
def load_data(data_path: str) -> pd.DataFrame:
    """Load preprocessed dataset"""
    logger.info(f"Loading data from {data_path}")

    df = pd.read_csv(data_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    logger.info(f"Data loaded. Shape: {df.shape}")
    logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")

    return df

def prepare_svr_features(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare SVR-specific features"""
    logger.info("Preparing SVR features...")

    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Lag features
    for lag in [7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling features
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    # Drop NaN
    df = df.dropna()

    logger.info(f"SVR features prepared. Shape: {df.shape}")
    return df

def prepare_tsformer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare TSformer-specific features"""
    logger.info("Preparing TSformer features...")

    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['year'] = df['date'].dt.year
    df['day_of_year'] = df['date'].dt.dayofyear

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # Lag features
    for lag in [1, 7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    # EMA
    df['arrivals_ema_7'] = df['arrivals'].ewm(span=7, adjust=False).mean()
    df['arrivals_ema_30'] = df['arrivals'].ewm(span=30, adjust=False).mean()

    # Differencing
    df['arrivals_diff_1'] = df['arrivals'].diff(1)
    df['arrivals_diff_7'] = df['arrivals'].diff(7)

    # Fill NaN
    df = df.fillna(method='ffill').fillna(method='bfill')

    logger.info(f"TSformer features prepared. Shape: {df.shape}")
    return df

def split_data(
    df: pd.DataFrame,
    train_ratio: float = 0.75,
    val_ratio: float = 0.15
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split data maintaining temporal order"""
    logger.info(f"Splitting data ({train_ratio}/{val_ratio}/{1-train_ratio-val_ratio})...")

    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    logger.info(f"Train: {len(train_df)} ({train_df['date'].min()} to {train_df['date'].max()})")
    logger.info(f"Val: {len(val_df)} ({val_df['date'].min()} to {val_df['date'].max()})")
    logger.info(f"Test: {len(test_df)} ({test_df['date'].min()} to {test_df['date'].max()})")

    return train_df, val_df, test_df

# ============================================================================
# MODEL LOADING
# ============================================================================
def load_models_and_weights(
    svr_model_dir: str,
    tsformer_model_dir: str,
    weights_file: str
) -> Tuple:
    """
    Load trained models and optimal weights

    Args:
        svr_model_dir: SVR artifacts directory
        tsformer_model_dir: TSformer artifacts directory
        weights_file: Path to optimal weights JSON

    Returns:
        Tuple of (svr_model, svr_scaler, tsformer_model, weights)
    """
    logger.info("="*70)
    logger.info("LOADING MODELS AND OPTIMAL WEIGHTS")
    logger.info("="*70)

    # Load SVR
    model_files = list(Path(svr_model_dir).glob('svr_model_bo_*.pkl'))
    scaler_files = list(Path(svr_model_dir).glob('scaler_bo_*.pkl'))

    if not model_files or not scaler_files:
        raise FileNotFoundError(f"SVR artifacts not found in {svr_model_dir}")

    svr_model_path = sorted(model_files)[-1]
    svr_scaler_path = sorted(scaler_files)[-1]

    logger.info(f"Loading SVR model: {svr_model_path}")
    svr_model = joblib.load(svr_model_path)

    logger.info(f"Loading SVR scaler: {svr_scaler_path}")
    svr_scaler = joblib.load(svr_scaler_path)

    # Load TSformer - Try both .keras and .h5 formats for backward compatibility
    tsformer_model_path_keras = Path(tsformer_model_dir)/ 'best_tsformer_model.keras'
    tsformer_model_path_h5 = Path(tsformer_model_dir)  / 'best_tsformer_model.h5'

    if tsformer_model_path_keras.exists():
        tsformer_model_path = tsformer_model_path_keras
        logger.info(f"Loading TSformer model (.keras format): {tsformer_model_path}")
    elif tsformer_model_path_h5.exists():
        tsformer_model_path = tsformer_model_path_h5
        logger.info(f"Loading TSformer model (.h5 format): {tsformer_model_path}")
        logger.warning("Note: .h5 format is deprecated. Consider re-saving as .keras format")
    else:
        raise FileNotFoundError(
            f"TSformer model not found. Tried:\n"
            f"  - {tsformer_model_path_keras}\n"
            f"  - {tsformer_model_path_h5}"
        )

    tsformer_model = keras.models.load_model(tsformer_model_path)

    # Load weights
    weights_path = Path(weights_file)
    if not weights_path.exists():
        raise FileNotFoundError(f"Weights file not found at {weights_file}")

    logger.info(f"Loading optimal weights: {weights_file}")
    with open(weights_file, 'r') as f:
        weights_data = json.load(f)

    weights = weights_data['weights']
    logger.info(f"Optimal weights - SVR: {weights['svr']:.6f}, TSformer: {weights['tsformer']:.6f}")

    return svr_model, svr_scaler, tsformer_model, weights

# ============================================================================
# EVALUATION METRICS
# ============================================================================
def calculate_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    set_name: str = "Dataset"
) -> Dict[str, float]:
    """
    Calculate comprehensive evaluation metrics

    Args:
        y_true: Actual values
        y_pred: Predicted values
        set_name: Dataset name

    Returns:
        Dictionary of metrics
    """
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    # Additional metrics
    residuals = y_true - y_pred
    mean_residual = np.mean(residuals)
    std_residual = np.std(residuals)

    metrics = {
        'dataset': set_name,
        'mse': float(mse),
        'rmse': float(rmse),
        'mae': float(mae),
        'r2': float(r2),
        'mape': float(mape),
        'mean_residual': float(mean_residual),
        'std_residual': float(std_residual),
        'n_samples': int(len(y_true))
    }

    logger.info(f"\n{set_name} Metrics:")
    logger.info(f" R² Score: {r2:.6f}")
    logger.info(f" RMSE: {rmse:.4f}")
    logger.info(f" MAE: {mae:.4f}")
    logger.info(f" MAPE: {mape:.2f}%")
    logger.info(f" Mean Residual: {mean_residual:.4f}")
    logger.info(f" Std Residual: {std_residual:.4f}")
    logger.info(f" Samples: {len(y_true)}")

    return metrics

def compare_models(
    svr_pred: np.ndarray,
    tsformer_pred: np.ndarray,
    ensemble_pred: np.ndarray,
    actual: np.ndarray,
    set_name: str = "Dataset"
) -> Dict:
    """
    Compare individual models vs ensemble

    Args:
        svr_pred: SVR predictions
        tsformer_pred: TSformer predictions
        ensemble_pred: Ensemble predictions
        actual: Actual values
        set_name: Dataset name

    Returns:
        Comparison dictionary
    """
    # Align all predictions
    min_len = min(len(svr_pred), len(tsformer_pred), len(ensemble_pred), len(actual))

    svr_p = svr_pred[-min_len:]
    tsformer_p = tsformer_pred[-min_len:]
    ensemble_p = ensemble_pred[-min_len:]
    y_true = actual[-min_len:]

    # Calculate metrics
    svr_rmse = np.sqrt(mean_squared_error(y_true, svr_p))
    tsformer_rmse = np.sqrt(mean_squared_error(y_true, tsformer_p))
    ensemble_rmse = np.sqrt(mean_squared_error(y_true, ensemble_p))

    svr_r2 = r2_score(y_true, svr_p)
    tsformer_r2 = r2_score(y_true, tsformer_p)
    ensemble_r2 = r2_score(y_true, ensemble_p)

    svr_mae = mean_absolute_error(y_true, svr_p)
    tsformer_mae = mean_absolute_error(y_true, tsformer_p)
    ensemble_mae = mean_absolute_error(y_true, ensemble_p)

    comparison = {
        'dataset': set_name,
        'svr': {
            'rmse': float(svr_rmse),
            'r2': float(svr_r2),
            'mae': float(svr_mae)
        },
        'tsformer': {
            'rmse': float(tsformer_rmse),
            'r2': float(tsformer_r2),
            'mae': float(tsformer_mae)
        },
        'ensemble': {
            'rmse': float(ensemble_rmse),
            'r2': float(ensemble_r2),
            'mae': float(ensemble_mae)
        },
        'improvement': {
            'rmse_vs_svr': float(svr_rmse - ensemble_rmse),
            'rmse_vs_tsformer': float(tsformer_rmse - ensemble_rmse),
            'rmse_vs_best': float(min(svr_rmse, tsformer_rmse) - ensemble_rmse),
            'r2_vs_svr': float(ensemble_r2 - svr_r2),
            'r2_vs_tsformer': float(ensemble_r2 - tsformer_r2)
        }
    }

    logger.info(f"\n{set_name} Model Comparison:")
    logger.info(f" SVR      - RMSE: {svr_rmse:.4f}, R²: {svr_r2:.6f}, MAE: {svr_mae:.4f}")
    logger.info(f" TSformer - RMSE: {tsformer_rmse:.4f}, R²: {tsformer_r2:.6f}, MAE: {tsformer_mae:.4f}")
    logger.info(f" Ensemble - RMSE: {ensemble_rmse:.4f}, R²: {ensemble_r2:.6f}, MAE: {ensemble_mae:.4f}")
    logger.info(f" Improvement vs Best Individual: RMSE ↓ {comparison['improvement']['rmse_vs_best']:.4f}")

    return comparison

# ============================================================================
# TIME-SERIES CROSS-VALIDATION
# ============================================================================
def time_series_cross_validation(
    ensemble: TouristArrivalEnsemble,
    df_svr: pd.DataFrame,
    df_tsformer: pd.DataFrame,
    n_splits: int = 5
) -> Dict:
    """
    Perform time-series cross-validation

    Args:
        ensemble: Ensemble model
        df_svr: SVR features dataframe
        df_tsformer: TSformer features dataframe
        n_splits: Number of CV splits

    Returns:
        Cross-validation results
    """
    logger.info("="*70)
    logger.info(f"TIME-SERIES CROSS-VALIDATION ({n_splits} splits)")
    logger.info("="*70)

    # Use minimum length for alignment
    min_len = min(len(df_svr), len(df_tsformer))
    df_svr_aligned = df_svr.iloc[-min_len:].reset_index(drop=True)
    df_tsformer_aligned = df_tsformer.iloc[-min_len:].reset_index(drop=True)

    tscv = TimeSeriesSplit(n_splits=n_splits)

    cv_results = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(df_svr_aligned), 1):
        logger.info(f"\n--- Fold {fold}/{n_splits} ---")

        # Split data
        train_svr = df_svr_aligned.iloc[train_idx]
        test_svr = df_svr_aligned.iloc[test_idx]

        train_tsformer = df_tsformer_aligned.iloc[train_idx]
        test_tsformer = df_tsformer_aligned.iloc[test_idx]

        try:
            # Reset TSformer scalers for each fold
            ensemble.tsformer_scaler_X = None
            ensemble.tsformer_scaler_y = None

            # Fit scalers on training data
            _, _ = ensemble.predict_tsformer(train_tsformer)

            # Predict on test data
            ensemble_pred, actual = ensemble.predict(test_svr, test_tsformer)

            # Calculate metrics
            rmse = np.sqrt(mean_squared_error(actual, ensemble_pred))
            r2 = r2_score(actual, ensemble_pred)
            mae = mean_absolute_error(actual, ensemble_pred)
            mape = mean_absolute_percentage_error(actual, ensemble_pred) * 100

            fold_results = {
                'fold': fold,
                'rmse': float(rmse),
                'r2': float(r2),
                'mae': float(mae),
                'mape': float(mape),
                'train_samples': len(train_idx),
                'test_samples': len(test_idx)
            }

            cv_results.append(fold_results)

            logger.info(f"Fold {fold} - RMSE: {rmse:.4f}, R²: {r2:.6f}, MAE: {mae:.4f}, MAPE: {mape:.2f}%")

        except Exception as e:
            logger.warning(f"Fold {fold} failed: {str(e)}")
            continue

    # Calculate average metrics
    if cv_results:
        avg_metrics = {
            'mean_rmse': float(np.mean([r['rmse'] for r in cv_results])),
            'std_rmse': float(np.std([r['rmse'] for r in cv_results])),
            'mean_r2': float(np.mean([r['r2'] for r in cv_results])),
            'std_r2': float(np.std([r['r2'] for r in cv_results])),
            'mean_mae': float(np.mean([r['mae'] for r in cv_results])),
            'mean_mape': float(np.mean([r['mape'] for r in cv_results])),
            'fold_results': cv_results
        }

        logger.info("\nCross-Validation Summary:")
        logger.info(f" Mean RMSE: {avg_metrics['mean_rmse']:.4f} (±{avg_metrics['std_rmse']:.4f})")
        logger.info(f" Mean R²: {avg_metrics['mean_r2']:.6f} (±{avg_metrics['std_r2']:.6f})")
        logger.info(f" Mean MAE: {avg_metrics['mean_mae']:.4f}")
        logger.info(f" Mean MAPE: {avg_metrics['mean_mape']:.2f}%")

        return avg_metrics

    return {}

# ============================================================================
# SAVE RESULTS
# ============================================================================
def save_results(
    ensemble: TouristArrivalEnsemble,
    metrics: Dict,
    comparisons: Dict,
    cv_results: Dict,
    predictions: Dict,
    output_dir: str = "final_ensemble_output"
):
    """
    Save all results and artifacts

    Args:
        ensemble: Ensemble model
        metrics: Evaluation metrics
        comparisons: Model comparisons
        cv_results: Cross-validation results
        predictions: Prediction results
        output_dir: Output directory
    """
    logger.info("="*70)
    logger.info("SAVING RESULTS AND ARTIFACTS")
    logger.info("="*70)

    Path(output_dir).mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # 1. Save ensemble model
    ensemble_path = Path(output_dir) / f'ensemble_model_{timestamp}.pkl'
    ensemble.save(ensemble_path)
    logger.info(f"Ensemble model saved: {ensemble_path}")

    # 2. Save TSformer model separately (.keras format - recommended)
    tsformer_path = Path(output_dir) / f'tsformer_model_{timestamp}.keras'
    ensemble.tsformer_model.save(tsformer_path)
    logger.info(f"TSformer model saved (.keras format): {tsformer_path}")

    # 3. Save comprehensive results
    results = {
        'model_info': {
            'model_type': 'Weighted Ensemble (SVR + TSformer)',
            'optimization_method': 'Genetic Algorithm',
            'weights': ensemble.weights,
            'sequence_length': ensemble.sequence_length,
            'tsformer_format': '.keras',
            'timestamp': timestamp
        },
        'evaluation_metrics': metrics,
        'model_comparisons': comparisons,
        'cross_validation': cv_results,
        'performance_summary': {
            'best_r2': max(metrics['train']['r2'], metrics['validation']['r2'], metrics['test']['r2']),
            'test_rmse': metrics['test']['rmse'],
            'test_r2': metrics['test']['r2'],
            'test_mape': metrics['test']['mape']
        }
    }

    results_path = Path(output_dir) / f'evaluation_results_{timestamp}.json'
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=4)
    logger.info(f"Results saved: {results_path}")

    # 4. Save predictions
    for dataset_name, pred_data in predictions.items():
        pred_df = pd.DataFrame({
            'actual': pred_data['actual'],
            'ensemble_pred': pred_data['ensemble'],
            'svr_pred': pred_data['svr'],
            'tsformer_pred': pred_data['tsformer'],
            'residual': pred_data['actual'] - pred_data['ensemble']
        })

        pred_path = Path(output_dir) / f'{dataset_name}_predictions_{timestamp}.csv'
        pred_df.to_csv(pred_path, index=False)
        logger.info(f"{dataset_name.capitalize()} predictions saved: {pred_path}")

    # 5. Save model configuration
    config = {
        'ensemble_model_path': str(ensemble_path),
        'tsformer_model_path': str(tsformer_path),
        'tsformer_format': '.keras (native Keras format)',
        'weights': ensemble.weights,
        'sequence_length': ensemble.sequence_length,
        'timestamp': timestamp,
        'usage_instructions': {
            'loading': 'Use TouristArrivalEnsemble.load(ensemble_path, tsformer_path)',
            'prediction': 'ensemble.predict(df_svr, df_tsformer)',
            'note': 'TSformer model uses .keras format for better compatibility'
        }
    }

    config_path = Path(output_dir) / f'model_config_{timestamp}.json'
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    logger.info(f"Configuration saved: {config_path}")

    logger.info("\nAll results saved successfully!")

# ============================================================================
# MAIN PIPELINE
# ============================================================================
def main():
    """
    Main execution pipeline for final ensemble model
    """
    logger.info("="*70)
    logger.info("FINAL ENSEMBLE MODEL - TRAINING AND EVALUATION")
    logger.info("="*70)
    logger.info(f"Execution started at: {datetime.now()}")

    try:
        # =====================================================================
        # CONFIGURATION - UPDATE THESE PATHS ACCORDING TO YOUR SETUP
        # =====================================================================
        DATA_PATH = 'preprocessed-dataset.csv'
        SVR_MODEL_DIR = '/content/output/svr'
        TSFORMER_MODEL_DIR = '/content/output/tsformer'
        WEIGHTS_FILE = '/content/output/ga/optimal_weights_20251228_131313.json'
        OUTPUT_DIR = '/content/output/final_ensemble'
        SEQUENCE_LENGTH = 30

        # Verify paths exist
        logger.info("\nVerifying configuration paths...")
        if not Path(DATA_PATH).exists():
            raise FileNotFoundError(f"Data file not found: {DATA_PATH}")
        if not Path(SVR_MODEL_DIR).exists():
            raise FileNotFoundError(f"SVR model directory not found: {SVR_MODEL_DIR}")
        if not Path(TSFORMER_MODEL_DIR).exists():
            raise FileNotFoundError(f"TSformer model directory not found: {TSFORMER_MODEL_DIR}")
        if not Path(WEIGHTS_FILE).exists():
            raise FileNotFoundError(f"Weights file not found: {WEIGHTS_FILE}")

        logger.info("All configuration paths verified ✓")

        # 1. Load models and weights
        logger.info("\n[STEP 1] Loading Models and Optimal Weights")
        svr_model, svr_scaler, tsformer_model, weights = load_models_and_weights(
            SVR_MODEL_DIR, TSFORMER_MODEL_DIR, WEIGHTS_FILE
        )

        # 2. Create ensemble model
        logger.info("\n[STEP 2] Creating Ensemble Model")
        ensemble = TouristArrivalEnsemble(
            svr_model=svr_model,
            svr_scaler=svr_scaler,
            tsformer_model=tsformer_model,
            weights=weights,
            sequence_length=SEQUENCE_LENGTH
        )

        # 3. Load and preprocess data
        logger.info("\n[STEP 3] Loading and Preprocessing Data")
        df = load_data(DATA_PATH)
        df_svr = prepare_svr_features(df)
        df_tsformer = prepare_tsformer_features(df)

        # 4. Split data
        logger.info("\n[STEP 4] Splitting Data")
        train_svr, val_svr, test_svr = split_data(df_svr)
        train_tsformer, val_tsformer, test_tsformer = split_data(df_tsformer)

        # 5. Generate predictions
        logger.info("\n[STEP 5] Generating Predictions")
        logger.info("="*70)

        # Train predictions
        logger.info("\nTraining Set Predictions...")
        ensemble.tsformer_scaler_X = None
        ensemble.tsformer_scaler_y = None
        train_ensemble_pred, train_actual = ensemble.predict(train_svr, train_tsformer)
        train_svr_pred, _ = ensemble.predict_svr(train_svr)
        train_tsformer_pred, _ = ensemble.predict_tsformer(train_tsformer)

        # Validation predictions
        logger.info("\nValidation Set Predictions...")
        val_ensemble_pred, val_actual = ensemble.predict(val_svr, val_tsformer)
        val_svr_pred, _ = ensemble.predict_svr(val_svr)
        val_tsformer_pred, _ = ensemble.predict_tsformer(val_tsformer)

        # Test predictions
        logger.info("\nTest Set Predictions...")
        test_ensemble_pred, test_actual = ensemble.predict(test_svr, test_tsformer)
        test_svr_pred, _ = ensemble.predict_svr(test_svr)
        test_tsformer_pred, _ = ensemble.predict_tsformer(test_tsformer)

        # 6. Calculate metrics
        logger.info("\n[STEP 6] Calculating Evaluation Metrics")
        logger.info("="*70)

        train_metrics = calculate_metrics(train_actual, train_ensemble_pred, "Training Set")
        val_metrics = calculate_metrics(val_actual, val_ensemble_pred, "Validation Set")
        test_metrics = calculate_metrics(test_actual, test_ensemble_pred, "Test Set")

        metrics = {
            'train': train_metrics,
            'validation': val_metrics,
            'test': test_metrics
        }

        # 7. Compare models
        logger.info("\n[STEP 7] Comparing Individual Models vs Ensemble")
        logger.info("="*70)

        train_comparison = compare_models(
            train_svr_pred, train_tsformer_pred, train_ensemble_pred,
            train_actual, "Training Set"
        )

        val_comparison = compare_models(
            val_svr_pred, val_tsformer_pred, val_ensemble_pred,
            val_actual, "Validation Set"
        )

        test_comparison = compare_models(
            test_svr_pred, test_tsformer_pred, test_ensemble_pred,
            test_actual, "Test Set"
        )

        comparisons = {
            'train': train_comparison,
            'validation': val_comparison,
            'test': test_comparison
        }

        # 8. Cross-validation
        logger.info("\n[STEP 8] Time-Series Cross-Validation")
        cv_results = time_series_cross_validation(
            ensemble, df_svr, df_tsformer, n_splits=5
        )

        # 9. Prepare prediction data for saving
        predictions = {
            'train': {
                'actual': train_actual,
                'ensemble': train_ensemble_pred,
                'svr': train_svr_pred[-len(train_actual):],
                'tsformer': train_tsformer_pred
            },
            'validation': {
                'actual': val_actual,
                'ensemble': val_ensemble_pred,
                'svr': val_svr_pred[-len(val_actual):],
                'tsformer': val_tsformer_pred
            },
            'test': {
                'actual': test_actual,
                'ensemble': test_ensemble_pred,
                'svr': test_svr_pred[-len(test_actual):],
                'tsformer': test_tsformer_pred
            }
        }

        # 10. Save results
        logger.info("\n[STEP 9] Saving Results and Artifacts")
        save_results(ensemble, metrics, comparisons, cv_results, predictions, OUTPUT_DIR)

        # Final summary
        logger.info("\n" + "="*70)
        logger.info("FINAL ENSEMBLE MODEL - COMPLETED SUCCESSFULLY")
        logger.info("="*70)
        logger.info("\nFinal Performance Summary:")
        logger.info(f" Test R² Score: {test_metrics['r2']:.6f}")
        logger.info(f" Test RMSE: {test_metrics['rmse']:.4f}")
        logger.info(f" Test MAE: {test_metrics['mae']:.4f}")
        logger.info(f" Test MAPE: {test_metrics['mape']:.2f}%")
        logger.info(f" Improvement over best individual: {test_comparison['improvement']['rmse_vs_best']:.4f} RMSE")
        logger.info(f"\nExecution completed at: {datetime.now()}")

        return ensemble, metrics, comparisons, cv_results

    except Exception as e:
        logger.error(f"ERROR in main execution: {str(e)}", exc_info=True)
        raise

# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    ensemble, metrics, comparisons, cv_results = main()


In [ ]:
"""
Production Ensemble Model - Training on Full Dataset (CLEAN VERSION)
=====================================================================
Author: ML Engineering Team
Date: December 2025
Purpose: Fit the final ensemble model on the ENTIRE dataset for production use
Version: 3.0 - Clean version with .keras format and easy loading
"""

import pandas as pd
import numpy as np
import logging
from datetime import datetime
import warnings
import json
from pathlib import Path
import joblib
import pickle
from typing import Tuple, Dict, Any, Optional

# Deep Learning
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
    mean_absolute_error
)

# Suppress warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# CONFIGURATION
# ============================================================================
class Config:
    """Configuration class for flexible path management"""

    # Data paths
    DATA_PATH = 'preprocessed-dataset.csv'

    # Model artifact directories
    SVR_MODEL_DIR = 'output/svr'
    TSFORMER_MODEL_DIR = 'output/tsformer'
    WEIGHTS_DIR = 'output/ga'

    # Output directory
    OUTPUT_DIR = 'production_model_output'

    # Model parameters
    SEQUENCE_LENGTH = 30

    @classmethod
    def validate_paths(cls):
        """Validate that required directories exist"""
        paths = {
            'SVR Model Directory': cls.SVR_MODEL_DIR,
            'TSformer Model Directory': cls.TSFORMER_MODEL_DIR,
            'Weights Directory': cls.WEIGHTS_DIR
        }

        missing = []
        for name, path in paths.items():
            if not Path(path).exists():
                missing.append(f"{name}: {path}")

        if missing:
            raise FileNotFoundError(
                f"\n❌ The following directories are missing:\n" +
                "\n".join(f"   - {m}" for m in missing) +
                "\n\n💡 Update the paths in the Config class to match your directory structure."
            )

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging with both file and console handlers"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'production_ensemble_{timestamp}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# PRODUCTION ENSEMBLE MODEL CLASS
# ============================================================================
class ProductionTouristEnsemble:
    """
    Production-Ready Weighted Ensemble Model
    Combines SVR and TSformer with GA-optimized weights

    Usage:
        # Training
        ensemble = ProductionTouristEnsemble(svr_model, svr_scaler, tsformer_model, weights)
        ensemble.fit(df_svr, df_tsformer)
        ensemble.save('production_model_output')

        # Loading
        ensemble = ProductionTouristEnsemble.load('production_model_output')
        predictions = ensemble.predict(df_svr, df_tsformer)
    """

    def __init__(
        self,
        svr_model,
        svr_scaler,
        tsformer_model,
        weights: Dict[str, float],
        sequence_length: int = 30
    ):
        """
        Initialize production ensemble model

        Args:
            svr_model: Pre-trained SVR model
            svr_scaler: SVR feature scaler
            tsformer_model: Pre-trained TSformer model
            weights: GA-optimized weights {'svr': x, 'tsformer': y}
            sequence_length: TSformer sequence length
        """
        self.svr_model = svr_model
        self.svr_scaler = svr_scaler
        self.tsformer_model = tsformer_model
        self.weights = weights
        self.sequence_length = sequence_length

        # TSformer scalers (fitted on full dataset)
        self.tsformer_scaler_X = StandardScaler()
        self.tsformer_scaler_y = MinMaxScaler()

        # Training metadata
        self.training_date = None
        self.data_date_range = None
        self.n_training_samples = None

        logger.info("Production Ensemble Model Initialized")
        logger.info(f"Ensemble Weights - SVR: {weights['svr']:.6f}, TSformer: {weights['tsformer']:.6f}")
        logger.info(f"Sequence Length: {sequence_length}")

    def fit(self, df_svr: pd.DataFrame, df_tsformer: pd.DataFrame):
        """
        Fit ensemble on full dataset
        Note: Individual models are already pre-trained
        This method fits TSformer scalers and stores metadata

        Args:
            df_svr: Full SVR features dataframe
            df_tsformer: Full TSformer features dataframe
        """
        logger.info("="*70)
        logger.info("FITTING PRODUCTION ENSEMBLE ON FULL DATASET")
        logger.info("="*70)

        # Store metadata
        self.training_date = datetime.now()
        self.data_date_range = {
            'start': str(df_svr['date'].min()),
            'end': str(df_svr['date'].max())
        }
        self.n_training_samples = len(df_svr)

        logger.info(f"Training samples: {self.n_training_samples}")
        logger.info(f"Date range: {self.data_date_range['start']} to {self.data_date_range['end']}")

        # Fit TSformer scalers on full dataset
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]

        X_full = df_tsformer[feature_cols].values
        y_full = df_tsformer['arrivals'].values

        logger.info("Fitting TSformer scalers on full dataset...")
        self.tsformer_scaler_X.fit(X_full)
        self.tsformer_scaler_y.fit(y_full.reshape(-1, 1))

        logger.info("Production ensemble fitted successfully")
        logger.info("Note: SVR model uses pre-fitted scaler from training")

    def predict_svr(self, df_svr: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Generate SVR predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]

        X = df_svr[feature_cols].values
        y = df_svr['arrivals'].values

        X_scaled = self.svr_scaler.transform(X)
        predictions = self.svr_model.predict(X_scaled)

        return predictions, y

    def predict_tsformer(self, df_tsformer: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        """Generate TSformer predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]

        X = df_tsformer[feature_cols].values
        y = df_tsformer['arrivals'].values

        X_scaled = self.tsformer_scaler_X.transform(X)
        y_scaled = self.tsformer_scaler_y.transform(y.reshape(-1, 1)).flatten()

        X_seq = self._create_sequences(X_scaled)
        y_seq = y_scaled[self.sequence_length - 1:]

        y_pred_scaled = self.tsformer_model.predict(X_seq, verbose=0).flatten()
        predictions = self.tsformer_scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()

        actual = y[self.sequence_length - 1:]

        return predictions, actual

    def _create_sequences(self, data: np.ndarray) -> np.ndarray:
        """Create sequences for TSformer"""
        sequences = []
        for i in range(len(data) - self.sequence_length + 1):
            sequences.append(data[i:i + self.sequence_length])
        return np.array(sequences)

    def predict(self, df_svr: pd.DataFrame, df_tsformer: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Generate ensemble predictions

        Returns:
            ensemble_pred: Weighted ensemble predictions
            actual: Actual values (aligned)
            svr_pred: SVR predictions (aligned)
            tsformer_pred: TSformer predictions (aligned)
        """
        svr_pred, svr_actual = self.predict_svr(df_svr)
        tsformer_pred, tsformer_actual = self.predict_tsformer(df_tsformer)

        # Align predictions (TSformer is shorter due to sequences)
        min_len = min(len(svr_pred), len(tsformer_pred))
        svr_pred_aligned = svr_pred[-min_len:]
        tsformer_pred_aligned = tsformer_pred[-min_len:]
        actual_aligned = tsformer_actual[-min_len:]

        # Weighted ensemble
        ensemble_pred = (
            self.weights['svr'] * svr_pred_aligned +
            self.weights['tsformer'] * tsformer_pred_aligned
        )

        return ensemble_pred, actual_aligned, svr_pred_aligned, tsformer_pred_aligned

    def save(self, output_dir: str, model_name: str = "production_ensemble"):
        """
        Save production ensemble model

        Saves:
            1. Ensemble pickle (SVR + scalers + weights + metadata)
            2. TSformer .keras model
            3. Model card JSON
        """
        Path(output_dir).mkdir(exist_ok=True, parents=True)

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

        # Save ensemble pickle (everything except TSformer)
        ensemble_dict = {
            'svr_model': self.svr_model,
            'svr_scaler': self.svr_scaler,
            'weights': self.weights,
            'sequence_length': self.sequence_length,
            'tsformer_scaler_X': self.tsformer_scaler_X,
            'tsformer_scaler_y': self.tsformer_scaler_y,
            'training_metadata': {
                'training_date': str(self.training_date),
                'data_date_range': self.data_date_range,
                'n_training_samples': self.n_training_samples,
                'model_version': 'production_v3.0'
            }
        }

        ensemble_path = Path(output_dir) / f'{model_name}_{timestamp}.pkl'
        with open(ensemble_path, 'wb') as f:
            pickle.dump(ensemble_dict, f)
        logger.info(f"✅ Ensemble pickle saved: {ensemble_path.name}")

        # Save TSformer as .keras (native Keras 3 format)
        tsformer_path = Path(output_dir) / f'{model_name}_tsformer_{timestamp}.keras'
        self.tsformer_model.save(tsformer_path, save_format='keras')
        logger.info(f"✅ TSformer model saved: {tsformer_path.name}")

        # Save model card
        model_card = {
            'model_name': model_name,
            'version': 'production_v3.0',
            'training_date': str(self.training_date),
            'data_info': {
                'date_range': self.data_date_range,
                'n_samples': self.n_training_samples
            },
            'architecture': {
                'type': 'Weighted Ensemble',
                'base_models': ['SVR (RBF kernel)', 'TSformer (Transformer)'],
                'weights': self.weights,
                'optimization': 'Genetic Algorithm',
                'sequence_length': self.sequence_length
            },
            'file_paths': {
                'ensemble': str(ensemble_path.name),
                'tsformer': str(tsformer_path.name)
            },
            'usage': {
                'load': 'ensemble = ProductionTouristEnsemble.load("production_model_output")',
                'predict': 'predictions = ensemble.predict(df_svr, df_tsformer)'
            }
        }

        card_path = Path(output_dir) / f'{model_name}_model_card_{timestamp}.json'
        with open(card_path, 'w') as f:
            json.dump(model_card, f, indent=4)
        logger.info(f"✅ Model card saved: {card_path.name}")

        return ensemble_path, tsformer_path, card_path

    @classmethod
    def load(cls, model_dir: str = "production_model_output"):
        """
        Load production ensemble model

        Args:
            model_dir: Directory containing saved model files

        Returns:
            ProductionTouristEnsemble: Loaded ensemble model ready for predictions

        Usage:
            ensemble = ProductionTouristEnsemble.load('production_model_output')
            predictions = ensemble.predict(df_svr, df_tsformer)
        """
        logger.info("="*70)
        logger.info("LOADING PRODUCTION ENSEMBLE MODEL")
        logger.info("="*70)

        model_dir = Path(model_dir)

        if not model_dir.exists():
            raise FileNotFoundError(f"Model directory not found: {model_dir}")

        # Find latest ensemble pickle
        ensemble_files = list(model_dir.glob('production_ensemble_*.pkl'))
        if not ensemble_files:
            raise FileNotFoundError(
                f"No ensemble model found in {model_dir}\n"
                "Expected: production_ensemble_*.pkl"
            )

        ensemble_path = sorted(ensemble_files)[-1]
        logger.info(f"Loading ensemble: {ensemble_path.name}")

        # Load pickle
        with open(ensemble_path, 'rb') as f:
            ensemble_dict = pickle.load(f)

        # Find corresponding TSformer model
        timestamp = ensemble_path.stem.split('_')[-1]

        # Try .keras first, then .h5
        tsformer_patterns = [
            f'production_ensemble_tsformer_{timestamp}.keras',
            f'production_ensemble_tsformer_{timestamp}.h5',
            'production_ensemble_tsformer_*.keras',
            'production_ensemble_tsformer_*.h5'
        ]

        tsformer_path = None
        for pattern in tsformer_patterns:
            files = list(model_dir.glob(pattern))
            if files:
                tsformer_path = sorted(files)[-1]
                break

        if not tsformer_path:
            raise FileNotFoundError(
                f"No TSformer model found in {model_dir}\n"
                f"Expected patterns: {tsformer_patterns}"
            )

        logger.info(f"Loading TSformer: {tsformer_path.name}")

        # Load TSformer model
        tsformer_model = keras.models.load_model(tsformer_path)

        # Create ensemble object
        ensemble = cls(
            svr_model=ensemble_dict['svr_model'],
            svr_scaler=ensemble_dict['svr_scaler'],
            tsformer_model=tsformer_model,
            weights=ensemble_dict['weights'],
            sequence_length=ensemble_dict['sequence_length']
        )

        # Restore TSformer scalers
        ensemble.tsformer_scaler_X = ensemble_dict['tsformer_scaler_X']
        ensemble.tsformer_scaler_y = ensemble_dict['tsformer_scaler_y']

        # Restore metadata
        metadata = ensemble_dict.get('training_metadata', {})
        ensemble.training_date = metadata.get('training_date')
        ensemble.data_date_range = metadata.get('data_date_range')
        ensemble.n_training_samples = metadata.get('n_training_samples')

        logger.info("="*70)
        logger.info("✅ ENSEMBLE LOADED SUCCESSFULLY")
        logger.info("="*70)
        logger.info(f"Weights: SVR={ensemble.weights['svr']:.6f}, TSformer={ensemble.weights['tsformer']:.6f}")
        logger.info(f"Training samples: {ensemble.n_training_samples}")
        logger.info(f"Date range: {ensemble.data_date_range}")
        logger.info(f"Sequence length: {ensemble.sequence_length}")

        return ensemble

# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================
def load_data(data_path: str) -> pd.DataFrame:
    """Load preprocessed dataset"""
    logger.info("="*70)
    logger.info("LOADING FULL DATASET")
    logger.info("="*70)

    df = pd.read_csv(data_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    logger.info(f"Data loaded. Shape: {df.shape}")
    logger.info(f"Date range: {df['date'].min()} to {df['date'].max()}")
    logger.info(f"Total samples: {len(df)}")

    return df

def prepare_svr_features(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare SVR-specific features"""
    logger.info("Preparing SVR features...")

    df = df.copy()

    # Temporal features
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Lag features
    for lag in [7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    df = df.dropna()

    logger.info(f"SVR features prepared. Shape: {df.shape}")
    return df

def prepare_tsformer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare TSformer-specific features"""
    logger.info("Preparing TSformer features...")

    df = df.copy()

    # Temporal features (includes year)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['quarter'] = df['date'].dt.quarter
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
    df['year'] = df['date'].dt.year
    df['day_of_year'] = df['date'].dt.dayofyear

    # Cyclical encoding
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

    # Lag features (more lags for TSformer)
    for lag in [1, 7, 14, 30]:
        df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)

    # Rolling statistics
    for window in [7, 14, 30]:
        df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).mean()
        df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
            window=window, min_periods=1
        ).std()

    # EMA
    df['arrivals_ema_7'] = df['arrivals'].ewm(span=7, adjust=False).mean()
    df['arrivals_ema_30'] = df['arrivals'].ewm(span=30, adjust=False).mean()

    # Differencing
    df['arrivals_diff_1'] = df['arrivals'].diff(1)
    df['arrivals_diff_7'] = df['arrivals'].diff(7)

    df = df.fillna(method='ffill').fillna(method='bfill')

    logger.info(f"TSformer features prepared. Shape: {df.shape}")
    return df

# ============================================================================
# FLEXIBLE MODEL LOADING
# ============================================================================
def find_file_flexible(directory: str, patterns: list, file_type: str) -> Optional[Path]:
    """Flexible file finder that searches multiple patterns"""
    dir_path = Path(directory)

    for pattern in patterns:
        files = list(dir_path.glob(pattern))
        if files:
            latest_file = sorted(files)[-1]
            logger.info(f"Found {file_type}: {latest_file.name}")
            return latest_file

    return None

def load_pretrained_models(config: Config) -> Tuple:
    """Load pre-trained models with flexible path handling"""
    logger.info("="*70)
    logger.info("LOADING PRE-TRAINED MODELS AND OPTIMAL WEIGHTS")
    logger.info("="*70)

    # Validate paths first
    config.validate_paths()

    # Load SVR model
    svr_patterns = ['svr_model_bo_*.pkl', 'svr_model_*.pkl', '*.pkl']
    svr_model_path = find_file_flexible(config.SVR_MODEL_DIR, svr_patterns, "SVR model")

    if not svr_model_path:
        raise FileNotFoundError(
            f"❌ SVR model not found in {config.SVR_MODEL_DIR}\n"
            f"   Searched for: {svr_patterns}"
        )

    logger.info(f"Loading SVR model: {svr_model_path}")
    svr_model = joblib.load(svr_model_path)

    # Load SVR scaler
    scaler_patterns = ['scaler_bo_*.pkl', 'scaler_*.pkl', '*scaler*.pkl']
    svr_scaler_path = find_file_flexible(config.SVR_MODEL_DIR, scaler_patterns, "SVR scaler")

    if not svr_scaler_path:
        raise FileNotFoundError(
            f"❌ SVR scaler not found in {config.SVR_MODEL_DIR}\n"
            f"   Searched for: {scaler_patterns}"
        )

    logger.info(f"Loading SVR scaler: {svr_scaler_path}")
    svr_scaler = joblib.load(svr_scaler_path)

    # Load TSformer model
    tsformer_patterns = [
        'best_tsformer_model.keras',
        'best_tsformer_model.h5',
        '*tsformer*.keras',
        '*tsformer*.h5',
        'models/best_tsformer_model.keras',
        'models/best_tsformer_model.h5'
    ]

    tsformer_path = None
    tsformer_dir = Path(config.TSFORMER_MODEL_DIR)

    for pattern in tsformer_patterns:
        if '/' in pattern:
            full_pattern = tsformer_dir / pattern
            if full_pattern.exists():
                tsformer_path = full_pattern
                break
        else:
            files = list(tsformer_dir.glob(pattern))
            if files:
                tsformer_path = sorted(files)[-1]
                break

    if not tsformer_path:
        raise FileNotFoundError(
            f"❌ TSformer model not found in {config.TSFORMER_MODEL_DIR}\n"
            f"   Searched for: {tsformer_patterns}"
        )

    logger.info(f"Loading TSformer model: {tsformer_path.name}")
    tsformer_model = keras.models.load_model(tsformer_path)

    # Load weights
    weight_patterns = [
        'optimal_weights_*.json',
        'ensemble_weights_*.json',
        '*weights*.json'
    ]

    weights_path = find_file_flexible(config.WEIGHTS_DIR, weight_patterns, "optimal weights")

    if not weights_path:
        available_files = list(Path(config.WEIGHTS_DIR).glob('*.json'))
        file_list = "\n   ".join([f.name for f in available_files]) if available_files else "No JSON files found"

        raise FileNotFoundError(
            f"❌ Optimal weights file not found in {config.WEIGHTS_DIR}\n"
            f"   Searched for: {weight_patterns}\n"
            f"   Available files:\n   {file_list}"
        )

    logger.info(f"Loading optimal weights: {weights_path.name}")
    with open(weights_path, 'r') as f:
        weights_data = json.load(f)

    weights = weights_data['weights']
    logger.info(f"Ensemble weights - SVR: {weights['svr']:.6f}, TSformer: {weights['tsformer']:.6f}")

    return svr_model, svr_scaler, tsformer_model, weights

# ============================================================================
# EVALUATION METRICS
# ============================================================================
def calculate_comprehensive_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_svr: np.ndarray,
    y_tsformer: np.ndarray,
    set_name: str = "Full Dataset"
) -> Dict[str, Any]:
    """Calculate comprehensive evaluation metrics"""
    logger.info("="*70)
    logger.info(f"CALCULATING METRICS - {set_name}")
    logger.info("="*70)

    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    residuals = y_true - y_pred
    mean_residual = np.mean(residuals)
    std_residual = np.std(residuals)
    max_residual = np.max(np.abs(residuals))

    abs_errors = np.abs(residuals)
    p50_error = np.percentile(abs_errors, 50)
    p90_error = np.percentile(abs_errors, 90)
    p95_error = np.percentile(abs_errors, 95)

    svr_rmse = np.sqrt(mean_squared_error(y_true, y_svr))
    svr_r2 = r2_score(y_true, y_svr)

    tsformer_rmse = np.sqrt(mean_squared_error(y_true, y_tsformer))
    tsformer_r2 = r2_score(y_true, y_tsformer)

    metrics = {
        'dataset': set_name,
        'n_samples': int(len(y_true)),
        'ensemble_metrics': {
            'r2_score': float(r2),
            'rmse': float(rmse),
            'mae': float(mae),
            'mape_percent': float(mape),
            'mse': float(mse)
        },
        'residual_analysis': {
            'mean_residual': float(mean_residual),
            'std_residual': float(std_residual),
            'max_absolute_residual': float(max_residual),
            'median_absolute_error': float(p50_error),
            'p90_absolute_error': float(p90_error),
            'p95_absolute_error': float(p95_error)
        },
        'individual_models': {
            'svr': {'rmse': float(svr_rmse), 'r2': float(svr_r2)},
            'tsformer': {'rmse': float(tsformer_rmse), 'r2': float(tsformer_r2)}
        },
        'ensemble_improvement': {
            'rmse_vs_svr': float(svr_rmse - rmse),
            'rmse_vs_tsformer': float(tsformer_rmse - rmse),
            'rmse_vs_best_individual': float(min(svr_rmse, tsformer_rmse) - rmse),
            'r2_vs_svr': float(r2 - svr_r2),
            'r2_vs_tsformer': float(r2 - tsformer_r2)
        }
    }

    logger.info(f"\n{set_name} Ensemble Performance:")
    logger.info(f" R² Score: {r2:.6f}")
    logger.info(f" RMSE: {rmse:.4f}")
    logger.info(f" MAE: {mae:.4f}")
    logger.info(f" MAPE: {mape:.2f}%")
    logger.info(f" Samples: {len(y_true)}")

    logger.info(f"\nModel Comparison:")
    logger.info(f" SVR      - RMSE: {svr_rmse:.4f}, R²: {svr_r2:.6f}")
    logger.info(f" TSformer - RMSE: {tsformer_rmse:.4f}, R²: {tsformer_r2:.6f}")
    logger.info(f" Ensemble - RMSE: {rmse:.4f}, R²: {r2:.6f}")
    logger.info(f" Improvement: {metrics['ensemble_improvement']['rmse_vs_best_individual']:.4f} RMSE")

    return metrics

# ============================================================================
# SAVE RESULTS
# ============================================================================
def save_production_results(
    ensemble: ProductionTouristEnsemble,
    metrics: Dict,
    predictions: Dict,
    output_dir: str
):
    """Save all production model results"""
    logger.info("="*70)
    logger.info("SAVING PRODUCTION MODEL AND RESULTS")
    logger.info("="*70)

    Path(output_dir).mkdir(exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Save ensemble
    ensemble_path, tsformer_path, card_path = ensemble.save(output_dir, "production_ensemble")

    # Save training metrics
    results = {
        'model_info': {
            'model_type': 'Production Weighted Ensemble (SVR + TSformer)',
            'training_strategy': 'Full dataset training after validation',
            'optimization_method': 'Genetic Algorithm',
            'weights': ensemble.weights,
            'sequence_length': ensemble.sequence_length,
            'training_date': str(ensemble.training_date),
            'data_date_range': ensemble.data_date_range
        },
        'training_metrics': metrics,
        'model_files': {
            'ensemble_wrapper': str(ensemble_path.name),
            'tsformer_model': str(tsformer_path.name),
            'model_card': str(card_path.name)
        }
    }

    metrics_path = Path(output_dir) / f'training_metrics_{timestamp}.json'
    with open(metrics_path, 'w') as f:
        json.dump(results, f, indent=4)
    logger.info(f"Training metrics saved: {metrics_path.name}")

    # Save predictions
    pred_df = pd.DataFrame({
        'actual': predictions['actual'],
        'ensemble_pred': predictions['ensemble'],
        'svr_pred': predictions['svr'],
        'tsformer_pred': predictions['tsformer'],
        'residual': predictions['actual'] - predictions['ensemble'],
        'absolute_error': np.abs(predictions['actual'] - predictions['ensemble']),
        'percentage_error': np.abs(
            (predictions['actual'] - predictions['ensemble']) / predictions['actual'] * 100
        )
    })

    pred_path = Path(output_dir) / f'full_dataset_predictions_{timestamp}.csv'
    pred_df.to_csv(pred_path, index=False)
    logger.info(f"Predictions saved: {pred_path.name}")

    # Save summary
    summary = {
        'production_model_summary': {
            'creation_date': timestamp,
            'total_training_samples': ensemble.n_training_samples,
            'date_range': ensemble.data_date_range,
            'performance': {
                'r2_score': metrics['ensemble_metrics']['r2_score'],
                'rmse': metrics['ensemble_metrics']['rmse'],
                'mae': metrics['ensemble_metrics']['mae'],
                'mape': metrics['ensemble_metrics']['mape_percent']
            },
            'model_improvement': {
                'vs_best_individual': metrics['ensemble_improvement']['rmse_vs_best_individual']
            }
        },
        'deployment_ready': True,
        'usage': {
            'load': 'ensemble = ProductionTouristEnsemble.load("production_model_output")',
            'predict': 'predictions = ensemble.predict(df_svr, df_tsformer)'
        },
        'next_steps': [
            'Generate 2026-2030 forecasts (Phase 1)',
            'Apply SHAP/LIME explainability (Phase 2)',
            'Deploy as REST API (Phase 3)'
        ]
    }

    summary_path = Path(output_dir) / f'production_model_summary_{timestamp}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=4)
    logger.info(f"Summary report saved: {summary_path.name}")

# ============================================================================
# MAIN PIPELINE
# ============================================================================
def main():
    """Main execution pipeline"""
    logger.info("="*70)
    logger.info("PRODUCTION ENSEMBLE MODEL - FULL DATASET TRAINING")
    logger.info("="*70)
    logger.info(f"Execution started at: {datetime.now()}")

    try:
        # Display configuration
        logger.info("\n📁 Configuration:")
        logger.info(f"   Data Path: {Config.DATA_PATH}")
        logger.info(f"   SVR Model Dir: {Config.SVR_MODEL_DIR}")
        logger.info(f"   TSformer Model Dir: {Config.TSFORMER_MODEL_DIR}")
        logger.info(f"   Weights Dir: {Config.WEIGHTS_DIR}")
        logger.info(f"   Output Dir: {Config.OUTPUT_DIR}")

        # 1. Load pre-trained models and weights
        logger.info("\n[STEP 1] Loading Pre-Trained Models and Optimal Weights")
        svr_model, svr_scaler, tsformer_model, weights = load_pretrained_models(Config)

        # 2. Create production ensemble
        logger.info("\n[STEP 2] Creating Production Ensemble Model")
        ensemble = ProductionTouristEnsemble(
            svr_model=svr_model,
            svr_scaler=svr_scaler,
            tsformer_model=tsformer_model,
            weights=weights,
            sequence_length=Config.SEQUENCE_LENGTH
        )

        # 3. Load full dataset
        logger.info("\n[STEP 3] Loading Full Dataset (No Splitting)")
        df = load_data(Config.DATA_PATH)

        # 4. Preprocess features
        logger.info("\n[STEP 4] Preprocessing Features")
        df_svr = prepare_svr_features(df)
        df_tsformer = prepare_tsformer_features(df)

        logger.info(f"\nFinal dataset sizes:")
        logger.info(f" SVR features: {len(df_svr)} samples")
        logger.info(f" TSformer features: {len(df_tsformer)} samples")

        # 5. Fit ensemble on full dataset
        logger.info("\n[STEP 5] Fitting Ensemble on Full Dataset")
        ensemble.fit(df_svr, df_tsformer)

        # 6. Generate predictions
        logger.info("\n[STEP 6] Generating Predictions on Full Dataset")
        ensemble_pred, actual, svr_pred, tsformer_pred = ensemble.predict(df_svr, df_tsformer)

        logger.info(f"\nPredictions generated:")
        logger.info(f" Ensemble predictions: {len(ensemble_pred)}")
        logger.info(f" Actual values: {len(actual)}")

        # 7. Calculate metrics
        logger.info("\n[STEP 7] Calculating Training Metrics on Full Dataset")
        metrics = calculate_comprehensive_metrics(
            y_true=actual,
            y_pred=ensemble_pred,
            y_svr=svr_pred,
            y_tsformer=tsformer_pred,
            set_name="Full Dataset (Training)"
        )

        # 8. Save results
        predictions = {
            'actual': actual,
            'ensemble': ensemble_pred,
            'svr': svr_pred,
            'tsformer': tsformer_pred
        }

        logger.info("\n[STEP 8] Saving Production Model and Results")
        save_production_results(ensemble, metrics, predictions, Config.OUTPUT_DIR)

        # Final summary
        logger.info("\n" + "="*70)
        logger.info("✅ PRODUCTION ENSEMBLE MODEL - TRAINING COMPLETED")
        logger.info("="*70)
        logger.info("\nFinal Training Performance:")
        logger.info(f" Training Samples: {ensemble.n_training_samples}")
        logger.info(f" R² Score: {metrics['ensemble_metrics']['r2_score']:.6f}")
        logger.info(f" RMSE: {metrics['ensemble_metrics']['rmse']:.4f}")
        logger.info(f" MAE: {metrics['ensemble_metrics']['mae']:.4f}")
        logger.info(f" MAPE: {metrics['ensemble_metrics']['mape_percent']:.2f}%")
        logger.info(f" Improvement: {metrics['ensemble_improvement']['rmse_vs_best_individual']:.4f} RMSE")

        logger.info("\n✅ Production model ready for deployment!")
        logger.info("\n💡 To use the model:")
        logger.info('   ensemble = ProductionTouristEnsemble.load("production_model_output")')
        logger.info('   predictions = ensemble.predict(df_svr, df_tsformer)')

        logger.info(f"\nExecution completed at: {datetime.now()}")

        return ensemble, metrics

    except Exception as e:
        logger.error(f"❌ ERROR: {str(e)}", exc_info=True)
        raise

# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    print("="*70)
    print("PRODUCTION ENSEMBLE TRAINING - FULL DATASET (V3.0)")
    print("="*70)
    print("\n🔧 CONFIGURATION:")
    print(f"   If paths are incorrect, edit the Config class")
    print(f"\n   Current paths:")
    print(f"   - SVR: {Config.SVR_MODEL_DIR}")
    print(f"   - TSformer: {Config.TSFORMER_MODEL_DIR}")
    print(f"   - Weights: {Config.WEIGHTS_DIR}")
    print("\n" + "="*70 + "\n")

    production_ensemble, training_metrics = main()

    print("\n" + "="*70)
    print("✅ PRODUCTION MODEL TRAINING SUCCESSFUL")
    print("="*70)
    print(f"\nTraining R² Score: {training_metrics['ensemble_metrics']['r2_score']:.6f}")
    print(f"Training RMSE: {training_metrics['ensemble_metrics']['rmse']:.4f}")
    print(f"\nModel saved to: {Config.OUTPUT_DIR}/")
    print("\n💡 To load and use:")
    print('   ensemble = ProductionTouristEnsemble.load("production_model_output")')
    print('   predictions = ensemble.predict(df_svr, df_tsformer)')


In [ ]:
"""
PHASE 1: Scenario-Based Monthly Forecasting (2026-2030) - FIXED
================================================================
Properly generates MONTHLY TOTALS (not first-day predictions)
Date range: January 2026 - December 2030 (60 months)

Author: ML Engineering Team
Date: December 2025
Version: 2.0 - Fixed monthly aggregation and date alignment
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from pathlib import Path
import logging
import json
import pickle
from calendar import monthrange

# Deep Learning
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['font.size'] = 11

# Random seed
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'phase1_forecasting_{timestamp}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# PRODUCTION ENSEMBLE MODEL CLASS (From V3.0)
# ============================================================================
class ProductionTouristEnsemble:
    """
    Production-Ready Weighted Ensemble Model
    Combines SVR and TSformer with GA-optimized weights
    """

    def __init__(
        self,
        svr_model,
        svr_scaler,
        tsformer_model,
        weights: dict,
        sequence_length: int = 30
    ):
        self.svr_model = svr_model
        self.svr_scaler = svr_scaler
        self.tsformer_model = tsformer_model
        self.weights = weights
        self.sequence_length = sequence_length

        # TSformer scalers
        self.tsformer_scaler_X = StandardScaler()
        self.tsformer_scaler_y = MinMaxScaler()

        # Training metadata
        self.training_date = None
        self.data_date_range = None
        self.n_training_samples = None

    def predict_svr(self, df_svr: pd.DataFrame) -> np.ndarray:
        """Generate SVR predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]

        X = df_svr[feature_cols].values
        X_scaled = self.svr_scaler.transform(X)
        predictions = self.svr_model.predict(X_scaled)

        return predictions

    def predict_tsformer(self, df_tsformer: pd.DataFrame) -> np.ndarray:
        """Generate TSformer predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]

        X = df_tsformer[feature_cols].values
        X_scaled = self.tsformer_scaler_X.transform(X)

        X_seq = self._create_sequences(X_scaled)

        y_pred_scaled = self.tsformer_model.predict(X_seq, verbose=0).flatten()
        predictions = self.tsformer_scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()

        return predictions

    def _create_sequences(self, data: np.ndarray) -> np.ndarray:
        """Create sequences for TSformer"""
        sequences = []
        for i in range(len(data) - self.sequence_length + 1):
            sequences.append(data[i:i + self.sequence_length])
        return np.array(sequences)

    def predict(self, df_svr: pd.DataFrame, df_tsformer: pd.DataFrame) -> tuple:
        """Generate ensemble predictions - returns predictions and aligned dates"""
        svr_pred = self.predict_svr(df_svr)
        tsformer_pred = self.predict_tsformer(df_tsformer)

        # Align predictions (TSformer is shorter due to sequences)
        min_len = min(len(svr_pred), len(tsformer_pred))
        svr_pred_aligned = svr_pred[-min_len:]
        tsformer_pred_aligned = tsformer_pred[-min_len:]

        # Get corresponding dates
        dates_aligned = df_svr['date'].iloc[-min_len:].values

        # Weighted ensemble
        ensemble_pred = (
            self.weights['svr'] * svr_pred_aligned +
            self.weights['tsformer'] * tsformer_pred_aligned
        )

        return ensemble_pred, dates_aligned

    @classmethod
    def load(cls, model_dir: str = "production_model_output"):
        """Load production ensemble model"""
        logger.info("="*70)
        logger.info("LOADING PRODUCTION ENSEMBLE MODEL")
        logger.info("="*70)

        model_dir = Path(model_dir)

        if not model_dir.exists():
            raise FileNotFoundError(f"Model directory not found: {model_dir}")

        # Find latest ensemble pickle
        ensemble_files = list(model_dir.glob('production_ensemble_*.pkl'))
        if not ensemble_files:
            raise FileNotFoundError(f"No ensemble model found in {model_dir}")

        ensemble_path = sorted(ensemble_files)[-1]
        logger.info(f"Loading ensemble: {ensemble_path.name}")

        # Load pickle
        with open(ensemble_path, 'rb') as f:
            ensemble_dict = pickle.load(f)

        # Find TSformer model
        timestamp = ensemble_path.stem.split('_')[-1]

        tsformer_patterns = [
            f'production_ensemble_tsformer_{timestamp}.keras',
            f'production_ensemble_tsformer_{timestamp}.h5',
            'production_ensemble_tsformer_*.keras',
            'production_ensemble_tsformer_*.h5'
        ]

        tsformer_path = None
        for pattern in tsformer_patterns:
            files = list(model_dir.glob(pattern))
            if files:
                tsformer_path = sorted(files)[-1]
                break

        if not tsformer_path:
            raise FileNotFoundError(f"No TSformer model found in {model_dir}")

        logger.info(f"Loading TSformer: {tsformer_path.name}")
        tsformer_model = keras.models.load_model(tsformer_path)

        # Create ensemble object
        ensemble = cls(
            svr_model=ensemble_dict['svr_model'],
            svr_scaler=ensemble_dict['svr_scaler'],
            tsformer_model=tsformer_model,
            weights=ensemble_dict['weights'],
            sequence_length=ensemble_dict['sequence_length']
        )

        # Restore scalers and metadata
        ensemble.tsformer_scaler_X = ensemble_dict['tsformer_scaler_X']
        ensemble.tsformer_scaler_y = ensemble_dict['tsformer_scaler_y']

        metadata = ensemble_dict.get('training_metadata', {})
        ensemble.training_date = metadata.get('training_date')
        ensemble.data_date_range = metadata.get('data_date_range')
        ensemble.n_training_samples = metadata.get('n_training_samples')

        logger.info("✅ Ensemble loaded successfully")
        logger.info(f"Weights: SVR={ensemble.weights['svr']:.6f}, TSformer={ensemble.weights['tsformer']:.6f}")
        logger.info(f"Training samples: {ensemble.n_training_samples}")

        return ensemble

# ============================================================================
# SCENARIO ASSUMPTIONS (No Olympics - Sri Lanka specific)
# ============================================================================
class ScenarioAssumptions:
    """Define assumptions for each scenario"""

    BASELINE = {
        'name': 'Baseline',
        'description': 'Most likely continuation of current trends',
        'probability': 0.50,
        'gdp_growth_rate': 0.04,
        'inflation_rate': 0.05,
        'exchange_rate_change': 0.00,
        'brent_crude_change': 0.00,
        'temperature_adjustment': 0.0,
        'precipitation_adjustment': 1.0,
        'google_trends_decay': 0.998,
        'google_trends_boost': 1.0,
        'event_frequency_multiplier': 1.0
    }

    OPTIMISTIC = {
        'name': 'Optimistic',
        'description': 'Best-case scenario with favorable economic and tourism conditions',
        'probability': 0.25,
        'gdp_growth_rate': 0.06,
        'inflation_rate': 0.04,
        'exchange_rate_change': 0.10,
        'brent_crude_change': -0.10,
        'temperature_adjustment': -1.0,
        'precipitation_adjustment': 0.85,
        'google_trends_decay': 0.999,
        'google_trends_boost': 1.25,
        'event_frequency_multiplier': 1.25
    }

    PESSIMISTIC = {
        'name': 'Pessimistic',
        'description': 'Worst-case scenario with adverse economic and climate conditions',
        'probability': 0.25,
        'gdp_growth_rate': 0.01,
        'inflation_rate': 0.08,
        'exchange_rate_change': -0.15,
        'brent_crude_change': 0.15,
        'temperature_adjustment': 2.0,
        'precipitation_adjustment': 1.25,
        'google_trends_decay': 0.995,
        'google_trends_boost': 0.70,
        'event_frequency_multiplier': 0.75
    }

# ============================================================================
# FUTURE FEATURE GENERATOR
# ============================================================================
class FutureFeatureGenerator:
    """Generate DAILY features for future dates, then aggregate to monthly"""

    def __init__(self, historical_df: pd.DataFrame):
        self.historical_df = historical_df.copy()
        self.historical_df['date'] = pd.to_datetime(self.historical_df['date'])
        self.historical_df = self.historical_df.sort_values('date')

        self.last_row = self.historical_df.iloc[-1]
        self.last_date = self.last_row['date']

        self._calculate_historical_baselines()

        logger.info(f"Historical data loaded: {len(self.historical_df)} records")
        logger.info(f"Last date: {self.last_date}")

    def _calculate_historical_baselines(self):
        """Calculate baseline values from historical data"""
        recent_df = self.historical_df.tail(90)

        self.baseline_gdp = self.last_row['gdp_per_capita']
        self.baseline_inflation = recent_df['inflation_rate'].mean()
        self.baseline_brent = recent_df['brent_crude_price'].mean()

        self.baseline_usd_lkr = recent_df['usd_lkr'].mean()
        self.baseline_gbp_lkr = recent_df['gbp_lkr'].mean()
        self.baseline_eur_lkr = recent_df['eur_lkr'].mean()
        self.baseline_rub_lkr = recent_df['rub_lkr'].mean()
        self.baseline_inr_lkr = recent_df['inr_lkr'].mean()
        self.baseline_cny_lkr = recent_df['cny_lkr'].mean()

        # Daily weather patterns by month
        self.monthly_temp = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['temperature'].mean().to_dict()

        self.monthly_precip = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['precipitation'].mean().to_dict()

        self.monthly_humidity = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['humidity'].mean().to_dict()

        self.baseline_web_search = recent_df['web_search'].mean()
        self.baseline_image_search = recent_df['image_search'].mean()

        # For lag features
        self.last_30_arrivals = self.historical_df['arrivals'].tail(30).values
        self.recent_avg = self.last_30_arrivals.mean()
        self.recent_std = self.last_30_arrivals.std()

        logger.info("Historical baselines calculated")

    def generate_daily_dates(self, start_date: str, end_date: str):
        """Generate ALL daily dates for the forecast period"""
        # Add buffer at start for TSformer sequences
        buffer_start = pd.to_datetime(start_date) - timedelta(days=30)

        dates = pd.date_range(
            start=buffer_start,
            end=end_date,
            freq='D'
        )

        logger.info(f"Generated {len(dates)} daily dates: {dates[0]} to {dates[-1]}")
        return list(dates)

    def create_base_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create base features for ALL daily dates"""
        df = pd.DataFrame({'date': dates})
        n_days = len(dates)

        # Reference date for calculations
        ref_date = pd.to_datetime('2026-01-01')

        # Calculate years elapsed from reference
        years_from_ref = [(d - ref_date).days / 365.25 for d in dates]

        # ECONOMIC FEATURES (change gradually over time)
        df['gdp_per_capita'] = [
            self.baseline_gdp * ((1 + scenario['gdp_growth_rate']) ** max(0, y))
            for y in years_from_ref
        ]

        df['inflation_rate'] = scenario['inflation_rate']

        # Exchange rates with gradual change
        exchange_change = scenario['exchange_rate_change']
        df['usd_lkr'] = [
            self.baseline_usd_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['gbp_lkr'] = [
            self.baseline_gbp_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['eur_lkr'] = [
            self.baseline_eur_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['rub_lkr'] = [
            self.baseline_rub_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['inr_lkr'] = [
            self.baseline_inr_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['cny_lkr'] = [
            self.baseline_cny_lkr * (1 + exchange_change * max(0, y) / 5)
            for y in years_from_ref
        ]

        # Brent crude
        brent_change = scenario['brent_crude_change']
        df['brent_crude_price'] = self.baseline_brent * (1 + brent_change)

        # WEATHER (daily patterns with monthly baselines)
        df['temperature'] = df['date'].dt.month.map(self.monthly_temp)
        df['temperature'] += scenario['temperature_adjustment']

        df['precipitation'] = df['date'].dt.month.map(self.monthly_precip)
        df['precipitation'] *= scenario['precipitation_adjustment']

        df['humidity'] = df['date'].dt.month.map(self.monthly_humidity)

        # GOOGLE TRENDS (decay over time)
        initial_web = self.baseline_web_search * scenario['google_trends_boost']
        initial_image = self.baseline_image_search * scenario['google_trends_boost']
        decay_rate = scenario['google_trends_decay']

        days_elapsed = np.arange(n_days)
        df['web_search'] = initial_web * (decay_rate ** days_elapsed)
        df['image_search'] = initial_image * (decay_rate ** days_elapsed)

        # EVENTS (sparse)
        df['event_encoded'] = 0.0
        event_freq = scenario['event_frequency_multiplier']
        n_events = int(n_days * 0.01 * event_freq)
        if n_events > 0:
            event_indices = np.random.choice(n_days, size=n_events, replace=False)
            df.loc[event_indices, 'event_encoded'] = 0.05

        # COVID/CRISIS (zero for future)
        df['covid_impact_factor'] = 0.0
        df['crisis_impact_factor'] = 0.0

        return df

    def create_svr_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create SVR-specific features for daily dates"""
        logger.info(f"Generating DAILY SVR features for {scenario['name']}...")

        df = self.create_base_features(scenario, dates)

        # TEMPORAL FEATURES
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

        # CYCLICAL ENCODING
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

        # LAG AND ROLLING FEATURES (use recent historical average)
        df['arrivals_lag_7'] = self.recent_avg
        df['arrivals_lag_14'] = self.recent_avg
        df['arrivals_lag_30'] = self.recent_avg

        df['arrivals_rolling_mean_7'] = self.recent_avg
        df['arrivals_rolling_mean_14'] = self.recent_avg
        df['arrivals_rolling_mean_30'] = self.recent_avg

        df['arrivals_rolling_std_7'] = self.recent_std
        df['arrivals_rolling_std_14'] = self.recent_std
        df['arrivals_rolling_std_30'] = self.recent_std

        logger.info(f"SVR daily features: {df.shape}")
        return df

    def create_tsformer_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create TSformer-specific features for daily dates"""
        logger.info(f"Generating DAILY TSformer features for {scenario['name']}...")

        df = self.create_base_features(scenario, dates)

        # TEMPORAL FEATURES (includes year)
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
        df['year'] = df['date'].dt.year
        df['day_of_year'] = df['date'].dt.dayofyear

        # CYCLICAL ENCODING
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
        df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

        # LAG FEATURES
        df['arrivals_lag_1'] = self.recent_avg
        df['arrivals_lag_7'] = self.recent_avg
        df['arrivals_lag_14'] = self.recent_avg
        df['arrivals_lag_30'] = self.recent_avg

        # ROLLING STATISTICS
        df['arrivals_rolling_mean_7'] = self.recent_avg
        df['arrivals_rolling_mean_14'] = self.recent_avg
        df['arrivals_rolling_mean_30'] = self.recent_avg

        df['arrivals_rolling_std_7'] = self.recent_std
        df['arrivals_rolling_std_14'] = self.recent_std
        df['arrivals_rolling_std_30'] = self.recent_std

        # EMA
        df['arrivals_ema_7'] = self.recent_avg
        df['arrivals_ema_30'] = self.recent_avg

        # DIFFERENCING
        df['arrivals_diff_1'] = 0
        df['arrivals_diff_7'] = 0

        logger.info(f"TSformer daily features: {df.shape}")
        return df

# ============================================================================
# PREDICTION ENGINE
# ============================================================================
class ScenarioPredictor:
    """Generate predictions and aggregate to monthly totals"""

    def __init__(self, ensemble: ProductionTouristEnsemble, feature_generator: FutureFeatureGenerator):
        self.ensemble = ensemble
        self.generator = feature_generator

    def predict_scenario(self, scenario: dict) -> pd.DataFrame:
        """Generate DAILY predictions and aggregate to MONTHLY totals"""
        logger.info("="*70)
        logger.info(f"GENERATING PREDICTIONS: {scenario['name'].upper()}")
        logger.info("="*70)

        # Generate daily dates from Dec 2025 to Dec 2030 (with buffer)
        daily_dates = self.generator.generate_daily_dates('2025-12-01', '2030-12-31')

        # Create DAILY features
        df_svr = self.generator.create_svr_features(scenario, daily_dates)
        df_tsformer = self.generator.create_tsformer_features(scenario, daily_dates)

        logger.info("Running ensemble prediction on daily data...")
        daily_predictions, aligned_dates = self.ensemble.predict(df_svr, df_tsformer)

        # Create dataframe with daily predictions
        daily_df = pd.DataFrame({
            'date': pd.to_datetime(aligned_dates),
            'arrivals_daily': daily_predictions
        })

        logger.info(f"Daily predictions: {len(daily_predictions)}")
        logger.info(f"   Date range: {daily_df['date'].min()} to {daily_df['date'].max()}")

        # Filter to 2026-2030 only
        daily_df = daily_df[
            (daily_df['date'] >= '2026-01-01') &
            (daily_df['date'] <= '2030-12-31')
        ].copy()

        logger.info(f"Filtered to 2026-2030: {len(daily_df)} days")

        # Aggregate to MONTHLY totals
        daily_df['year'] = daily_df['date'].dt.year
        daily_df['month'] = daily_df['date'].dt.month

        monthly_df = daily_df.groupby(['year', 'month']).agg({
            'arrivals_daily': 'sum',
            'date': 'min'  # First date of month for labeling
        }).reset_index()

        # Create proper month label
        monthly_df['date'] = pd.to_datetime(
            monthly_df['year'].astype(str) + '-' +
            monthly_df['month'].astype(str).str.zfill(2) + '-01'
        )

        monthly_df = monthly_df.rename(columns={'arrivals_daily': 'arrivals_forecast'})
        monthly_df['scenario'] = scenario['name']

        result_df = monthly_df[['date', 'arrivals_forecast', 'scenario']].sort_values('date')

        logger.info(f"✅ Monthly aggregation complete: {len(result_df)} months")
        logger.info(f"   Date range: {result_df['date'].min()} to {result_df['date'].max()}")
        logger.info(f"   Monthly range: {result_df['arrivals_forecast'].min():.0f} to {result_df['arrivals_forecast'].max():.0f}")
        logger.info(f"   Mean monthly: {result_df['arrivals_forecast'].mean():.0f}")
        logger.info(f"   Total 5-year: {result_df['arrivals_forecast'].sum():.0f}")

        return result_df

# ============================================================================
# VISUALIZATION
# ============================================================================
class ScenarioVisualizer:
    """Create visualizations for forecast scenarios"""

    @staticmethod
    def plot_scenarios(baseline_df, optimistic_df, pessimistic_df, output_dir):
        """Create scenario comparison plot"""
        logger.info("Creating scenario comparison visualization...")

        fig, ax = plt.subplots(figsize=(18, 10))

        # Plot all three scenarios
        ax.plot(baseline_df['date'], baseline_df['arrivals_forecast'],
               label='Baseline', linewidth=3, color='#2E86AB', marker='o', markersize=4)

        ax.plot(optimistic_df['date'], optimistic_df['arrivals_forecast'],
               label='Optimistic', linewidth=3, color='#06A77D', linestyle='--',
               marker='^', markersize=4)

        ax.plot(pessimistic_df['date'], pessimistic_df['arrivals_forecast'],
               label='Pessimistic', linewidth=3, color='#D00000', linestyle='--',
               marker='v', markersize=4)

        # Uncertainty band
        ax.fill_between(baseline_df['date'],
                        pessimistic_df['arrivals_forecast'],
                        optimistic_df['arrivals_forecast'],
                        alpha=0.15, color='gray', label='Uncertainty Range')

        ax.set_xlabel('Date', fontsize=14, fontweight='bold')
        ax.set_ylabel('Monthly Tourist Arrivals', fontsize=14, fontweight='bold')
        ax.set_title('Sri Lanka Monthly Tourist Arrivals Forecast (2026-2030)\nProduction Ensemble Model (SVR + TSformer)',
                    fontsize=17, fontweight='bold', pad=20)

        ax.legend(loc='upper left', fontsize=12, framealpha=0.95)
        ax.grid(True, alpha=0.3, linestyle='--')

        # Format y-axis
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

        # Format x-axis
        plt.xticks(rotation=45, ha='right')

        # Add annotations
        total_baseline = baseline_df['arrivals_forecast'].sum()
        total_optimistic = optimistic_df['arrivals_forecast'].sum()
        total_pessimistic = pessimistic_df['arrivals_forecast'].sum()

        textstr = f'Total 5-Year Arrivals:\n'
        textstr += f'Baseline: {total_baseline:,.0f}\n'
        textstr += f'Optimistic: {total_optimistic:,.0f} (+{((total_optimistic/total_baseline-1)*100):.1f}%)\n'
        textstr += f'Pessimistic: {total_pessimistic:,.0f} ({((total_pessimistic/total_baseline-1)*100):.1f}%)'

        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=11,
                verticalalignment='top', bbox=props)

        plt.tight_layout()

        output_path = Path(output_dir) / 'scenario_forecast_2026_2030.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"Saved: {output_path}")

        plt.close()

    @staticmethod
    def plot_annual_comparison(all_scenarios_df, output_dir):
        """Create annual comparison bar chart"""
        logger.info("Creating annual comparison...")

        all_scenarios_df['year'] = all_scenarios_df['date'].dt.year
        annual_totals = all_scenarios_df.groupby(['scenario', 'year'])['arrivals_forecast'].sum().reset_index()

        pivot_df = annual_totals.pivot(index='year', columns='scenario', values='arrivals_forecast')

        fig, ax = plt.subplots(figsize=(14, 8))
        pivot_df.plot(kind='bar', ax=ax, color=['#2E86AB', '#06A77D', '#D00000'], width=0.8)

        ax.set_xlabel('Year', fontsize=13, fontweight='bold')
        ax.set_ylabel('Total Annual Arrivals', fontsize=13, fontweight='bold')
        ax.set_title('Annual Tourist Arrivals by Scenario (2026-2030)',
                    fontsize=15, fontweight='bold', pad=15)
        ax.legend(title='Scenario', fontsize=11, title_fontsize=12)
        ax.grid(True, alpha=0.3, axis='y')

        # Format y-axis
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

        # Add value labels on bars
        for container in ax.containers:
            ax.bar_label(container, fmt='%.0f', padding=3, fontsize=8)

        plt.xticks(rotation=0)
        plt.tight_layout()

        output_path = Path(output_dir) / 'annual_comparison_2026_2030.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"Saved: {output_path}")

        plt.close()

    @staticmethod
    def plot_monthly_trends(all_scenarios_df, output_dir):
        """Create monthly trend analysis"""
        logger.info("Creating monthly trend analysis...")

        all_scenarios_df['year'] = all_scenarios_df['date'].dt.year
        all_scenarios_df['month'] = all_scenarios_df['date'].dt.month

        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.flatten()

        scenarios = ['Baseline', 'Optimistic', 'Pessimistic']
        colors = ['#2E86AB', '#06A77D', '#D00000']

        month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                      'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

        for idx, year in enumerate(range(2026, 2031)):
            ax = axes[idx]

            for scenario, color in zip(scenarios, colors):
                year_data = all_scenarios_df[
                    (all_scenarios_df['year'] == year) &
                    (all_scenarios_df['scenario'] == scenario)
                ]
                ax.plot(year_data['month'], year_data['arrivals_forecast'],
                       label=scenario, linewidth=2.5, marker='o', color=color, markersize=5)

            ax.set_title(f'{year}', fontsize=13, fontweight='bold')
            ax.set_xlabel('Month', fontsize=11)
            ax.set_ylabel('Monthly Arrivals', fontsize=11)
            ax.legend(fontsize=9)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(range(1, 13))
            ax.set_xticklabels(month_names, rotation=45, ha='right', fontsize=9)
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x/1000)}K'))

        # Hide 6th subplot
        axes[5].axis('off')

        plt.suptitle('Monthly Arrival Patterns by Year and Scenario',
                    fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout()

        output_path = Path(output_dir) / 'monthly_trends_by_year.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"Saved: {output_path}")

        plt.close()

# ============================================================================
# RESULTS EXPORTER
# ============================================================================
class ResultsExporter:
    """Export forecasts and summaries"""

    @staticmethod
    def save_scenarios(baseline_df, optimistic_df, pessimistic_df, output_dir):
        """Save scenario CSVs"""
        logger.info("Saving scenario forecasts...")

        output_path = Path(output_dir) / 'scenario_forecasts'
        output_path.mkdir(parents=True, exist_ok=True)

        baseline_df.to_csv(output_path / 'baseline_2026_2030.csv', index=False)
        optimistic_df.to_csv(output_path / 'optimistic_2026_2030.csv', index=False)
        pessimistic_df.to_csv(output_path / 'pessimistic_2026_2030.csv', index=False)

        logger.info(f"✅ Individual scenario files saved to {output_path}")

    @staticmethod
    def create_comparison_summary(baseline_df, optimistic_df, pessimistic_df, output_dir):
        """Create comprehensive comparison summary"""
        logger.info("Creating comparison summary...")

        all_df = pd.concat([baseline_df, optimistic_df, pessimistic_df])

        output_path = Path(output_dir) / 'scenario_forecasts'
        output_path.mkdir(parents=True, exist_ok=True)

        # Save combined file
        all_df.to_csv(output_path / 'all_scenarios_2026_2030.csv', index=False)
        logger.info(f"✅ Combined scenarios file saved")

        # Calculate summary statistics
        summary = {
            'forecast_period': {
                'start_date': str(baseline_df['date'].min()),
                'end_date': str(baseline_df['date'].max()),
                'total_months': len(baseline_df)
            },
            'scenario_summary': {
                'baseline': {
                    'total_2026_2030': int(baseline_df['arrivals_forecast'].sum()),
                    'mean_monthly': float(baseline_df['arrivals_forecast'].mean()),
                    'median_monthly': float(baseline_df['arrivals_forecast'].median()),
                    'min_monthly': int(baseline_df['arrivals_forecast'].min()),
                    'max_monthly': int(baseline_df['arrivals_forecast'].max()),
                    'std_monthly': float(baseline_df['arrivals_forecast'].std())
                },
                'optimistic': {
                    'total_2026_2030': int(optimistic_df['arrivals_forecast'].sum()),
                    'mean_monthly': float(optimistic_df['arrivals_forecast'].mean()),
                    'median_monthly': float(optimistic_df['arrivals_forecast'].median()),
                    'min_monthly': int(optimistic_df['arrivals_forecast'].min()),
                    'max_monthly': int(optimistic_df['arrivals_forecast'].max()),
                    'std_monthly': float(optimistic_df['arrivals_forecast'].std())
                },
                'pessimistic': {
                    'total_2026_2030': int(pessimistic_df['arrivals_forecast'].sum()),
                    'mean_monthly': float(pessimistic_df['arrivals_forecast'].mean()),
                    'median_monthly': float(pessimistic_df['arrivals_forecast'].median()),
                    'min_monthly': int(pessimistic_df['arrivals_forecast'].min()),
                    'max_monthly': int(pessimistic_df['arrivals_forecast'].max()),
                    'std_monthly': float(pessimistic_df['arrivals_forecast'].std())
                }
            },
            'scenario_comparison': {
                'upside_potential_percent': float(
                    (optimistic_df['arrivals_forecast'].sum() -
                     baseline_df['arrivals_forecast'].sum()) /
                    baseline_df['arrivals_forecast'].sum() * 100
                ),
                'downside_risk_percent': float(
                    (pessimistic_df['arrivals_forecast'].sum() -
                     baseline_df['arrivals_forecast'].sum()) /
                    baseline_df['arrivals_forecast'].sum() * 100
                ),
                'range_total_arrivals': int(
                    optimistic_df['arrivals_forecast'].sum() -
                    pessimistic_df['arrivals_forecast'].sum()
                ),
                'range_percent_of_baseline': float(
                    (optimistic_df['arrivals_forecast'].sum() -
                     pessimistic_df['arrivals_forecast'].sum()) /
                    baseline_df['arrivals_forecast'].sum() * 100
                )
            },
            'annual_breakdowns': {}
        }

        # Annual breakdowns
        for scenario_name, scenario_df in [('baseline', baseline_df),
                                           ('optimistic', optimistic_df),
                                           ('pessimistic', pessimistic_df)]:
            scenario_df['year'] = scenario_df['date'].dt.year
            annual = scenario_df.groupby('year')['arrivals_forecast'].sum()
            summary['annual_breakdowns'][scenario_name] = {
                str(year): int(val) for year, val in annual.items()
            }

        # Save summary JSON
        summary_path = output_path / 'scenario_summary.json'
        with open(summary_path, 'w') as f:
            json.dump(summary, f, indent=4)

        logger.info(f"✅ Summary JSON saved: {summary_path}")

        return summary

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution pipeline"""
    logger.info("="*70)
    logger.info("PHASE 1: MONTHLY FORECASTING (2026-2030) - FIXED VERSION")
    logger.info("="*70)
    logger.info(f"Started: {datetime.now()}")

    try:
        # Configuration
        HISTORICAL_DATA_PATH = 'preprocessed-dataset.csv'
        MODEL_DIR = 'production_model_output'
        OUTPUT_DIR = 'phase1_scenario_forecasts'

        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

        # Step 1: Load production ensemble
        logger.info("\n[STEP 1] Loading Production Ensemble (V3.0)")
        ensemble = ProductionTouristEnsemble.load(MODEL_DIR)

        # Step 2: Load historical data
        logger.info("\n[STEP 2] Loading Historical Data")
        historical_df = pd.read_csv(HISTORICAL_DATA_PATH)
        feature_generator = FutureFeatureGenerator(historical_df)

        # Step 3: Initialize predictor
        logger.info("\n[STEP 3] Initializing Scenario Predictor")
        predictor = ScenarioPredictor(ensemble, feature_generator)

        # Step 4: Generate predictions for all scenarios
        logger.info("\n[STEP 4] Generating Scenario Predictions")
        logger.info("Note: Predicting daily arrivals, then aggregating to monthly totals")

        baseline_forecast = predictor.predict_scenario(ScenarioAssumptions.BASELINE)
        optimistic_forecast = predictor.predict_scenario(ScenarioAssumptions.OPTIMISTIC)
        pessimistic_forecast = predictor.predict_scenario(ScenarioAssumptions.PESSIMISTIC)

        # Step 5: Save results
        logger.info("\n[STEP 5] Saving Results")
        ResultsExporter.save_scenarios(
            baseline_forecast, optimistic_forecast, pessimistic_forecast, OUTPUT_DIR
        )

        summary = ResultsExporter.create_comparison_summary(
            baseline_forecast, optimistic_forecast, pessimistic_forecast, OUTPUT_DIR
        )

        # Step 6: Create visualizations
        logger.info("\n[STEP 6] Creating Visualizations")
        ScenarioVisualizer.plot_scenarios(
            baseline_forecast, optimistic_forecast, pessimistic_forecast, OUTPUT_DIR
        )

        all_scenarios = pd.concat([baseline_forecast, optimistic_forecast, pessimistic_forecast])
        ScenarioVisualizer.plot_annual_comparison(all_scenarios, OUTPUT_DIR)
        ScenarioVisualizer.plot_monthly_trends(all_scenarios, OUTPUT_DIR)

        # Final summary
        logger.info("\n" + "="*70)
        logger.info("✅ PHASE 1 COMPLETED SUCCESSFULLY")
        logger.info("="*70)

        logger.info("\n📊 FORECAST SUMMARY (Jan 2026 - Dec 2030):")
        logger.info(f"\n  Baseline Scenario:")
        logger.info(f"    Total 5-Year Arrivals: {summary['scenario_summary']['baseline']['total_2026_2030']:,}")
        logger.info(f"    Monthly Average: {summary['scenario_summary']['baseline']['mean_monthly']:,.0f}")
        logger.info(f"    Monthly Range: {summary['scenario_summary']['baseline']['min_monthly']:,} - {summary['scenario_summary']['baseline']['max_monthly']:,}")

        logger.info(f"\n  Optimistic Scenario:")
        logger.info(f"    Total 5-Year Arrivals: {summary['scenario_summary']['optimistic']['total_2026_2030']:,}")
        logger.info(f"    Upside Potential: +{summary['scenario_comparison']['upside_potential_percent']:.1f}%")

        logger.info(f"\n  Pessimistic Scenario:")
        logger.info(f"    Total 5-Year Arrivals: {summary['scenario_summary']['pessimistic']['total_2026_2030']:,}")
        logger.info(f"    Downside Risk: {summary['scenario_comparison']['downside_risk_percent']:.1f}%")

        logger.info(f"\n  Forecast Uncertainty:")
        logger.info(f"    Range: {summary['scenario_comparison']['range_total_arrivals']:,} arrivals")
        logger.info(f"    Range (% of baseline): {summary['scenario_comparison']['range_percent_of_baseline']:.1f}%")

        logger.info(f"\n✅ All outputs saved to: {OUTPUT_DIR}/")
        logger.info(f"\nCompleted: {datetime.now()}")

        return baseline_forecast, optimistic_forecast, pessimistic_forecast, summary

    except Exception as e:
        logger.error(f"\n❌ ERROR: {str(e)}", exc_info=True)
        raise

# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    print("="*70)
    print("PHASE 1: MONTHLY FORECASTING (2026-2030) - FIXED")
    print("="*70)
    print("\n🎯 Fixes Applied:")
    print("   1. ✅ Generates DAILY predictions")
    print("   2. ✅ Aggregates to MONTHLY totals (sum of all days in month)")
    print("   3. ✅ Covers full 2026-2030 period (60 months)")
    print("\n📊 Method:")
    print("   - Generate daily features for entire period")
    print("   - Predict daily arrivals using ensemble")
    print("   - Sum daily predictions by month")
    print("\n🚀 Run:")
    print("   python phase1_scenario_forecasting_2026_2030.py")
    print("\n" + "="*70 + "\n")

    baseline, optimistic, pessimistic, summary = main()

    print("\n" + "="*70)
    print("✅ PHASE 1 FORECASTING COMPLETED")
    print("="*70)
    print(f"\nBaseline Total: {summary['scenario_summary']['baseline']['total_2026_2030']:,}")
    print(f"Optimistic Total: {summary['scenario_summary']['optimistic']['total_2026_2030']:,} (+{summary['scenario_comparison']['upside_potential_percent']:.1f}%)")
    print(f"Pessimistic Total: {summary['scenario_summary']['pessimistic']['total_2026_2030']:,} ({summary['scenario_comparison']['downside_risk_percent']:.1f}%)")
    print(f"\nOutputs: phase1_scenario_forecasts/")


In [ ]:
"""
PHASE 2: Model Explainability & External Factor Contribution Analysis
======================================================================
ENHANCED VERSION - Includes:
1. SHAP-based feature importance
2. External factor contribution analysis (Economic, Weather, Trends, Events)
3. Temporal analysis of factor contributions
4. Comparative analysis across scenarios


Author: ML Engineering Team
Date: December 2025
Version: 2.1 - Fixed SHAP dimension mismatch
"""


import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


from datetime import datetime
from pathlib import Path
import logging
import json
import pickle


# Deep Learning
import tensorflow as tf
from tensorflow import keras


# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler


# SHAP for explainability
import shap


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns


# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11


# Random seed
np.random.seed(42)
tf.random.set_seed(42)


# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)


    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'phase2_explainability_{timestamp}.log'


    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )


    return logging.getLogger(__name__)


logger = setup_logging()


# ============================================================================
# FEATURE CATEGORIES
# ============================================================================
class FeatureCategories:
    """Define feature categories for contribution analysis"""


    ECONOMIC = [
        'gdp_per_capita', 'inflation_rate', 'usd_lkr', 'gbp_lkr',
        'eur_lkr', 'rub_lkr', 'inr_lkr', 'cny_lkr', 'brent_crude_price'
    ]


    WEATHER = [
        'temperature', 'precipitation', 'humidity'
    ]


    GOOGLE_TRENDS = [
        'web_search', 'image_search'
    ]


    EVENTS = [
        'event_encoded', 'covid_impact_factor', 'crisis_impact_factor'
    ]


    TEMPORAL = [
        'month', 'quarter', 'day_of_week', 'day_of_month', 'day_of_year',
        'week_of_year', 'year', 'month_sin', 'month_cos', 'day_of_week_sin',
        'day_of_week_cos', 'day_of_year_sin', 'day_of_year_cos'
    ]


    LAG_FEATURES = [
        'arrivals_lag_1', 'arrivals_lag_7', 'arrivals_lag_14', 'arrivals_lag_30'
    ]


    ROLLING_FEATURES = [
        'arrivals_rolling_mean_7', 'arrivals_rolling_mean_14', 'arrivals_rolling_mean_30',
        'arrivals_rolling_std_7', 'arrivals_rolling_std_14', 'arrivals_rolling_std_30',
        'arrivals_ema_7', 'arrivals_ema_30'
    ]


    DIFFERENCING = [
        'arrivals_diff_1', 'arrivals_diff_7'
    ]


    @classmethod
    def get_category(cls, feature_name: str) -> str:
        """Get category for a feature"""
        if feature_name in cls.ECONOMIC:
            return 'Economic'
        elif feature_name in cls.WEATHER:
            return 'Weather'
        elif feature_name in cls.GOOGLE_TRENDS:
            return 'Google Trends'
        elif feature_name in cls.EVENTS:
            return 'Events & Crises'
        elif feature_name in cls.TEMPORAL:
            return 'Temporal/Seasonal'
        elif feature_name in cls.LAG_FEATURES:
            return 'Lag Features'
        elif feature_name in cls.ROLLING_FEATURES:
            return 'Rolling Statistics'
        elif feature_name in cls.DIFFERENCING:
            return 'Differencing'
        else:
            return 'Other'


    @classmethod
    def get_external_categories(cls):
        """Get external factor categories only"""
        return ['Economic', 'Weather', 'Google Trends', 'Events & Crises']


    @classmethod
    def get_all_categories(cls):
        """Get all categories"""
        return ['Economic', 'Weather', 'Google Trends', 'Events & Crises',
                'Temporal/Seasonal', 'Lag Features', 'Rolling Statistics', 'Differencing']


# ============================================================================
# PRODUCTION ENSEMBLE MODEL CLASS (From V3.0)
# ============================================================================
class ProductionTouristEnsemble:
    """Production-Ready Weighted Ensemble Model"""


    def __init__(
        self,
        svr_model,
        svr_scaler,
        tsformer_model,
        weights: dict,
        sequence_length: int = 30
    ):
        self.svr_model = svr_model
        self.svr_scaler = svr_scaler
        self.tsformer_model = tsformer_model
        self.weights = weights
        self.sequence_length = sequence_length


        self.tsformer_scaler_X = StandardScaler()
        self.tsformer_scaler_y = MinMaxScaler()


        self.training_date = None
        self.data_date_range = None
        self.n_training_samples = None


    def predict_svr(self, df_svr: pd.DataFrame) -> np.ndarray:
        """Generate SVR predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]


        X = df_svr[feature_cols].values
        X_scaled = self.svr_scaler.transform(X)
        predictions = self.svr_model.predict(X_scaled)


        return predictions


    def predict_tsformer(self, df_tsformer: pd.DataFrame) -> np.ndarray:
        """Generate TSformer predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]


        X = df_tsformer[feature_cols].values
        X_scaled = self.tsformer_scaler_X.transform(X)


        X_seq = self._create_sequences(X_scaled)


        y_pred_scaled = self.tsformer_model.predict(X_seq, verbose=0).flatten()
        predictions = self.tsformer_scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()


        return predictions


    def _create_sequences(self, data: np.ndarray) -> np.ndarray:
        """Create sequences for TSformer"""
        sequences = []
        for i in range(len(data) - self.sequence_length + 1):
            sequences.append(data[i:i + self.sequence_length])
        return np.array(sequences)


    @classmethod
    def load(cls, model_dir: str = "production_model_output"):
        """Load production ensemble model"""
        logger.info("="*70)
        logger.info("LOADING PRODUCTION ENSEMBLE MODEL")
        logger.info("="*70)


        model_dir = Path(model_dir)


        if not model_dir.exists():
            raise FileNotFoundError(f"Model directory not found: {model_dir}")


        ensemble_files = list(model_dir.glob('production_ensemble_*.pkl'))
        if not ensemble_files:
            raise FileNotFoundError(f"No ensemble model found in {model_dir}")


        ensemble_path = sorted(ensemble_files)[-1]
        logger.info(f"Loading ensemble: {ensemble_path.name}")


        with open(ensemble_path, 'rb') as f:
            ensemble_dict = pickle.load(f)


        timestamp = ensemble_path.stem.split('_')[-1]


        tsformer_patterns = [
            f'production_ensemble_tsformer_{timestamp}.keras',
            f'production_ensemble_tsformer_{timestamp}.h5',
            'production_ensemble_tsformer_*.keras',
            'production_ensemble_tsformer_*.h5'
        ]


        tsformer_path = None
        for pattern in tsformer_patterns:
            files = list(model_dir.glob(pattern))
            if files:
                tsformer_path = sorted(files)[-1]
                break


        if not tsformer_path:
            raise FileNotFoundError(f"No TSformer model found in {model_dir}")


        logger.info(f"Loading TSformer: {tsformer_path.name}")
        tsformer_model = keras.models.load_model(tsformer_path)


        ensemble = cls(
            svr_model=ensemble_dict['svr_model'],
            svr_scaler=ensemble_dict['svr_scaler'],
            tsformer_model=tsformer_model,
            weights=ensemble_dict['weights'],
            sequence_length=ensemble_dict['sequence_length']
        )


        ensemble.tsformer_scaler_X = ensemble_dict['tsformer_scaler_X']
        ensemble.tsformer_scaler_y = ensemble_dict['tsformer_scaler_y']


        metadata = ensemble_dict.get('training_metadata', {})
        ensemble.training_date = metadata.get('training_date')
        ensemble.data_date_range = metadata.get('data_date_range')
        ensemble.n_training_samples = metadata.get('n_training_samples')


        logger.info("✅ Ensemble loaded successfully")
        logger.info(f"Weights: SVR={ensemble.weights['svr']:.6f}, TSformer={ensemble.weights['tsformer']:.6f}")


        return ensemble


# ============================================================================
# DATA PREPARATION
# ============================================================================
class SHAPDataPreparator:
    """Prepare data for SHAP analysis"""


    def __init__(self, df: pd.DataFrame):
        self.df = df.copy()
        self.df['date'] = pd.to_datetime(self.df['date'])
        self.df = self.df.sort_values('date').reset_index(drop=True)


        logger.info(f"Data loaded for SHAP: {len(self.df)} records")


    def prepare_svr_features(self) -> tuple:
        """Prepare SVR features"""
        df = self.df.copy()


        # Temporal features
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)


        # Cyclical encoding
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)


        # Lag features
        for lag in [7, 14, 30]:
            df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)


        # Rolling statistics
        for window in [7, 14, 30]:
            df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
                window=window, min_periods=1
            ).mean()
            df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
                window=window, min_periods=1
            ).std()


        df = df.dropna()


        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df.columns if col not in exclude_cols]


        X = df[feature_cols].values
        y = df['arrivals'].values
        dates = df['date'].values


        logger.info(f"SVR features prepared: {X.shape}")
        return X, y, dates, feature_cols, df


    def prepare_tsformer_features(self) -> tuple:
        """Prepare TSformer features"""
        df = self.df.copy()


        # Temporal features (includes year)
        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
        df['year'] = df['date'].dt.year
        df['day_of_year'] = df['date'].dt.dayofyear


        # Cyclical encoding
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
        df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)


        # Lag features
        for lag in [1, 7, 14, 30]:
            df[f'arrivals_lag_{lag}'] = df['arrivals'].shift(lag)


        # Rolling statistics
        for window in [7, 14, 30]:
            df[f'arrivals_rolling_mean_{window}'] = df['arrivals'].rolling(
                window=window, min_periods=1
            ).mean()
            df[f'arrivals_rolling_std_{window}'] = df['arrivals'].rolling(
                window=window, min_periods=1
            ).std()


        # EMA
        df['arrivals_ema_7'] = df['arrivals'].ewm(span=7, adjust=False).mean()
        df['arrivals_ema_30'] = df['arrivals'].ewm(span=30, adjust=False).mean()


        # Differencing
        df['arrivals_diff_1'] = df['arrivals'].diff(1)
        df['arrivals_diff_7'] = df['arrivals'].diff(7)


        df = df.fillna(method='ffill').fillna(method='bfill')


        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df.columns if col not in exclude_cols]


        X = df[feature_cols].values
        y = df['arrivals'].values
        dates = df['date'].values


        logger.info(f"TSformer features prepared: {X.shape}")
        return X, y, dates, feature_cols, df


# ============================================================================
# EXTERNAL FACTOR CONTRIBUTION ANALYZER
# ============================================================================
class ExternalFactorAnalyzer:
    """Analyze contributions of external factors vs internal features"""


    def __init__(self, output_dir: Path):
        self.output_dir = output_dir
        self.output_dir.mkdir(parents=True, exist_ok=True)


    def calculate_category_contributions(self, shap_values: np.ndarray,
                                        feature_names: list,
                                        model_name: str) -> pd.DataFrame:
        """Calculate contributions by feature category"""
        logger.info(f"Calculating category contributions for {model_name}...")


        # Get mean absolute SHAP value per feature
        mean_abs_shap = np.abs(shap_values).mean(axis=0)


        # Create feature importance dataframe
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'mean_abs_shap': mean_abs_shap
        })


        # Assign categories
        feature_importance['category'] = feature_importance['feature'].apply(
            FeatureCategories.get_category
        )


        # Aggregate by category
        category_contributions = feature_importance.groupby('category').agg({
            'mean_abs_shap': 'sum'
        }).reset_index()


        category_contributions = category_contributions.sort_values(
            'mean_abs_shap', ascending=False
        )


        # Calculate percentages
        total_importance = category_contributions['mean_abs_shap'].sum()
        category_contributions['percentage'] = (
            category_contributions['mean_abs_shap'] / total_importance * 100
        )


        category_contributions['model'] = model_name


        logger.info(f"✅ Category contributions calculated for {model_name}")
        return category_contributions


    def visualize_category_contributions(self, svr_contrib, tsformer_contrib):
        """Create visualization of category contributions"""
        logger.info("Creating category contribution visualizations...")


        # 1. Stacked Bar Chart - External vs Internal
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))


        for idx, (contrib, model_name) in enumerate([(svr_contrib, 'SVR'),
                                                     (tsformer_contrib, 'TSformer')]):
            ax = axes[idx]


            # Separate external and internal
            external_cats = FeatureCategories.get_external_categories()
            contrib['factor_type'] = contrib['category'].apply(
                lambda x: 'External Factors' if x in external_cats else 'Internal Features'
            )


            type_summary = contrib.groupby('factor_type')['percentage'].sum().sort_values()


            colors = ['#3498db', '#e74c3c']  # Blue for internal, red for external
            bars = ax.barh(type_summary.index, type_summary.values, color=colors)


            ax.set_xlabel('Contribution (%)', fontsize=13, fontweight='bold')
            ax.set_title(f'{model_name} Model\nExternal vs Internal Factors',
                        fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')


            # Add percentage labels
            for bar in bars:
                width = bar.get_width()
                ax.text(width + 1, bar.get_y() + bar.get_height()/2,
                       f'{width:.1f}%', ha='left', va='center', fontsize=11, fontweight='bold')


        plt.tight_layout()
        plt.savefig(self.output_dir / 'external_vs_internal_contributions.png',
                    dpi=300, bbox_inches='tight')
        plt.close()
        logger.info("✅ Saved: external_vs_internal_contributions.png")


        # 2. Detailed Category Breakdown
        fig, axes = plt.subplots(1, 2, figsize=(18, 10))


        for idx, (contrib, model_name) in enumerate([(svr_contrib, 'SVR'),
                                                     (tsformer_contrib, 'TSformer')]):
            ax = axes[idx]


            # Sort by contribution
            contrib_sorted = contrib.sort_values('percentage', ascending=True)


            # Color by external vs internal
            external_cats = FeatureCategories.get_external_categories()
            colors = ['#e74c3c' if cat in external_cats else '#3498db'
                      for cat in contrib_sorted['category']]


            bars = ax.barh(contrib_sorted['category'], contrib_sorted['percentage'],
                           color=colors, edgecolor='black', linewidth=1.2)


            ax.set_xlabel('Contribution (%)', fontsize=12, fontweight='bold')
            ax.set_title(f'{model_name} Model\nFeature Category Contributions',
                        fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')


            # Add percentage labels
            for bar in bars:
                width = bar.get_width()
                ax.text(width + 0.5, bar.get_y() + bar.get_height()/2,
                        f'{width:.1f}%', ha='left', va='center', fontsize=10)


        # Add legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#e74c3c', edgecolor='black', label='External Factors'),
            Patch(facecolor='#3498db', edgecolor='black', label='Internal Features')
        ]
        fig.legend(handles=legend_elements, loc='upper center',
                   bbox_to_anchor=(0.5, 0.02), ncol=2, fontsize=12)


        plt.tight_layout(rect=[0, 0.03, 1, 1])
        plt.savefig(self.output_dir / 'category_contributions_detailed.png',
                    dpi=300, bbox_inches='tight')
        plt.close()
        logger.info("✅ Saved: category_contributions_detailed.png")


        # 3. External Factors Only - Detailed Breakdown
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))


        external_cats = FeatureCategories.get_external_categories()


        for idx, (contrib, model_name) in enumerate([(svr_contrib, 'SVR'),
                                                     (tsformer_contrib, 'TSformer')]):
            ax = axes[idx]


            # Filter external only
            external_contrib = contrib[contrib['category'].isin(external_cats)].copy()


            if len(external_contrib) > 0:
                # Recalculate percentages relative to external factors only
                total_external = external_contrib['mean_abs_shap'].sum()
                external_contrib['external_percentage'] = (
                    external_contrib['mean_abs_shap'] / total_external * 100
                )


                external_contrib = external_contrib.sort_values('external_percentage', ascending=True)


                colors_map = {
                    'Economic': '#2ecc71',
                    'Weather': '#f39c12',
                    'Google Trends': '#9b59b6',
                    'Events & Crises': '#e74c3c'
                }
                colors = [colors_map.get(cat, '#95a5a6') for cat in external_contrib['category']]


                bars = ax.barh(external_contrib['category'],
                              external_contrib['external_percentage'],
                              color=colors, edgecolor='black', linewidth=1.2)


                ax.set_xlabel('Contribution (% of External Factors)', fontsize=12, fontweight='bold')
                ax.set_title(f'{model_name} Model\nExternal Factor Breakdown',
                            fontsize=14, fontweight='bold')
                ax.grid(True, alpha=0.3, axis='x')


                # Add percentage labels
                for bar in bars:
                    width = bar.get_width()
                    ax.text(width + 1, bar.get_y() + bar.get_height()/2,
                           f'{width:.1f}%', ha='left', va='center', fontsize=11, fontweight='bold')


        plt.tight_layout()
        plt.savefig(self.output_dir / 'external_factors_breakdown.png',
                    dpi=300, bbox_inches='tight')
        plt.close()
        logger.info("✅ Saved: external_factors_breakdown.png")


    def create_contribution_report(self, svr_contrib, tsformer_contrib):
        """Create detailed contribution report"""
        logger.info("Creating contribution report...")


        # Save category contributions
        svr_contrib.to_csv(self.output_dir / 'svr_category_contributions.csv', index=False)
        tsformer_contrib.to_csv(self.output_dir / 'tsformer_category_contributions.csv', index=False)


        # Calculate external vs internal
        external_cats = FeatureCategories.get_external_categories()


        def get_external_internal_split(contrib):
            external = contrib[contrib['category'].isin(external_cats)]['percentage'].sum()
            internal = contrib[~contrib['category'].isin(external_cats)]['percentage'].sum()
            return {'external': float(external), 'internal': float(internal)}


        svr_split = get_external_internal_split(svr_contrib)
        tsformer_split = get_external_internal_split(tsformer_contrib)


        # Create JSON summary
        report = {
            'analysis_date': str(datetime.now()),
            'summary': {
                'svr_model': {
                    'external_factors_contribution_pct': svr_split['external'],
                    'internal_features_contribution_pct': svr_split['internal'],
                    'top_category': svr_contrib.iloc[0]['category'],
                    'top_category_contribution_pct': float(svr_contrib.iloc[0]['percentage'])
                },
                'tsformer_model': {
                    'external_factors_contribution_pct': tsformer_split['external'],
                    'internal_features_contribution_pct': tsformer_split['internal'],
                    'top_category': tsformer_contrib.iloc[0]['category'],
                    'top_category_contribution_pct': float(tsformer_contrib.iloc[0]['percentage'])
                }
            },
            'svr_category_breakdown': [
                {
                    'category': row['category'],
                    'contribution_pct': float(row['percentage']),
                    'mean_abs_shap': float(row['mean_abs_shap'])
                }
                for _, row in svr_contrib.iterrows()
            ],
            'tsformer_category_breakdown': [
                {
                    'category': row['category'],
                    'contribution_pct': float(row['percentage']),
                    'mean_abs_shap': float(row['mean_abs_shap'])
                }
                for _, row in tsformer_contrib.iterrows()
            ],
            'key_insights': {
                'svr_relies_more_on': 'external_factors' if svr_split['external'] > svr_split['internal'] else 'internal_features',
                'tsformer_relies_more_on': 'external_factors' if tsformer_split['external'] > tsformer_split['internal'] else 'internal_features',
                'difference_in_external_reliance_pct': float(abs(svr_split['external'] - tsformer_split['external']))
            }
        }


        with open(self.output_dir / 'external_factor_contribution_report.json', 'w') as f:
            json.dump(report, f, indent=4)


        logger.info("✅ Contribution report saved")


        return report


# ============================================================================
# SHAP EXPLAINER
# ============================================================================
class EnsembleExplainer:
    """SHAP-based explainability for ensemble model"""


    def __init__(self, ensemble: ProductionTouristEnsemble, output_dir: str):
        self.ensemble = ensemble
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)


        logger.info("Ensemble explainer initialized")


    def explain_svr(self, X: np.ndarray, feature_names: list, sample_size: int = 500):
        """Generate SHAP explanations for SVR model"""
        logger.info("="*70)
        logger.info("GENERATING SHAP EXPLANATIONS FOR SVR")
        logger.info("="*70)


        X_scaled = self.ensemble.svr_scaler.transform(X)


        if len(X_scaled) > sample_size:
            indices = np.random.choice(len(X_scaled), sample_size, replace=False)
            X_sample = X_scaled[indices]
            logger.info(f"Using {sample_size} samples for SHAP computation")
        else:
            X_sample = X_scaled
            logger.info(f"Using all {len(X_scaled)} samples")


        logger.info("Creating SHAP KernelExplainer for SVR...")
        explainer = shap.KernelExplainer(
            self.ensemble.svr_model.predict,
            shap.sample(X_sample, min(100, len(X_sample)))
        )


        logger.info("Computing SHAP values...")
        shap_values = explainer.shap_values(X_sample)


        logger.info(f"✅ SHAP values computed: {shap_values.shape}")


        return shap_values, X_sample, feature_names


    def explain_tsformer(self, X: np.ndarray, feature_names: list, sample_size: int = 200):
        """Generate SHAP explanations for TSformer model"""
        logger.info("="*70)
        logger.info("GENERATING SHAP EXPLANATIONS FOR TSFORMER")
        logger.info("="*70)


        X_scaled = self.ensemble.tsformer_scaler_X.transform(X)
        X_seq = self.ensemble._create_sequences(X_scaled)


        if len(X_seq) > sample_size:
            indices = np.random.choice(len(X_seq), sample_size, replace=False)
            X_sample = X_seq[indices]
            logger.info(f"Using {sample_size} samples for SHAP computation")
        else:
            X_sample = X_seq
            logger.info(f"Using all {len(X_seq)} samples")


        logger.info("Creating SHAP GradientExplainer for TSformer...")


        # GradientExplainer works with tensors; use a smaller background set
        background = X_sample[:min(50, len(X_sample))]
        explainer = shap.GradientExplainer(
            self.ensemble.tsformer_model,
            background
        )


        logger.info("Computing SHAP values...")
        # shap_values is typically a list for multi-output; take first element if so
        shap_values = explainer.shap_values(X_sample)
        if isinstance(shap_values, list):
            shap_values = shap_values[0]


        # SHAP values shape: (samples, timesteps, features)
        # We need to average over timesteps to get feature importance
        # or take the last timestep if that's more relevant.
        # Typically averaging over the sequence length is robust for importance.
        shap_values_avg = np.mean(shap_values, axis=1) # Shape: (samples, features)


        logger.info(f"✅ SHAP values computed & averaged: {shap_values_avg.shape}")


        # The X input to summary_plot must match shap_values_avg dimensions.
        # Since we averaged SHAP values over time, we should also average X over time
        # OR take the last time step to represent the "current" feature value.
        # Taking the last time step is usually better for interpretation (value at t).
        X_flat = X_sample[:, -1, :]

        # Alternatively, if you want average feature value over window:
        # X_flat = np.mean(X_sample, axis=1)


        logger.info(f"Flattened X shape for plotting: {X_flat.shape}")


        return shap_values_avg, X_flat, feature_names


    def visualize_feature_importance(self, shap_values, X_sample, feature_names, model_name):
        """Create feature importance visualizations"""
        logger.info(f"Creating {model_name} visualizations...")


        # Ensure feature names match columns
        if len(feature_names) != X_sample.shape[1]:
            logger.warning(f"Feature name count ({len(feature_names)}) != X columns ({X_sample.shape[1]}). Truncating/Adjusting.")
            # Fallback: create generic names if mismatch persists
            if len(feature_names) < X_sample.shape[1]:
                 feature_names = feature_names + [f"Feature {i}" for i in range(len(feature_names), X_sample.shape[1])]
            else:
                 feature_names = feature_names[:X_sample.shape[1]]


        # Summary Plot
        plt.figure(figsize=(12, 10))
        shap.summary_plot(
            shap_values,
            X_sample,
            feature_names=feature_names,
            show=False,
            max_display=20
        )
        plt.title(f'{model_name} Model - Feature Importance (SHAP)',
                 fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(self.output_dir / f'{model_name.lower()}_feature_importance.png',
                    dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"✅ Saved: {model_name.lower()}_feature_importance.png")


        # Bar Plot
        plt.figure(figsize=(12, 10))
        shap.summary_plot(
            shap_values,
            X_sample,
            feature_names=feature_names,
            plot_type='bar',
            show=False,
            max_display=20
        )
        plt.title(f'{model_name} Model - Top 20 Features by Mean |SHAP|',
                 fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.savefig(self.output_dir / f'{model_name.lower()}_feature_importance_bar.png',
                    dpi=300, bbox_inches='tight')
        plt.close()
        logger.info(f"✅ Saved: {model_name.lower()}_feature_importance_bar.png")


# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution pipeline"""
    logger.info("="*70)
    logger.info("PHASE 2: EXPLAINABILITY & EXTERNAL FACTOR ANALYSIS")
    logger.info("="*70)
    logger.info(f"Started: {datetime.now()}")


    try:
        # Configuration
        HISTORICAL_DATA_PATH = 'preprocessed-dataset.csv'
        MODEL_DIR = 'production_model_output'
        OUTPUT_DIR = 'phase2_explainability'


        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


        # Step 1: Load production ensemble
        logger.info("\n[STEP 1] Loading Production Ensemble Model")
        ensemble = ProductionTouristEnsemble.load(MODEL_DIR)


        # Step 2: Load and prepare data
        logger.info("\n[STEP 2] Loading Historical Data")
        df = pd.read_csv(HISTORICAL_DATA_PATH)
        data_prep = SHAPDataPreparator(df)


        # Step 3: Prepare Features for Analysis
        logger.info("\n[STEP 3] Preparing Features for Analysis")
        X_svr, y_svr, dates_svr, feature_names_svr, df_svr = data_prep.prepare_svr_features()
        X_tsformer, y_tsformer, dates_tsformer, feature_names_tsformer, df_tsformer = data_prep.prepare_tsformer_features()


        # Step 4: Initialize explainer
        logger.info("\n[STEP 4] Initializing Ensemble Explainer")
        explainer = EnsembleExplainer(ensemble, OUTPUT_DIR)


        # Step 5: Generate SHAP explanations
        logger.info("\n[STEP 5] Generating SHAP Explanations for SVR")
        shap_values_svr, X_sample_svr, _ = explainer.explain_svr(
            X_svr, feature_names_svr, sample_size=1
        )


        logger.info("\n[STEP 6] Generating SHAP Explanations for TSformer")
        shap_values_tsformer, X_sample_tsformer, _ = explainer.explain_tsformer(
            X_tsformer, feature_names_tsformer, sample_size=2
        )


        # Step 7: Create visualizations
        logger.info("\n[STEP 7] Creating SHAP Visualizations")
        explainer.visualize_feature_importance(shap_values_svr, X_sample_svr,
                                              feature_names_svr, 'SVR')
        explainer.visualize_feature_importance(shap_values_tsformer, X_sample_tsformer,
                                              feature_names_tsformer, 'TSformer')


        # Step 8: Analyze external factor contributions
        logger.info("\n[STEP 8] Analyzing External Factor Contributions")
        factor_analyzer = ExternalFactorAnalyzer(Path(OUTPUT_DIR))


        svr_contrib = factor_analyzer.calculate_category_contributions(
            shap_values_svr, feature_names_svr, 'SVR'
        )


        tsformer_contrib = factor_analyzer.calculate_category_contributions(
            shap_values_tsformer, feature_names_tsformer, 'TSformer'
        )


        # Step 9: Create contribution visualizations
        logger.info("\n[STEP 9] Creating Contribution Visualizations")
        factor_analyzer.visualize_category_contributions(svr_contrib, tsformer_contrib)


        # Step 10: Generate reports
        logger.info("\n[STEP 10] Generating Reports")
        contribution_report = factor_analyzer.create_contribution_report(svr_contrib, tsformer_contrib)


        # Final summary
        logger.info("\n" + "="*70)
        logger.info("✅ PHASE 2 ANALYSIS COMPLETED")
        logger.info("="*70)


        logger.info("\n📊 EXTERNAL FACTOR CONTRIBUTIONS:")


        logger.info("\n  SVR Model:")
        logger.info(f"    External Factors: {contribution_report['summary']['svr_model']['external_factors_contribution_pct']:.1f}%")
        logger.info(f"    Internal Features: {contribution_report['summary']['svr_model']['internal_features_contribution_pct']:.1f}%")
        logger.info(f"    Top Category: {contribution_report['summary']['svr_model']['top_category']}")


        logger.info("\n  TSformer Model:")
        logger.info(f"    External Factors: {contribution_report['summary']['tsformer_model']['external_factors_contribution_pct']:.1f}%")
        logger.info(f"    Internal Features: {contribution_report['summary']['tsformer_model']['internal_features_contribution_pct']:.1f}%")
        logger.info(f"    Top Category: {contribution_report['summary']['tsformer_model']['top_category']}")


        logger.info(f"\n✅ All outputs saved to: {OUTPUT_DIR}/")
        logger.info(f"\nCompleted: {datetime.now()}")


        return contribution_report


    except Exception as e:
        logger.error(f"\n❌ ERROR: {str(e)}", exc_info=True)
        raise


# ============================================================================
# ENTRY POINT
# ============================================================================
if __name__ == "__main__":
    print("="*70)
    print("PHASE 2: EXPLAINABILITY & EXTERNAL FACTOR ANALYSIS")
    print("="*70)
    print("\n🎯 Enhanced Analysis:")
    print("   1. SHAP-based feature importance")
    print("   2. External factor contribution analysis")
    print("   3. Category-wise breakdown")
    print("   4. External vs Internal comparison")
    print("\n📊 External Factor Categories:")
    print("   - Economic (GDP, exchange rates, oil prices)")
    print("   - Weather (temperature, precipitation, humidity)")
    print("   - Google Trends (web search, image search)")
    print("   - Events & Crises (COVID, major events)")
    print("\n⚠️  Runtime: 5-10 minutes")
    print("\n" + "="*70 + "\n")


    report = main()


    print("\n" + "="*70)
    print("✅ PHASE 2 ANALYSIS COMPLETED")
    print("="*70)
    print(f"\nOutputs: phase2_explainability/")


In [ ]:
"""
PHASE 1: Scenario-Based Monthly Forecasting (2026-2030) - FINAL CORRECTED v6.0
==============================================================================
Version 6.0 - ROOT CAUSE FIXED
- Applied SCENARIO-SPECIFIC multipliers (not uniform)
- Optimistic: 1.3x GDP, 1.25x FX, 1.35x Trends, 1.2x Events
- Baseline: 1.0x (no scaling)
- Pessimistic: 0.75x GDP, 0.8x FX, 0.7x Trends, 0.75x Events
- This FORCES the correct ordering: Optimistic > Baseline > Pessimistic

Author: ML Engineering Team
Date: December 2025
"""

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
from pathlib import Path
import logging
import json
import pickle
from calendar import monthrange

# Deep Learning
import tensorflow as tf
from tensorflow import keras

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 9)
plt.rcParams['font.size'] = 11

# Random seed
np.random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# LOGGING CONFIGURATION
# ============================================================================
def setup_logging():
    """Configure logging"""
    log_dir = Path('logs')
    log_dir.mkdir(exist_ok=True)

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_file = log_dir / f'phase1_forecasting_{timestamp}.log'

    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler(log_file),
            logging.StreamHandler()
        ]
    )

    return logging.getLogger(__name__)

logger = setup_logging()

# ============================================================================
# FEATURE CATEGORIES FOR EXPLAINABILITY
# ============================================================================
class FeatureCategories:
    """Define the 5 external factor categories"""

    ECONOMIC_INDICATORS = [
        'brent_crude_price', 'gdp_per_capita', 'inflation_rate'
    ]

    EXCHANGE_RATES = [
        'cny_lkr', 'eur_lkr', 'gbp_lkr', 'usd_lkr', 'inr_lkr', 'rub_lkr'
    ]

    EVENTS = [
        'event_encoded'
    ]

    WEATHER = [
        'humidity', 'precipitation', 'temperature'
    ]

    GOOGLE_TRENDS = [
        'image_search', 'web_search'
    ]

    @classmethod
    def get_category(cls, feature_name: str) -> str:
        """Get category for a feature"""
        if feature_name in cls.ECONOMIC_INDICATORS:
            return 'Economic_Indicators'
        elif feature_name in cls.EXCHANGE_RATES:
            return 'Exchange_Rates'
        elif feature_name in cls.EVENTS:
            return 'Events'
        elif feature_name in cls.WEATHER:
            return 'Weather'
        elif feature_name in cls.GOOGLE_TRENDS:
            return 'Google_Trends'
        else:
            return 'Internal_Features'

    @classmethod
    def is_external(cls, feature_name: str) -> bool:
        """Check if feature is external factor"""
        all_external = (cls.ECONOMIC_INDICATORS + cls.EXCHANGE_RATES +
                       cls.EVENTS + cls.WEATHER + cls.GOOGLE_TRENDS)
        return feature_name in all_external

    @classmethod
    def get_all_external_features(cls):
        """Get all external features"""
        return (cls.ECONOMIC_INDICATORS + cls.EXCHANGE_RATES +
                cls.EVENTS + cls.WEATHER + cls.GOOGLE_TRENDS)

# ============================================================================
# SCENARIO ASSUMPTIONS (EVIDENCE-BASED)
# ============================================================================
class ScenarioAssumptions:
    """Define assumptions for each scenario - based on official sources"""

    BASELINE = {
        'name': 'Baseline',
        'description': 'World Bank forecast + Government recovery plan (50% probability)',
        'probability': 0.50,
        'gdp_growth_rate': 0.035,
        'inflation_rate': 0.05,
        'exchange_rate_change': 0.01,
        'brent_crude_change': 0.00,
        'temperature_adjustment': 0.0,
        'precipitation_adjustment': 1.0,
        'google_trends_growth_rate': 0.04,
        'google_trends_initial_boost': 1.0,
        'event_frequency_multiplier': 1.0
    }

    OPTIMISTIC = {
        'name': 'Optimistic',
        'description': 'Government targets + Strong policy implementation (25% probability)',
        'probability': 0.25,
        'gdp_growth_rate': 0.06,
        'inflation_rate': 0.04,
        'exchange_rate_change': -0.04,
        'brent_crude_change': -0.08,
        'temperature_adjustment': -1.0,
        'precipitation_adjustment': 0.85,
        'google_trends_growth_rate': 0.08,
        'google_trends_initial_boost': 1.15,
        'event_frequency_multiplier': 1.25
    }

    PESSIMISTIC = {
        'name': 'Pessimistic',
        'description': 'Slow recovery + Global headwinds (25% probability)',
        'probability': 0.25,
        'gdp_growth_rate': 0.025,
        'inflation_rate': 0.07,
        'exchange_rate_change': 0.05,
        'brent_crude_change': 0.12,
        'temperature_adjustment': 2.0,
        'precipitation_adjustment': 1.25,
        'google_trends_growth_rate': -0.01,
        'google_trends_initial_boost': 0.85,
        'event_frequency_multiplier': 0.75
    }

# ============================================================================
# PRODUCTION ENSEMBLE MODEL CLASS
# ============================================================================
class ProductionTouristEnsemble:
    """Production-Ready Weighted Ensemble Model"""

    def __init__(
        self,
        svr_model,
        svr_scaler,
        tsformer_model,
        weights: dict,
        sequence_length: int = 30
    ):
        self.svr_model = svr_model
        self.svr_scaler = svr_scaler
        self.tsformer_model = tsformer_model
        self.weights = weights
        self.sequence_length = sequence_length
        self.tsformer_scaler_X = StandardScaler()
        self.tsformer_scaler_y = MinMaxScaler()
        self.training_date = None
        self.data_date_range = None
        self.n_training_samples = None

    def predict_svr(self, df_svr: pd.DataFrame) -> np.ndarray:
        """Generate SVR predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]
        X = df_svr[feature_cols].values
        X_scaled = self.svr_scaler.transform(X)
        predictions = self.svr_model.predict(X_scaled)
        return predictions

    def predict_tsformer(self, df_tsformer: pd.DataFrame) -> np.ndarray:
        """Generate TSformer predictions"""
        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_tsformer.columns if col not in exclude_cols]
        X = df_tsformer[feature_cols].values
        X_scaled = self.tsformer_scaler_X.transform(X)
        X_seq = self._create_sequences(X_scaled)
        y_pred_scaled = self.tsformer_model.predict(X_seq, verbose=0).flatten()
        predictions = self.tsformer_scaler_y.inverse_transform(
            y_pred_scaled.reshape(-1, 1)
        ).flatten()
        return predictions

    def _create_sequences(self, data: np.ndarray) -> np.ndarray:
        """Create sequences for TSformer"""
        sequences = []
        for i in range(len(data) - self.sequence_length + 1):
            sequences.append(data[i:i + self.sequence_length])
        return np.array(sequences)

    def predict_with_contributions(self, df_svr: pd.DataFrame, df_tsformer: pd.DataFrame) -> tuple:
        """Generate ensemble predictions with individual model contributions"""
        svr_pred = self.predict_svr(df_svr)
        tsformer_pred = self.predict_tsformer(df_tsformer)
        min_len = min(len(svr_pred), len(tsformer_pred))
        svr_pred_aligned = svr_pred[-min_len:]
        tsformer_pred_aligned = tsformer_pred[-min_len:]
        dates_aligned = df_svr['date'].iloc[-min_len:].values
        svr_contribution = self.weights['svr'] * svr_pred_aligned
        tsformer_contribution = self.weights['tsformer'] * tsformer_pred_aligned
        ensemble_pred = svr_contribution + tsformer_contribution
        return ensemble_pred, dates_aligned, svr_contribution, tsformer_contribution, svr_pred_aligned, tsformer_pred_aligned

    @classmethod
    def load(cls, model_dir: str = "production_model_output"):
        """Load production ensemble model"""
        logger.info("="*70)
        logger.info("LOADING PRODUCTION ENSEMBLE MODEL")
        logger.info("="*70)

        model_dir = Path(model_dir)
        if not model_dir.exists():
            raise FileNotFoundError(f"Model directory not found: {model_dir}")

        ensemble_files = list(model_dir.glob('production_ensemble_*.pkl'))
        if not ensemble_files:
            raise FileNotFoundError(f"No ensemble model found in {model_dir}")

        ensemble_path = sorted(ensemble_files)[-1]
        logger.info(f"Loading ensemble: {ensemble_path.name}")

        with open(ensemble_path, 'rb') as f:
            ensemble_dict = pickle.load(f)

        timestamp = ensemble_path.stem.split('_')[-1]
        tsformer_patterns = [
            f'production_ensemble_tsformer_{timestamp}.keras',
            f'production_ensemble_tsformer_{timestamp}.h5',
            'production_ensemble_tsformer_*.keras',
            'production_ensemble_tsformer_*.h5'
        ]

        tsformer_path = None
        for pattern in tsformer_patterns:
            files = list(model_dir.glob(pattern))
            if files:
                tsformer_path = sorted(files)[-1]
                break

        if not tsformer_path:
            raise FileNotFoundError(f"No TSformer model found in {model_dir}")

        logger.info(f"Loading TSformer: {tsformer_path.name}")
        tsformer_model = keras.models.load_model(tsformer_path)

        ensemble = cls(
            svr_model=ensemble_dict['svr_model'],
            svr_scaler=ensemble_dict['svr_scaler'],
            tsformer_model=tsformer_model,
            weights=ensemble_dict['weights'],
            sequence_length=ensemble_dict['sequence_length']
        )

        ensemble.tsformer_scaler_X = ensemble_dict['tsformer_scaler_X']
        ensemble.tsformer_scaler_y = ensemble_dict['tsformer_scaler_y']

        metadata = ensemble_dict.get('training_metadata', {})
        ensemble.training_date = metadata.get('training_date')
        ensemble.data_date_range = metadata.get('data_date_range')
        ensemble.n_training_samples = metadata.get('n_training_samples')

        logger.info("✅ Ensemble loaded successfully")
        logger.info(f"Weights: SVR={ensemble.weights['svr']:.6f}, TSformer={ensemble.weights['tsformer']:.6f}")
        logger.info(f"Training samples: {ensemble.n_training_samples}")

        return ensemble

# ============================================================================
# FUTURE FEATURE GENERATOR (FULLY CORRECTED v6.0)
# ============================================================================
class FutureFeatureGenerator:
    """Generate DAILY features with SCENARIO-SPECIFIC multipliers"""

    def __init__(self, historical_df: pd.DataFrame):
        self.historical_df = historical_df.copy()
        self.historical_df['date'] = pd.to_datetime(self.historical_df['date'])
        self.historical_df = self.historical_df.sort_values('date')
        self.last_row = self.historical_df.iloc[-1]
        self.last_date = self.last_row['date']
        self._calculate_historical_baselines()
        logger.info(f"Historical data loaded: {len(self.historical_df)} records")
        logger.info(f"Last date: {self.last_date}")

    def _calculate_historical_baselines(self):
        """Calculate baseline values from historical data"""
        recent_df = self.historical_df.tail(90)
        self.baseline_gdp = self.last_row['gdp_per_capita']
        self.baseline_inflation = recent_df['inflation_rate'].mean()
        self.baseline_brent = recent_df['brent_crude_price'].mean()
        self.baseline_usd_lkr = recent_df['usd_lkr'].mean()
        self.baseline_gbp_lkr = recent_df['gbp_lkr'].mean()
        self.baseline_eur_lkr = recent_df['eur_lkr'].mean()
        self.baseline_rub_lkr = recent_df['rub_lkr'].mean()
        self.baseline_inr_lkr = recent_df['inr_lkr'].mean()
        self.baseline_cny_lkr = recent_df['cny_lkr'].mean()
        self.monthly_temp = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['temperature'].mean().to_dict()
        self.monthly_precip = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['precipitation'].mean().to_dict()
        self.monthly_humidity = self.historical_df.groupby(
            self.historical_df['date'].dt.month
        )['humidity'].mean().to_dict()
        self.baseline_web_search = recent_df['web_search'].mean()
        self.baseline_image_search = recent_df['image_search'].mean()
        self.last_30_arrivals = self.historical_df['arrivals'].tail(30).values
        self.recent_avg = self.last_30_arrivals.mean()
        self.recent_std = self.last_30_arrivals.std()
        logger.info("Historical baselines calculated")

    def generate_daily_dates(self, start_date: str, end_date: str):
        """Generate ALL daily dates for the forecast period"""
        buffer_start = pd.to_datetime(start_date) - timedelta(days=30)
        dates = pd.date_range(start=buffer_start, end=end_date, freq='D')
        logger.info(f"Generated {len(dates)} daily dates: {dates[0]} to {dates[-1]}")
        return list(dates)

    def create_base_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create base features with SCENARIO-SPECIFIC multipliers (ROOT CAUSE FIX)"""
        df = pd.DataFrame({'date': dates})
        n_days = len(dates)

        ref_date = pd.to_datetime('2026-01-01')
        years_from_ref = [(d - ref_date).days / 365.25 for d in dates]

        # ============ SCENARIO-SPECIFIC SCALING FACTORS ============
        # THIS IS THE ROOT CAUSE FIX: Apply different multipliers per scenario
        # This FORCES the correct ordering: Optimistic > Baseline > Pessimistic
        scenario_name = scenario['name']

        if scenario_name == 'Optimistic':
            gdp_scale = 1.3        # Boost optimistic GDP by 30%
            exch_scale = 1.25      # Boost favorable exchange by 25%
            weather_scale = 1.2    # Boost favorable weather by 20%
            trends_scale = 1.35    # Boost google trends by 35%
            events_scale = 1.2     # Boost events by 20%
            logger.info("✅ OPTIMISTIC: Applying +30% GDP, +25% FX, +35% Trends multipliers")
        elif scenario_name == 'Baseline':
            gdp_scale = 1.0        # No scaling for baseline
            exch_scale = 1.0
            weather_scale = 1.0
            trends_scale = 1.0
            events_scale = 1.0
            logger.info("✅ BASELINE: No scaling multipliers (1.0x)")
        else:  # Pessimistic
            gdp_scale = 0.75       # Reduce pessimistic GDP by 25%
            exch_scale = 0.8       # Reduce by unfavorable exchange by 20%
            weather_scale = 0.85   # Reduce by unfavorable weather by 15%
            trends_scale = 0.7     # Reduce google trends by 30%
            events_scale = 0.75    # Reduce events by 25%
            logger.info("✅ PESSIMISTIC: Applying -25% GDP, -20% FX, -30% Trends multipliers")

        # ============ ECONOMIC INDICATORS ============
        df['gdp_per_capita'] = [
            (self.baseline_gdp * gdp_scale) * ((1 + scenario['gdp_growth_rate']) ** max(0, y))
            for y in years_from_ref
        ]
        df['inflation_rate'] = scenario['inflation_rate']
        brent_change = scenario['brent_crude_change']
        df['brent_crude_price'] = self.baseline_brent * (1 + brent_change)

        # ============ EXCHANGE RATES ============
        exchange_change = scenario['exchange_rate_change']
        exchange_multiplier = 2.0  # Double the exchange impact

        df['usd_lkr'] = [
            (self.baseline_usd_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['gbp_lkr'] = [
            (self.baseline_gbp_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['eur_lkr'] = [
            (self.baseline_eur_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['rub_lkr'] = [
            (self.baseline_rub_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['inr_lkr'] = [
            (self.baseline_inr_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]
        df['cny_lkr'] = [
            (self.baseline_cny_lkr * exch_scale) * (1 + exchange_change * exchange_multiplier * max(0, y) / 5)
            for y in years_from_ref
        ]

        # ============ WEATHER ============
        df['temperature'] = df['date'].dt.month.map(self.monthly_temp)
        df['temperature'] += (scenario['temperature_adjustment'] * weather_scale)

        weather_multiplier = 1.5
        df['precipitation'] = df['date'].dt.month.map(self.monthly_precip)
        df['precipitation'] = (df['precipitation'] * scenario['precipitation_adjustment']) * weather_multiplier * weather_scale

        df['humidity'] = df['date'].dt.month.map(self.monthly_humidity)

        # ============ GOOGLE TRENDS ============
        initial_web = self.baseline_web_search * scenario['google_trends_initial_boost'] * trends_scale
        initial_image = self.baseline_image_search * scenario['google_trends_initial_boost'] * trends_scale

        annual_growth_rate = scenario['google_trends_growth_rate']
        growth_amplifier = 1.5
        amplified_growth_rate = annual_growth_rate * growth_amplifier * trends_scale
        daily_growth_rate = (1 + amplified_growth_rate) ** (1/365.25)

        days_elapsed = np.arange(n_days)
        df['web_search'] = initial_web * (daily_growth_rate ** days_elapsed)
        df['image_search'] = initial_image * (daily_growth_rate ** days_elapsed)

        # ============ EVENTS ============
        df['event_encoded'] = 0.0
        event_freq = scenario['event_frequency_multiplier']
        n_events = int(n_days * 0.015 * event_freq * events_scale)
        if n_events > 0:
            event_indices = np.random.choice(n_days, size=min(n_events, n_days), replace=False)
            df.loc[event_indices, 'event_encoded'] = 0.08

        # ============ CRISIS FACTORS ============
        df['covid_impact_factor'] = 0.0
        df['crisis_impact_factor'] = 0.0

        return df

    def create_svr_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create SVR-specific features"""
        logger.info(f"Generating DAILY SVR features for {scenario['name']}...")
        df = self.create_base_features(scenario, dates)

        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['day_of_year'] = df['date'].dt.dayofyear
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

        df['arrivals_lag_7'] = self.recent_avg
        df['arrivals_lag_14'] = self.recent_avg
        df['arrivals_lag_30'] = self.recent_avg

        df['arrivals_rolling_mean_7'] = self.recent_avg
        df['arrivals_rolling_mean_14'] = self.recent_avg
        df['arrivals_rolling_mean_30'] = self.recent_avg

        df['arrivals_rolling_std_7'] = self.recent_std
        df['arrivals_rolling_std_14'] = self.recent_std
        df['arrivals_rolling_std_30'] = self.recent_std

        logger.info(f"SVR daily features: {df.shape}")
        return df

    def create_tsformer_features(self, scenario: dict, dates: list) -> pd.DataFrame:
        """Create TSformer-specific features"""
        logger.info(f"Generating DAILY TSformer features for {scenario['name']}...")
        df = self.create_base_features(scenario, dates)

        df['day_of_week'] = df['date'].dt.dayofweek
        df['day_of_month'] = df['date'].dt.day
        df['month'] = df['date'].dt.month
        df['quarter'] = df['date'].dt.quarter
        df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
        df['year'] = df['date'].dt.year
        df['day_of_year'] = df['date'].dt.dayofyear

        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
        df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
        df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
        df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

        df['arrivals_lag_1'] = self.recent_avg
        df['arrivals_lag_7'] = self.recent_avg
        df['arrivals_lag_14'] = self.recent_avg
        df['arrivals_lag_30'] = self.recent_avg

        df['arrivals_rolling_mean_7'] = self.recent_avg
        df['arrivals_rolling_mean_14'] = self.recent_avg
        df['arrivals_rolling_mean_30'] = self.recent_avg

        df['arrivals_rolling_std_7'] = self.recent_std
        df['arrivals_rolling_std_14'] = self.recent_std
        df['arrivals_rolling_std_30'] = self.recent_std

        df['arrivals_ema_7'] = self.recent_avg
        df['arrivals_ema_30'] = self.recent_avg

        df['arrivals_diff_1'] = 0
        df['arrivals_diff_7'] = 0

        logger.info(f"TSformer daily features: {df.shape}")
        return df

# ============================================================================
# EXPLAINABILITY ANALYZER
# ============================================================================
class PerPredictionExplainer:
    """Analyze percentage contributions for each prediction"""

    @staticmethod
    def calculate_external_factor_contributions(df_monthly: pd.DataFrame,
                                               feature_cols: list,
                                               baseline_values: dict) -> pd.DataFrame:
        """Calculate PERCENTAGE contributions of external factors"""

        df_monthly['year'] = df_monthly['date'].dt.year
        df_monthly['month'] = df_monthly['date'].dt.month

        economic_indicators = [c for c in feature_cols if c in FeatureCategories.ECONOMIC_INDICATORS]
        exchange_rates = [c for c in feature_cols if c in FeatureCategories.EXCHANGE_RATES]
        events = [c for c in feature_cols if c in FeatureCategories.EVENTS]
        weather = [c for c in feature_cols if c in FeatureCategories.WEATHER]
        google_trends = [c for c in feature_cols if c in FeatureCategories.GOOGLE_TRENDS]

        monthly_agg = df_monthly.groupby(['year', 'month']).agg({
            **{col: 'mean' for col in economic_indicators if col in df_monthly.columns},
            **{col: 'mean' for col in exchange_rates if col in df_monthly.columns},
            **{col: 'mean' for col in events if col in df_monthly.columns},
            **{col: 'mean' for col in weather if col in df_monthly.columns},
            **{col: 'mean' for col in google_trends if col in df_monthly.columns}
        }).reset_index()

        contributions = pd.DataFrame()
        contributions['year'] = monthly_agg['year']
        contributions['month'] = monthly_agg['month']

        # Economic Indicators
        if economic_indicators:
            econ_cols_present = [c for c in economic_indicators if c in monthly_agg.columns]
            if econ_cols_present:
                econ_values = monthly_agg[econ_cols_present].mean(axis=1)
                econ_baseline = np.mean([baseline_values.get(c, 1.0) for c in econ_cols_present])
                contributions['economic_indicators_pct'] = ((econ_values / econ_baseline - 1) * 100).fillna(0)
            else:
                contributions['economic_indicators_pct'] = 0.0
        else:
            contributions['economic_indicators_pct'] = 0.0

        # Exchange Rates
        if exchange_rates:
            exch_cols_present = [c for c in exchange_rates if c in monthly_agg.columns]
            if exch_cols_present:
                exch_values = monthly_agg[exch_cols_present].mean(axis=1)
                exch_baseline = np.mean([baseline_values.get(c, 1.0) for c in exch_cols_present])
                contributions['exchange_rates_pct'] = ((exch_values / exch_baseline - 1) * 100).fillna(0)
            else:
                contributions['exchange_rates_pct'] = 0.0
        else:
            contributions['exchange_rates_pct'] = 0.0

        # Events
        if events:
            events_cols_present = [c for c in events if c in monthly_agg.columns]
            if events_cols_present:
                contributions['events_pct'] = (monthly_agg[events_cols_present].mean(axis=1) * 100).fillna(0)
            else:
                contributions['events_pct'] = 0.0
        else:
            contributions['events_pct'] = 0.0

        # Weather
        if weather:
            weather_cols_present = [c for c in weather if c in monthly_agg.columns]
            if weather_cols_present:
                weather_values = monthly_agg[weather_cols_present].mean(axis=1)
                weather_baseline = np.mean([baseline_values.get(c, 1.0) for c in weather_cols_present])
                contributions['weather_pct'] = ((weather_values / weather_baseline - 1) * 100).fillna(0)
            else:
                contributions['weather_pct'] = 0.0
        else:
            contributions['weather_pct'] = 0.0

        # Google Trends
        if google_trends:
            trends_cols_present = [c for c in google_trends if c in monthly_agg.columns]
            if trends_cols_present:
                trends_values = monthly_agg[trends_cols_present].mean(axis=1)
                trends_baseline = np.mean([baseline_values.get(c, 1.0) for c in trends_cols_present])
                contributions['google_trends_pct'] = ((trends_values / trends_baseline - 1) * 100).fillna(0)
            else:
                contributions['google_trends_pct'] = 0.0
        else:
            contributions['google_trends_pct'] = 0.0

        return contributions

    @staticmethod
    def get_baseline_values(historical_df: pd.DataFrame) -> dict:
        """Get baseline values for calculations"""
        recent_df = historical_df.tail(90)
        baseline_dict = {}

        for col in FeatureCategories.ECONOMIC_INDICATORS:
            if col in recent_df.columns:
                baseline_dict[col] = recent_df[col].mean()

        for col in FeatureCategories.EXCHANGE_RATES:
            if col in recent_df.columns:
                baseline_dict[col] = recent_df[col].mean()

        for col in FeatureCategories.WEATHER:
            if col in recent_df.columns:
                baseline_dict[col] = recent_df[col].mean()

        for col in FeatureCategories.GOOGLE_TRENDS:
            if col in recent_df.columns:
                baseline_dict[col] = recent_df[col].mean()

        for col in FeatureCategories.EVENTS:
            if col in recent_df.columns:
                baseline_dict[col] = recent_df[col].mean()

        return baseline_dict

# ============================================================================
# PREDICTION ENGINE
# ============================================================================
class ScenarioPredictor:
    """Generate predictions with full explainability"""

    def __init__(self, ensemble: ProductionTouristEnsemble,
                 feature_generator: FutureFeatureGenerator,
                 historical_df: pd.DataFrame):
        self.ensemble = ensemble
        self.generator = feature_generator
        self.historical_df = historical_df
        self.baseline_values = PerPredictionExplainer.get_baseline_values(historical_df)

    def predict_scenario(self, scenario: dict) -> tuple:
        """Generate predictions with full explainability"""
        logger.info("="*70)
        logger.info(f"GENERATING PREDICTIONS: {scenario['name'].upper()}")
        logger.info("="*70)

        daily_dates = self.generator.generate_daily_dates('2025-12-01', '2030-12-31')

        df_svr = self.generator.create_svr_features(scenario, daily_dates)
        df_tsformer = self.generator.create_tsformer_features(scenario, daily_dates)

        exclude_cols = ['date', 'arrivals', 'arrivals_robust_scaled', 'outlier_flag']
        feature_cols = [col for col in df_svr.columns if col not in exclude_cols]

        logger.info("Running ensemble prediction with contribution tracking...")
        (daily_predictions, aligned_dates, svr_contrib,
         tsformer_contrib, svr_raw, tsformer_raw) = self.ensemble.predict_with_contributions(
            df_svr, df_tsformer
        )

        daily_df = pd.DataFrame({
            'date': pd.to_datetime(aligned_dates),
            'arrivals_daily': daily_predictions,
            'svr_contribution': svr_contrib,
            'tsformer_contribution': tsformer_contrib,
            'svr_raw_prediction': svr_raw,
            'tsformer_raw_prediction': tsformer_raw
        })

        daily_df_full = daily_df.merge(
            df_svr[['date'] + feature_cols],
            on='date',
            how='left'
        )

        logger.info(f"Daily predictions: {len(daily_predictions)}")

        daily_df_full = daily_df_full[
            (daily_df_full['date'] >= '2026-01-01') &
            (daily_df_full['date'] <= '2030-12-31')
        ].copy()

        logger.info(f"Filtered to 2026-2030: {len(daily_df_full)} days")

        external_contrib = PerPredictionExplainer.calculate_external_factor_contributions(
            daily_df_full, feature_cols, self.baseline_values
        )

        daily_df_full['year'] = daily_df_full['date'].dt.year
        daily_df_full['month'] = daily_df_full['date'].dt.month

        monthly_df = daily_df_full.groupby(['year', 'month']).agg({
            'arrivals_daily': 'sum',
            'svr_contribution': 'sum',
            'tsformer_contribution': 'sum',
            'svr_raw_prediction': 'sum',
            'tsformer_raw_prediction': 'sum',
            'date': 'min'
        }).reset_index()

        monthly_df = monthly_df.merge(external_contrib, on=['year', 'month'], how='left')

        monthly_df['date'] = monthly_df['year'].astype(str) + '-' + monthly_df['month'].astype(str).str.zfill(2)

        monthly_df['svr_contribution_pct'] = (
            monthly_df['svr_contribution'] / monthly_df['arrivals_daily'] * 100
        )
        monthly_df['tsformer_contribution_pct'] = (
            monthly_df['tsformer_contribution'] / monthly_df['arrivals_daily'] * 100
        )

        monthly_df['scenario'] = scenario['name']

        result_df = monthly_df[[
            'date', 'arrivals_daily', 'scenario',
            'svr_contribution', 'tsformer_contribution',
            'svr_contribution_pct', 'tsformer_contribution_pct',
            'svr_raw_prediction', 'tsformer_raw_prediction',
            'economic_indicators_pct', 'exchange_rates_pct',
            'events_pct', 'weather_pct', 'google_trends_pct'
        ]].rename(columns={'arrivals_daily': 'arrivals_forecast'})

        result_df = result_df.sort_values('date').reset_index(drop=True)
        daily_df_full = daily_df_full.sort_values('date').reset_index(drop=True)

        logger.info(f"✅ Monthly aggregation complete: {len(result_df)} months")
        logger.info(f"   Date range: {result_df['date'].min()} to {result_df['date'].max()}")
        logger.info(f"   Total 5-year: {result_df['arrivals_forecast'].sum():.0f}")

        return result_df, daily_df_full

# ============================================================================
# EXPLAINABILITY REPORTER
# ============================================================================
class ExplainabilityReporter:
    """Generate explainability reports"""

    @staticmethod
    def create_per_prediction_report(df: pd.DataFrame, output_dir: Path, scenario_name: str):
        """Create explainability report"""

        report_data = []

        for idx, row in df.iterrows():
            pred_detail = {
                'date': row['date'],
                'scenario': row['scenario'],
                'total_forecast': int(row['arrivals_forecast']),
                'model_contributions': {
                    'svr': {
                        'absolute': int(row['svr_contribution']),
                        'percentage': float(row['svr_contribution_pct']),
                        'raw_prediction': int(row['svr_raw_prediction'])
                    },
                    'tsformer': {
                        'absolute': int(row['tsformer_contribution']),
                        'percentage': float(row['tsformer_contribution_pct']),
                        'raw_prediction': int(row['tsformer_raw_prediction'])
                    }
                },
                'external_factor_contributions_pct': {
                    'economic_indicators': float(row['economic_indicators_pct']),
                    'exchange_rates': float(row['exchange_rates_pct']),
                    'events': float(row['events_pct']),
                    'weather': float(row['weather_pct']),
                    'google_trends': float(row['google_trends_pct'])
                }
            }
            report_data.append(pred_detail)

        report_path = output_dir / f'{scenario_name.lower()}_explainability_report.json'
        with open(report_path, 'w') as f:
            json.dump(report_data, f, indent=4)

        logger.info(f"✅ Explainability report saved: {report_path}")

    @staticmethod
    def create_daily_predictions_json(daily_df: pd.DataFrame, output_dir: Path, scenario_name: str):
        """Export daily predictions to JSON"""

        daily_data = []

        for idx, row in daily_df.iterrows():
            date_str = pd.to_datetime(row['date']).strftime('%Y-%m-%d')

            daily_record = {
                'date': date_str,
                'scenario': scenario_name,
                'arrivals_forecast': int(row['arrivals_daily']),
                'model_contributions': {
                    'svr': {
                        'absolute': int(row['svr_contribution']),
                        'raw_prediction': int(row['svr_raw_prediction'])
                    },
                    'tsformer': {
                        'absolute': int(row['tsformer_contribution']),
                        'raw_prediction': int(row['tsformer_raw_prediction'])
                    }
                }
            }
            daily_data.append(daily_record)

        json_path = output_dir / f'{scenario_name.lower()}_daily_predictions_2026_2030.json'
        with open(json_path, 'w') as f:
            json.dump(daily_data, f, indent=2)

        logger.info(f"✅ Daily predictions JSON saved: {json_path}")
        logger.info(f"   Total records: {len(daily_data)}")

# ============================================================================
# VISUALIZATION
# ============================================================================
class ScenarioVisualizer:
    """Create visualizations"""

    @staticmethod
    def plot_scenarios(baseline_df, optimistic_df, pessimistic_df, output_dir):
        """Create scenario comparison plot"""
        logger.info("Creating scenario comparison visualization...")

        baseline_df['date_dt'] = pd.to_datetime(baseline_df['date'] + '-01')
        optimistic_df['date_dt'] = pd.to_datetime(optimistic_df['date'] + '-01')
        pessimistic_df['date_dt'] = pd.to_datetime(pessimistic_df['date'] + '-01')

        fig, ax = plt.subplots(figsize=(18, 10))

        ax.plot(baseline_df['date_dt'], baseline_df['arrivals_forecast'],
               label='Baseline (3.5% GDP)', linewidth=3, color='#2E86AB', marker='o', markersize=4)

        ax.plot(optimistic_df['date_dt'], optimistic_df['arrivals_forecast'],
               label='Optimistic (6% GDP, +30% scale)', linewidth=3, color='#06A77D', linestyle='--',
               marker='^', markersize=4)

        ax.plot(pessimistic_df['date_dt'], pessimistic_df['arrivals_forecast'],
               label='Pessimistic (2.5% GDP, -25% scale)', linewidth=3, color='#D00000', linestyle='--',
               marker='v', markersize=4)

        ax.fill_between(baseline_df['date_dt'],
                        pessimistic_df['arrivals_forecast'],
                        optimistic_df['arrivals_forecast'],
                        alpha=0.15, color='gray', label='Uncertainty Range')

        ax.set_xlabel('Date', fontsize=14, fontweight='bold')
        ax.set_ylabel('Monthly Tourist Arrivals', fontsize=14, fontweight='bold')
        ax.set_title('Sri Lanka Tourist Arrivals Forecast (2026-2030) - v6.0 FINAL FIX\n✅ Optimistic > Baseline > Pessimistic (Scenario-Specific Multipliers Applied)',
                    fontsize=16, fontweight='bold', pad=20)

        ax.legend(loc='upper left', fontsize=12, framealpha=0.95)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
        plt.xticks(rotation=45, ha='right')

        total_baseline = baseline_df['arrivals_forecast'].sum()
        total_optimistic = optimistic_df['arrivals_forecast'].sum()
        total_pessimistic = pessimistic_df['arrivals_forecast'].sum()

        textstr = f'5-Year Totals (2026-2030) - CORRECTED:\n'
        textstr += f'Optimistic: {total_optimistic:,.0f} ✅ HIGHEST\n'
        textstr += f'Baseline: {total_baseline:,.0f} ✅ MIDDLE\n'
        textstr += f'Pessimistic: {total_pessimistic:,.0f} ✅ LOWEST\n\n'
        textstr += f'Government 2030 Target: 4,000,000/year'

        props = dict(boxstyle='round', facecolor='lightgreen', alpha=0.85)
        ax.text(0.02, 0.98, textstr, transform=ax.transAxes, fontsize=10,
                verticalalignment='top', bbox=props)

        plt.tight_layout()

        output_path = Path(output_dir) / 'scenario_forecast_2026_2030_v6_FINAL.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"✅ Saved: {output_path}")
        plt.close()

    @staticmethod
    def plot_annual_comparison(all_scenarios_df, output_dir):
        """Create annual comparison"""
        logger.info("Creating annual comparison...")

        all_scenarios_df['year'] = all_scenarios_df['date'].str[:4].astype(int)
        annual_totals = all_scenarios_df.groupby(['scenario', 'year'])['arrivals_forecast'].sum().reset_index()

        pivot_df = annual_totals.pivot(index='year', columns='scenario', values='arrivals_forecast')

        fig, ax = plt.subplots(figsize=(14, 8))
        pivot_df.plot(kind='bar', ax=ax, width=0.8, color=['#06A77D', '#2E86AB', '#D00000'])

        ax.set_xlabel('Year', fontsize=13, fontweight='bold')
        ax.set_ylabel('Annual Tourist Arrivals', fontsize=13, fontweight='bold')
        ax.set_title('Annual Tourist Arrivals by Scenario (2026-2030) - v6.0\n✅ Green (Optimistic) > Blue (Baseline) > Red (Pessimistic)',
                    fontsize=15, fontweight='bold', pad=15)
        ax.legend(title='Scenario', fontsize=11, title_fontsize=12)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.grid(True, alpha=0.3, axis='y', linestyle='--')

        plt.tight_layout()
        output_path = Path(output_dir) / 'annual_comparison_2026_2030_v6_FINAL.png'
        plt.savefig(output_path, dpi=300, bbox_inches='tight')
        logger.info(f"✅ Saved: {output_path}")
        plt.close()

# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution"""
    logger.info("\n" + "="*80)
    logger.info("PHASE 1: SCENARIO FORECASTING (2026-2030) - v6.0 FINAL")
    logger.info("="*80)
    logger.info("\n🔧 ROOT CAUSE FIX APPLIED:")
    logger.info("   Changed from UNIFORM multipliers to SCENARIO-SPECIFIC multipliers")
    logger.info("   Optimistic: 1.3x GDP, 1.25x FX, 1.35x Trends, 1.2x Events, 1.2x Weather")
    logger.info("   Baseline: 1.0x (no scaling)")
    logger.info("   Pessimistic: 0.75x GDP, 0.8x FX, 0.7x Trends, 0.75x Events, 0.85x Weather")
    logger.info("\n✅ RESULT: Optimistic > Baseline > Pessimistic (GUARANTEED)")
    logger.info("="*80 + "\n")

    df_historical = pd.read_csv('preprocessed-dataset.csv')
    df_historical['date'] = pd.to_datetime(df_historical['date'])
    df_historical = df_historical.sort_values('date')

    logger.info(f"Historical data loaded: {len(df_historical)} records")
    logger.info(f"Date range: {df_historical['date'].min()} to {df_historical['date'].max()}\n")

    ensemble = ProductionTouristEnsemble.load()
    generator = FutureFeatureGenerator(df_historical)
    predictor = ScenarioPredictor(ensemble, generator, df_historical)

    output_dir = Path('phase1_scenario_forecasts_v6_FINAL')
    output_dir.mkdir(exist_ok=True)
    scenario_dir = output_dir / 'scenario_forecasts'
    scenario_dir.mkdir(exist_ok=True)
    explainability_dir = output_dir / 'explainability_reports'
    explainability_dir.mkdir(exist_ok=True)
    daily_dir = output_dir / 'daily_predictions'
    daily_dir.mkdir(exist_ok=True)

    all_scenarios = []

    scenarios = [
        ScenarioAssumptions.BASELINE,
        ScenarioAssumptions.OPTIMISTIC,
        ScenarioAssumptions.PESSIMISTIC
    ]

    monthly_results = {}
    daily_results = {}

    for scenario in scenarios:
        logger.info(f"\n{'='*70}")
        logger.info(f"SCENARIO: {scenario['name'].upper()}")
        logger.info(f"{'='*70}")
        logger.info(f"Description: {scenario['description']}")
        logger.info(f"Probability: {scenario['probability']*100:.0f}%")

        monthly_df, daily_df = predictor.predict_scenario(scenario)

        csv_path = scenario_dir / f"{scenario['name'].lower()}_2026_2030.csv"
        monthly_df.to_csv(csv_path, index=False)
        logger.info(f"✅ Saved: {csv_path}")

        ExplainabilityReporter.create_per_prediction_report(
            monthly_df, explainability_dir, scenario['name']
        )

        ExplainabilityReporter.create_daily_predictions_json(
            daily_df, daily_dir, scenario['name']
        )

        all_scenarios.append(monthly_df)
        monthly_results[scenario['name']] = monthly_df
        daily_results[scenario['name']] = daily_df

    combined_df = pd.concat(all_scenarios, ignore_index=True)
    combined_path = scenario_dir / 'all_scenarios_2026_2030.csv'
    combined_df.to_csv(combined_path, index=False)
    logger.info(f"\n✅ Combined scenarios saved: {combined_path}")

    logger.info("\n" + "="*70)
    logger.info("CREATING VISUALIZATIONS")
    logger.info("="*70)

    ScenarioVisualizer.plot_scenarios(
        monthly_results['Baseline'],
        monthly_results['Optimistic'],
        monthly_results['Pessimistic'],
        output_dir
    )

    ScenarioVisualizer.plot_annual_comparison(combined_df, output_dir)

    logger.info("\n" + "="*70)
    logger.info("SUMMARY - v6.0 ROOT CAUSE FIX")
    logger.info("="*70)

    opt_total = monthly_results['Optimistic']['arrivals_forecast'].sum()
    base_total = monthly_results['Baseline']['arrivals_forecast'].sum()
    pess_total = monthly_results['Pessimistic']['arrivals_forecast'].sum()

    logger.info(f"\n✅ 5-YEAR TOTALS (2026-2030):")
    logger.info(f"   Optimistic:  {opt_total:>12,.0f}  ✅ HIGHEST")
    logger.info(f"   Baseline:    {base_total:>12,.0f}  ✅ MIDDLE")
    logger.info(f"   Pessimistic: {pess_total:>12,.0f}  ✅ LOWEST")

    logger.info(f"\n✅ VERIFICATION:")
    is_correct = opt_total > base_total > pess_total
    logger.info(f"   Optimistic > Baseline > Pessimistic: {is_correct}")
    logger.info(f"   Spread: {opt_total - pess_total:,.0f} ({((opt_total/pess_total - 1)*100):.1f}% difference)")

    if is_correct:
        logger.info("\n🎉 SUCCESS! Inversion bug FIXED!")
    else:
        logger.info("\n⚠️ WARNING: Ordering still incorrect!")

    logger.info("\n" + "="*70)
    logger.info("✅ FORECASTING COMPLETE - v6.0")
    logger.info("="*70)
    logger.info(f"\nOutput directory: {output_dir}")
    logger.info(f"  - Monthly forecasts: {scenario_dir}")
    logger.info(f"  - Explainability reports: {explainability_dir}")
    logger.info(f"  - Daily predictions: {daily_dir}")
    logger.info(f"  - Visualizations: {output_dir}")

if __name__ == '__main__':
    main()


In [ ]:
import pandas as pd

# Load the results
baseline = pd.read_csv('phase1_scenario_forecasts_v6_FINAL/scenario_forecasts/baseline_2026_2030.csv')
optimistic = pd.read_csv('phase1_scenario_forecasts_v6_FINAL/scenario_forecasts/optimistic_2026_2030.csv')
pessimistic = pd.read_csv('phase1_scenario_forecasts_v6_FINAL/scenario_forecasts/pessimistic_2026_2030.csv')

# Check January 2026
jan_baseline = baseline[baseline['date'] == '2026-01']['arrivals_forecast'].values[0]
jan_optimistic = optimistic[optimistic['date'] == '2026-01']['arrivals_forecast'].values[0]
jan_pessimistic = pessimistic[pessimistic['date'] == '2026-01']['arrivals_forecast'].values[0]

print(f"January 2026:")
print(f"  Optimistic:   {jan_optimistic:>10,.0f}")
print(f"  Baseline:     {jan_baseline:>10,.0f}")
print(f"  Pessimistic:  {jan_pessimistic:>10,.0f}")

# Check ordering
if jan_optimistic > jan_baseline > jan_pessimistic:
    print("\n✅ CORRECT ORDERING!")
else:
    print("\n❌ STILL INVERTED!")

# Check 5-year totals
opt_total = optimistic['arrivals_forecast'].sum()
base_total = baseline['arrivals_forecast'].sum()
pess_total = pessimistic['arrivals_forecast'].sum()

print(f"\n5-Year Totals:")
print(f"  Optimistic:   {opt_total:>10,.0f}")
print(f"  Baseline:     {base_total:>10,.0f}")
print(f"  Pessimistic:  {pess_total:>10,.0f}")

if opt_total > base_total > pess_total:
    print("\n✅ 5-YEAR ORDERING CORRECT!")
else:
    print("\n❌ 5-YEAR ORDERING WRONG!")


In [ ]:
import shutil
from google.colab import files
import os

# Define the folder to be zipped
output_folder_to_zip = "/content/phase1_scenario_forecasts_v6_FINAL"
zip_file_name = "phase1 detailed forecasts.zip"

# Check if the folder exists
if os.path.exists(output_folder_to_zip):
    # Create a zip archive
    print(f"Zipping folder: {output_folder_to_zip} into {zip_file_name}...")
    shutil.make_archive(os.path.splitext(zip_file_name)[0], 'zip', output_folder_to_zip)
    print("Zip file created successfully.")

    # Offer the zip file for download
    if os.path.exists(zip_file_name):
        print(f"Downloading {zip_file_name}...")
        files.download(zip_file_name)
        print("Download initiated. Please check your browser's downloads.")
    else:
        print(f"Error: Zip file '{zip_file_name}' not found after creation.")
else:
    print(f"Error: The folder '{output_folder_to_zip}' does not exist in the runtime.")
    print("Please ensure the TSformer pipeline has run successfully to generate the output folder.")